# Guide Excursions Monitor

Kaggle notebook for guide excursions fact-first monitoring and exporting `guide_excursions_results.json`.


In [ ]:
from __future__ import annotations

from pathlib import Path as _GuideNotebookPath
import sys as _GuideNotebookSys
_GUIDE_EMBEDDED_GOOGLE_AI = {"__init__.py": "\"\"\"Google AI SDK with rate limiting and secrets management.\n\nThis module provides:\n- SecretsProvider: Unified secret retrieval (env -> Kaggle Secrets -> encrypted datasets)\n- GoogleAIClient: Wrapper over google.generativeai with Supabase-based rate limiting\n- RateLimitError, ProviderError: Custom exceptions\n\"\"\"\n\nfrom google_ai.exceptions import RateLimitError, ProviderError\nfrom google_ai.secrets import SecretsProvider, get_secret, get_secret_pool\nfrom google_ai.client import ExternalCallLease, GoogleAIClient\nfrom google_ai.interactions import (\n    ANTIGRAVITY_AGENT,\n    AntigravityInteractionsClient,\n    InteractionDeadlineExceeded,\n    ProviderInteraction,\n)\n\n__all__ = [\n    \"SecretsProvider\",\n    \"get_secret\",\n    \"get_secret_pool\",\n    \"GoogleAIClient\",\n    \"ExternalCallLease\",\n    \"ANTIGRAVITY_AGENT\",\n    \"AntigravityInteractionsClient\",\n    \"InteractionDeadlineExceeded\",\n    \"ProviderInteraction\",\n    \"RateLimitError\",\n    \"ProviderError\",\n]\n", "client.py": "\"\"\"Google AI client with Supabase-based rate limiting.\n\nFeatures:\n- Wrapper over google.genai with legacy google.generativeai fallback\n- Atomic reserve/finalize through Supabase RPC\n- NO_WAIT policy: raises RateLimitError immediately on limit exceeded\n- Retries only on provider errors (max 3)\n- Structured logging (JSON lines)\n- Idempotency via request_uid\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport json\nimport logging\nimport os\nimport random\nimport re\nimport time\nimport uuid\nimport urllib.error\nimport urllib.request\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone, timedelta\nfrom typing import Any, Callable, Optional, Sequence, TYPE_CHECKING\nfrom time import monotonic as _monotonic\n\nfrom google_ai.exceptions import RateLimitError, ProviderError, ReservationError\nfrom google_ai.limiter_supabase import (\n    GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV,\n    GOOGLE_AI_LIMITER_SUPABASE_URL_ENV,\n)\n\nif TYPE_CHECKING:\n    from supabase import Client as SupabaseClient\n\nlogger = logging.getLogger(__name__)\nIncidentNotifier = Callable[[str, dict[str, Any]], Any]\n\n_DEFAULT_ENV_CANDIDATE_CACHE: dict[tuple[str, tuple[str, ...]], tuple[str, ...] | None] = {}\n# Cache of active key ids for the emergency-overflow env names (keyed by the\n# normalized alias tuple). Separate from the scoped default-env cache so the\n# two lookups never clobber each other.\n_OVERFLOW_ENV_CANDIDATE_CACHE: dict[tuple[str, ...], tuple[str, ...] | None] = {}\n_NORMAL_POOL_ENV_CANDIDATE_CACHE: dict[tuple[str, ...], tuple[str, ...] | None] = {}\n_NORMAL_POOL_CURSOR: dict[tuple[str, ...], int] = {}\n\n\n@dataclass\nclass ReserveResult:\n    \"\"\"Result of a successful rate limit reservation.\"\"\"\n    ok: bool\n    api_key_id: Optional[str] = None\n    env_var_name: Optional[str] = None\n    key_alias: Optional[str] = None\n    minute_bucket: Optional[str] = None\n    day_bucket: Optional[str] = None\n    limits: Optional[dict] = None\n    used_after: Optional[dict] = None\n    blocked_reason: Optional[str] = None\n    retry_after_ms: Optional[int] = None\n    quota_scope: Optional[str] = None\n    limiter_contract: Optional[str] = None\n    bucket_strategy: Optional[str] = None\n\n\n@dataclass\nclass UsageInfo:\n    \"\"\"Token usage information from provider response.\"\"\"\n    input_tokens: int = 0\n    output_tokens: int = 0\n    total_tokens: int = 0\n    model: str = \"\"\n\n\n@dataclass\nclass RequestContext:\n    \"\"\"Context for a single request (may have multiple attempts).\"\"\"\n    request_uid: str\n    consumer: str\n    account_name: Optional[str]\n    model: str\n    reserved_tpm: int\n    requested_model: Optional[str] = None\n    provider_model: Optional[str] = None\n    provider_model_name: Optional[str] = None\n    api_key_id: Optional[str] = None\n    quota_scope: Optional[str] = None\n    started_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))\n\n\n@dataclass(frozen=True)\nclass ExternalCallLease:\n    \"\"\"Serializable identity for one strictly-accounted external provider POST.\n\n    The lease deliberately stores the environment variable name rather than the\n    secret.  Callers can pass it back to :class:`GoogleAIClient` to mark/finalize\n    the request, while provider adapters resolve the key only immediately before\n    the HTTP call.  One lease is valid for exactly one provider POST.\n    \"\"\"\n\n    request_uid: str\n    attempt_no: int\n    consumer: str\n    account_name: Optional[str]\n    model: str\n    reserved_tpm: int\n    api_key_id: str\n    env_var_name: str\n    key_alias: Optional[str]\n    minute_bucket: Optional[str]\n    day_bucket: Optional[str]\n    started_at: datetime\n\n    def to_dict(self) -> dict[str, Any]:\n        \"\"\"JSON-safe checkpoint form; it never contains an API-key value.\"\"\"\n\n        return {\n            \"request_uid\": self.request_uid,\n            \"attempt_no\": self.attempt_no,\n            \"consumer\": self.consumer,\n            \"account_name\": self.account_name,\n            \"model\": self.model,\n            \"reserved_tpm\": self.reserved_tpm,\n            \"api_key_id\": self.api_key_id,\n            \"env_var_name\": self.env_var_name,\n            \"key_alias\": self.key_alias,\n            \"minute_bucket\": self.minute_bucket,\n            \"day_bucket\": self.day_bucket,\n            \"started_at\": self.started_at.astimezone(timezone.utc).isoformat(),\n        }\n\n    @classmethod\n    def from_dict(cls, value: dict[str, Any]) -> \"ExternalCallLease\":\n        \"\"\"Restore a checkpointed lease without resolving its secret.\"\"\"\n\n        started_at = datetime.fromisoformat(str(value[\"started_at\"]))\n        if started_at.tzinfo is None:\n            started_at = started_at.replace(tzinfo=timezone.utc)\n        return cls(\n            request_uid=str(value[\"request_uid\"]),\n            attempt_no=int(value[\"attempt_no\"]),\n            consumer=str(value[\"consumer\"]),\n            account_name=(\n                str(value[\"account_name\"])\n                if value.get(\"account_name\") is not None\n                else None\n            ),\n            model=str(value[\"model\"]),\n            reserved_tpm=int(value[\"reserved_tpm\"]),\n            api_key_id=str(value[\"api_key_id\"]),\n            env_var_name=str(value[\"env_var_name\"]),\n            key_alias=(\n                str(value[\"key_alias\"])\n                if value.get(\"key_alias\") is not None\n                else None\n            ),\n            minute_bucket=(\n                str(value[\"minute_bucket\"])\n                if value.get(\"minute_bucket\") is not None\n                else None\n            ),\n            day_bucket=(\n                str(value[\"day_bucket\"])\n                if value.get(\"day_bucket\") is not None\n                else None\n            ),\n            started_at=started_at,\n        )\n\n\nclass GoogleAIClient:\n    \"\"\"Google AI client with rate limiting and retry logic.\n    \n    Usage:\n        client = GoogleAIClient(\n            supabase_client=get_supabase_client(),\n            secrets_provider=get_provider(),\n        )\n        \n        response = await client.generate_content_async(\n            model=\"gemma-3-27b\",\n            prompt=\"Hello, world!\",\n        )\n    \"\"\"\n    \n    # Default values\n    DEFAULT_MAX_OUTPUT_TOKENS = 8192\n    DEFAULT_TPM_RESERVE_EXTRA = 1000\n    DEFAULT_MULTIMODAL_IMAGE_TOKENS = 1600\n    DEFAULT_EMBEDDING_TPM_RESERVE_EXTRA = 64\n    # Heuristic budget for prompt token estimation. We intentionally overestimate\n    # to avoid passing Supabase reserve checks and then hitting provider 429\n    # on input-token-per-minute quotas.\n    _BYTES_PER_TOKEN_ESTIMATE = 4.0\n    MAX_RETRIES = 3\n    RETRY_DELAYS_MS = [250, 500, 1000]  # Backoff delays\n    RESERVE_FALLBACK_ENV = \"GOOGLE_AI_ALLOW_RESERVE_FALLBACK\"\n    LOCAL_FALLBACK_ENV = \"GOOGLE_AI_LOCAL_LIMITER_FALLBACK\"\n    LOCAL_FALLBACK_ON_ERROR_ENV = \"GOOGLE_AI_LOCAL_LIMITER_ON_RESERVE_ERROR\"\n    LOCAL_RPM_ENV = \"GOOGLE_AI_LOCAL_RPM\"\n    LOCAL_TPM_ENV = \"GOOGLE_AI_LOCAL_TPM\"\n    LOCAL_RPD_ENV = \"GOOGLE_AI_LOCAL_RPD\"\n    RESERVE_RPC_RECHECK_ENV = \"GOOGLE_AI_RESERVE_RPC_RECHECK_SECONDS\"\n    RESERVE_RPC_RETRY_ATTEMPTS_ENV = \"GOOGLE_AI_RESERVE_RPC_RETRY_ATTEMPTS\"\n    RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV = \"GOOGLE_AI_RESERVE_RPC_RETRY_BASE_DELAY_MS\"\n    INCIDENT_NOTIFICATIONS_ENV = \"GOOGLE_AI_INCIDENT_NOTIFICATIONS\"\n    INCIDENT_COOLDOWN_ENV = \"GOOGLE_AI_INCIDENT_COOLDOWN_SECONDS\"\n    TEXT_PRIMARY_MODEL = \"gemma-3-27b\"\n    TEXT_MIN_GEMMA_B = 12\n    # When Supabase client is configured with a non-public schema, PostgREST can\n    # return 404 for RPC that exists in public. In that case we can retry via\n    # direct REST call with explicit schema headers.\n    RESERVE_DIRECT_RETRY_ENV = \"GOOGLE_AI_RESERVE_DIRECT_RETRY\"\n    RESERVE_DIRECT_SCHEMA_ENV = \"GOOGLE_AI_RESERVE_DIRECT_SCHEMA\"\n    RESERVE_SCOPE_TO_DEFAULT_ENV_ENV = \"GOOGLE_AI_RESERVE_SCOPE_TO_DEFAULT_ENV\"\n    # Shared normal allocation for ordinary Google AI consumers. Rotation and\n    # provider-side 429 failover belong to this gateway, not to feature code.\n    # Explicit constructor pools remain available for genuinely scoped lanes.\n    NORMAL_KEY_ENVS_ENV = \"GOOGLE_AI_NORMAL_KEY_ENVS\"\n    # Comma-separated env names whose keys may be borrowed as emergency overflow\n    # when the scoped lane is exhausted for the day. Empty = no overflow.\n    RESERVE_OVERFLOW_KEY_ENVS_ENV = \"GOOGLE_AI_RESERVE_OVERFLOW_KEY_ENVS\"\n    # A scoped reservation only spills into the overflow pool when it is blocked\n    # for a *day-level* reason (the lane is out of daily budget) — never for a\n    # per-minute spike (rpm/tpm), where waiting on the same key is correct.\n    _RESERVE_OVERFLOW_TRIGGER_REASONS = frozenset({\"rpd\", \"no_keys\"})\n    PROVIDER_TIMEOUT_ENV = \"GOOGLE_AI_PROVIDER_TIMEOUT_SEC\"\n    # A successful reservation is safe for concurrent production clients only\n    # when the database proves it aggregates and locks by Cloud project/model.\n    # Older key/model-only RPCs omit this marker and are rejected fail-closed.\n    REQUIRED_LIMITER_CONTRACT = \"google_ai_project_model_atomic_v1\"\n    # The atomic v1 contract predates rolling-window/Pacific-day accounting.\n    # Require the strategy marker as a second, fail-closed schema capability so\n    # an old fixed-minute/UTC-day RPC cannot be accepted after this rollout.\n    REQUIRED_BUCKET_STRATEGY = \"rolling_60s_pacific_day_v2\"\n    EXTERNAL_ACCOUNTING_COMPAT_ENV = \"GOOGLE_AI_EXTERNAL_ACCOUNTING_COMPAT\"\n    _EXTERNAL_TERMINAL_STATUSES = frozenset(\n        {\n            \"requires_action\",\n            \"completed\",\n            \"failed\",\n            \"cancelled\",\n            \"incomplete\",\n            \"budget_exceeded\",\n        }\n    )\n    _EXTERNAL_SEMANTIC_STATUSES = frozenset(\n        {\"not_evaluated\", \"passed\", \"failed\"}\n    )\n\n    # Process-local limiter (used when Supabase reserve RPC is missing/flaky).\n    _local_limiter_lock = asyncio.Lock()\n    _local_limiter_minute_bucket: int | None = None\n    _local_limiter_used_rpm: int = 0\n    _local_limiter_used_tpm: int = 0\n    _local_limiter_day_bucket: str | None = None\n    _local_limiter_used_rpd: int = 0\n\n    @staticmethod\n    def _normalize_rate_limit_model(model: str) -> str:\n        \"\"\"Normalize model id used in Supabase RPC quota tables.\n\n        Provider may use interactive Gemma variants (`-it`), while quota tables\n        in some projects store base model ids (`gemma-3-27b`).\n        \"\"\"\n        raw = (model or \"\").strip()\n        if raw.startswith(\"models/\"):\n            raw = raw.split(\"/\", 1)[1].strip()\n        if raw.startswith(\"gemma-\") and raw.endswith(\"-it\"):\n            return raw[:-3]\n        return raw\n\n    @classmethod\n    def _gemma_b_size(cls, model: str) -> Optional[int]:\n        normalized = cls._normalize_rate_limit_model(model).strip().lower()\n        match = re.match(r\"^gemma-\\d+(?:\\.\\d+)?-(\\d+)b$\", normalized)\n        if not match:\n            return None\n        try:\n            return int(match.group(1))\n        except Exception:\n            return None\n\n    @classmethod\n    def _is_gemma_model(cls, model: str) -> bool:\n        normalized = cls._normalize_rate_limit_model(model).strip().lower()\n        return normalized.startswith(\"gemma-\")\n\n    @classmethod\n    def _is_gemma4_model(cls, model: str) -> bool:\n        normalized = cls._normalize_rate_limit_model(model).strip().lower()\n        return normalized.startswith(\"gemma-4-\")\n\n    @classmethod\n    def _is_disallowed_text_model(cls, model: str) -> bool:\n        size = cls._gemma_b_size(model)\n        return size is not None and size < cls.TEXT_MIN_GEMMA_B\n\n    @staticmethod\n    def _resolve_provider_model(model: str) -> tuple[str, str]:\n        \"\"\"Resolve model id passed to provider and fully-qualified model_name.\n\n        Returns:\n            Tuple of (provider_model, provider_model_name):\n            - provider_model: short provider id (e.g. \"gemma-3-27b-it\")\n            - provider_model_name: fully-qualified API id (e.g. \"models/gemma-3-27b-it\")\n        \"\"\"\n        raw = (model or \"\").strip()\n        if raw.startswith(\"models/\"):\n            provider_model = raw.split(\"/\", 1)[1].strip() or raw\n            return provider_model, raw\n\n        provider_model = raw\n        if provider_model.startswith(\"gemma-\") and not provider_model.endswith(\"-it\"):\n            provider_model = f\"{provider_model}-it\"\n        return provider_model, f\"models/{provider_model}\"\n    \n    def __init__(\n        self,\n        supabase_client: Optional[\"SupabaseClient\"] = None,\n        secrets_provider: Optional[Any] = None,\n        consumer: str = \"bot\",\n        account_name: Optional[str] = None,\n        default_env_var_name: Optional[str] = None,\n        dry_run: bool = False,\n        incident_notifier: Optional[IncidentNotifier] = None,\n        reserve_overflow_key_envs: Optional[Any] = None,\n        reserve_key_envs: Optional[Any] = None,\n    ):\n        \"\"\"Initialize the client.\n        \n        Args:\n            supabase_client: Supabase client for rate limiting RPC calls\n            secrets_provider: Provider for API keys (if None, uses env directly)\n            consumer: Consumer identifier (bot/kaggle/script)\n            account_name: Account name for logging (from GOOGLE_API_LOCALNAME)\n            dry_run: If True, skip actual API calls (for testing)\n        \"\"\"\n        self.supabase = supabase_client\n        self.secrets_provider = secrets_provider\n        self.consumer = consumer\n        self.account_name = account_name or os.getenv(\"GOOGLE_API_LOCALNAME\")\n        self.default_env_var_name = (default_env_var_name or \"GOOGLE_API_KEY\").strip() or \"GOOGLE_API_KEY\"\n        self.dry_run = dry_run\n        self.allow_reserve_fallback = (\n            os.getenv(self.RESERVE_FALLBACK_ENV, \"0\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.allow_local_limiter_fallback = (\n            os.getenv(self.LOCAL_FALLBACK_ENV, \"0\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.allow_local_limiter_on_reserve_error = (\n            os.getenv(self.LOCAL_FALLBACK_ON_ERROR_ENV, \"0\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.incident_notifier = incident_notifier\n        self.incident_notifications_enabled = (\n            os.getenv(self.INCIDENT_NOTIFICATIONS_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.incident_cooldown_seconds = self._read_int_env(\n            self.INCIDENT_COOLDOWN_ENV,\n            900,\n        )\n        self.reserve_rpc_recheck_seconds = self._read_int_env(\n            self.RESERVE_RPC_RECHECK_ENV,\n            60,\n        )\n        self.max_retries = self._read_int_env(\"GOOGLE_AI_MAX_RETRIES\", self.MAX_RETRIES)\n        # A declared normal pool normally rotates to another member after a\n        # provider-side 429.  Consumers with a hard *physical send* budget may\n        # disable that behavior explicitly; ordinary callers retain the\n        # availability-oriented default.\n        self.allow_provider_429_rotation = True\n        # Model fallback after a provider request is availability-oriented by\n        # default. Consumers with a strict physical-send budget can disable it\n        # while still allowing a fallback when the limiter blocks before send.\n        self.allow_provider_model_fallback = True\n        # When true, the current google.genai SDK is mandatory and receives an\n        # explicit one-attempt HTTP policy.  This is stronger than\n        # ``max_retries=1``: the deprecated google.generativeai/GAPIC path can\n        # retry transient HTTP failures underneath the application loop.\n        self.hard_single_provider_attempt = False\n        self.retry_delays_ms = self._read_retry_delays()\n        self.fallback_models = self._read_fallback_models()\n        self.provider_timeout_seconds = self._read_float_env(self.PROVIDER_TIMEOUT_ENV, 0.0)\n        self.external_accounting_compat = (\n            os.getenv(self.EXTERNAL_ACCOUNTING_COMPAT_ENV, \"0\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self._incident_last_sent: dict[str, float] = {}\n        self.scope_reserve_to_default_env = (\n            os.getenv(self.RESERVE_SCOPE_TO_DEFAULT_ENV_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        # Emergency overflow: when the scoped lane is out of daily budget, these\n        # extra env-named keys may be borrowed. Explicit constructor arg wins;\n        # otherwise fall back to the global env (unset = disabled).\n        self.reserve_overflow_key_envs = self._normalize_overflow_envs(\n            reserve_overflow_key_envs\n            if reserve_overflow_key_envs is not None\n            else os.getenv(self.RESERVE_OVERFLOW_KEY_ENVS_ENV, \"\")\n        )\n        # A normal pool participates from the first reservation. It is owned by\n        # the shared gateway rather than any one consumer: when no explicit\n        # scoped override or explicit default lane is supplied, ordinary\n        # clients inherit the same operator-configured allocation. Every\n        # request starts at the next pool member and atomically probes only\n        # eligible members through the RPC.\n        self.reserve_key_envs = self._normalize_overflow_envs(\n            reserve_key_envs\n            if reserve_key_envs is not None\n            else (\n                os.getenv(self.NORMAL_KEY_ENVS_ENV, \"\")\n                if default_env_var_name is None\n                else \"\"\n            )\n        )\n\n        # Cache missing Supabase RPCs to avoid noisy per-request fallbacks when\n        # the Supabase project hasn't been migrated yet (PGRST202).\n        self._reserve_rpc_missing = False\n        self._reserve_rpc_missing_since = 0.0\n        self._mark_sent_rpc_missing = False\n        self._finalize_rpc_missing = False\n        self._legacy_finalize_rpc_missing = False\n        self._missing_rpc_logged: set[str] = set()\n        \n        # Lazy imports for Google provider SDKs. Prefer google.genai; keep the\n        # legacy SDK only as a compatibility fallback while rollout finishes.\n        self._genai_new = None\n        self._genai = None\n\n    @staticmethod\n    def _read_int_env(name: str, default: int) -> int:\n        raw = (os.getenv(name) or \"\").strip()\n        if not raw:\n            return default\n        try:\n            value = int(raw)\n            return max(1, value)\n        except Exception:\n            return default\n\n    @staticmethod\n    def _read_float_env(name: str, default: float) -> float:\n        raw = (os.getenv(name) or \"\").strip()\n        if not raw:\n            return default\n        try:\n            return max(0.0, float(raw))\n        except Exception:\n            return default\n\n    def _read_retry_delays(self) -> list[int]:\n        raw = (os.getenv(\"GOOGLE_AI_RETRY_DELAYS_MS\") or \"\").strip()\n        if not raw:\n            return list(self.RETRY_DELAYS_MS)\n        out: list[int] = []\n        for part in raw.split(\",\"):\n            p = part.strip()\n            if not p:\n                continue\n            try:\n                out.append(max(50, int(p)))\n            except Exception:\n                continue\n        return out or list(self.RETRY_DELAYS_MS)\n\n    @staticmethod\n    def _default_env_aliases(name: str | None) -> list[str]:\n        raw = (name or \"\").strip()\n        if not raw:\n            return []\n        names = [raw]\n        match = re.match(r\"^(GOOGLE_API_KEY)_?(\\d+)$\", raw)\n        if match:\n            prefix, suffix = match.groups()\n            compact = f\"{prefix}{suffix}\"\n            underscored = f\"{prefix}_{suffix}\"\n            for alias in (compact, underscored):\n                if alias not in names:\n                    names.append(alias)\n        return names\n\n    def _resolve_default_env_candidate_key_ids(\n        self,\n        *,\n        consumer: str,\n    ) -> list[str] | None:\n        if self.supabase is None or not self.scope_reserve_to_default_env:\n            return None\n        env_names = tuple(self._default_env_aliases(self.default_env_var_name))\n        if not env_names:\n            return None\n        cache_key = (consumer, env_names)\n        if cache_key in _DEFAULT_ENV_CANDIDATE_CACHE:\n            cached = _DEFAULT_ENV_CANDIDATE_CACHE[cache_key]\n            if cached is None:\n                return None\n            return list(cached)\n        try:\n            result = (\n                self.supabase.table(\"google_ai_api_keys\")\n                .select(\"id, env_var_name, priority\")\n                .eq(\"is_active\", True)\n                .in_(\"env_var_name\", list(env_names))\n                .order(\"priority\")\n                .order(\"id\")\n                .execute()\n            )\n            rows = list(result.data or [])\n        except Exception as exc:\n            logger.error(\n                \"google_ai.default_env_candidates_failed consumer=%s env=%s err=%s\",\n                consumer,\n                \",\".join(env_names),\n                exc,\n            )\n            raise ReservationError(\n                \"Google AI key metadata lookup failed for the explicitly scoped lane\"\n            ) from exc\n        ids = tuple(\n            str(row.get(\"id\"))\n            for row in rows\n            if row.get(\"id\") and str(row.get(\"env_var_name\") or \"\") in env_names\n        )\n        if not ids:\n            logger.warning(\n                \"google_ai.default_env_candidates_missing consumer=%s env=%s\",\n                consumer,\n                \",\".join(env_names),\n            )\n            _DEFAULT_ENV_CANDIDATE_CACHE[cache_key] = ()\n            return []\n        _DEFAULT_ENV_CANDIDATE_CACHE[cache_key] = ids\n        return list(ids)\n\n    @staticmethod\n    def _normalize_overflow_envs(value: Any) -> list[str]:\n        \"\"\"Parse the overflow-key-env configuration (CSV string or list).\"\"\"\n        if value is None:\n            return []\n        raw_items = value.split(\",\") if isinstance(value, str) else list(value)\n        out: list[str] = []\n        seen: set[str] = set()\n        for item in raw_items:\n            name = str(item or \"\").strip()\n            if not name or name in seen:\n                continue\n            seen.add(name)\n            out.append(name)\n        return out\n\n    def _resolve_overflow_candidate_key_ids(\n        self,\n        *,\n        exclude: list[str] | None,\n    ) -> list[str] | None:\n        \"\"\"Active key ids for the configured overflow env names, minus ``exclude``.\n\n        Returns the ids ordered by ``priority`` so the RPC borrows the\n        cheapest-priority spare key first. None when overflow is unconfigured,\n        unavailable, or fully covered by the scoped pool.\n        \"\"\"\n        if self.supabase is None or not self.reserve_overflow_key_envs:\n            return None\n        env_names: list[str] = []\n        seen: set[str] = set()\n        for raw in self.reserve_overflow_key_envs:\n            for alias in self._default_env_aliases(raw):\n                if alias not in seen:\n                    seen.add(alias)\n                    env_names.append(alias)\n        env_tuple = tuple(env_names)\n        if not env_tuple:\n            return None\n        if env_tuple in _OVERFLOW_ENV_CANDIDATE_CACHE:\n            ids = _OVERFLOW_ENV_CANDIDATE_CACHE[env_tuple]\n        else:\n            try:\n                result = (\n                    self.supabase.table(\"google_ai_api_keys\")\n                    .select(\"id, env_var_name, priority\")\n                    .eq(\"is_active\", True)\n                    .in_(\"env_var_name\", list(env_tuple))\n                    .order(\"priority\")\n                    .order(\"id\")\n                    .execute()\n                )\n                rows = list(result.data or [])\n                ids = tuple(\n                    str(row.get(\"id\"))\n                    for row in rows\n                    if row.get(\"id\") and str(row.get(\"env_var_name\") or \"\") in env_tuple\n                )\n            except Exception as exc:\n                logger.warning(\n                    \"google_ai.overflow_candidates_failed envs=%s err=%s\",\n                    \",\".join(env_tuple),\n                    exc,\n                )\n                ids = None\n            _OVERFLOW_ENV_CANDIDATE_CACHE[env_tuple] = ids\n        if not ids:\n            return None\n        exclude_set = set(exclude or [])\n        out = [key_id for key_id in ids if key_id not in exclude_set]\n        return out or None\n\n    def _resolve_normal_pool_candidate_key_ids(self) -> list[str] | None:\n        \"\"\"Resolve the explicitly configured normal key pool.\n\n        Missing registry members never widen the request to all active keys.\n        A completely unresolved pool is represented as ``[]`` so strict\n        consumers can fail closed instead of silently using the default lane.\n        \"\"\"\n\n        if self.supabase is None or not self.reserve_key_envs:\n            return None\n        env_names: list[str] = []\n        seen: set[str] = set()\n        for raw in self.reserve_key_envs:\n            for alias in self._default_env_aliases(raw):\n                if alias not in seen:\n                    seen.add(alias)\n                    env_names.append(alias)\n        env_tuple = tuple(env_names)\n        if not env_tuple:\n            return []\n        if env_tuple in _NORMAL_POOL_ENV_CANDIDATE_CACHE:\n            cached = _NORMAL_POOL_ENV_CANDIDATE_CACHE[env_tuple]\n            return [] if cached == () else (list(cached) if cached else None)\n        try:\n            result = (\n                self.supabase.table(\"google_ai_api_keys\")\n                .select(\"id, env_var_name, priority\")\n                .eq(\"is_active\", True)\n                .in_(\"env_var_name\", list(env_tuple))\n                .order(\"priority\")\n                .order(\"id\")\n                .execute()\n            )\n            rows = list(result.data or [])\n            by_env = {\n                str(row.get(\"env_var_name\") or \"\"): str(row.get(\"id\"))\n                for row in rows\n                if row.get(\"id\")\n            }\n            # Preserve operator pool order, including compact/underscored aliases.\n            ids: list[str] = []\n            for configured in self.reserve_key_envs:\n                for alias in self._default_env_aliases(configured):\n                    key_id = by_env.get(alias)\n                    if key_id and key_id not in ids:\n                        ids.append(key_id)\n                        break\n            missing = [\n                name\n                for name in self.reserve_key_envs\n                if not any(alias in by_env for alias in self._default_env_aliases(name))\n            ]\n            if missing:\n                logger.warning(\n                    \"google_ai.normal_pool_members_missing consumer=%s envs=%s\",\n                    self.consumer,\n                    \",\".join(missing),\n                )\n                # This is a declared normal allocation, not a best-effort\n                # fallback list. A partial pool would hide registry drift and\n                # defeat the promised rotation/capacity contract.\n                _NORMAL_POOL_ENV_CANDIDATE_CACHE[env_tuple] = ()\n                return []\n            cached_ids: tuple[str, ...] = tuple(ids)\n            _NORMAL_POOL_ENV_CANDIDATE_CACHE[env_tuple] = cached_ids\n            return list(cached_ids)\n        except Exception as exc:\n            logger.error(\n                \"google_ai.normal_pool_candidates_failed consumer=%s envs=%s err=%s\",\n                self.consumer,\n                \",\".join(env_tuple),\n                exc,\n            )\n            raise ReservationError(\n                \"Google AI key metadata lookup failed for the normal pool\"\n            ) from exc\n\n    def _candidate_quota_scopes(self, key_ids: list[str]) -> dict[str, str]:\n        \"\"\"Resolve key -> provider quota scope without ever widening on error.\"\"\"\n\n        if self.supabase is None or not key_ids:\n            return {}\n        try:\n            result = (\n                self.supabase.table(\"google_ai_api_keys\")\n                .select(\"id, quota_scope\")\n                .eq(\"is_active\", True)\n                .in_(\"id\", key_ids)\n                .execute()\n            )\n            rows = list(result.data or [])\n        except Exception as exc:\n            logger.error(\n                \"google_ai.quota_scope_metadata_failed consumer=%s err=%s\",\n                self.consumer,\n                exc,\n            )\n            return {}\n        scopes = {\n            str(row.get(\"id\")): str(row.get(\"quota_scope\") or \"\").strip()\n            for row in rows\n            if row.get(\"id\") and str(row.get(\"quota_scope\") or \"\").strip()\n        }\n        if any(key_id not in scopes for key_id in key_ids):\n            logger.error(\n                \"google_ai.quota_scope_metadata_incomplete consumer=%s requested=%s resolved=%s\",\n                self.consumer,\n                len(key_ids),\n                len(scopes),\n            )\n            return {}\n        return scopes\n\n    def _provider_429_rotation_candidates(\n        self,\n        *,\n        explicit_candidate_key_ids: Optional[list[str]],\n        exclude_key_ids: set[str],\n        exclude_quota_scopes: set[str],\n    ) -> list[str]:\n        \"\"\"Return unused members of an explicitly configured normal pool.\n\n        Provider-side quota can drift from the shared reservation ledger when\n        another process/project consumer uses the same Google key.  Only a\n        declared normal pool may absorb that drift: emergency overflow remains\n        fail-fast, and an explicit caller scope is never widened.\n        \"\"\"\n\n        if not self.reserve_key_envs:\n            return []\n        pool = self._resolve_normal_pool_candidate_key_ids() or []\n        if explicit_candidate_key_ids is not None:\n            allowed = set(explicit_candidate_key_ids)\n            pool = [key_id for key_id in pool if key_id in allowed]\n        pool = [key_id for key_id in pool if key_id not in exclude_key_ids]\n        scopes = self._candidate_quota_scopes(pool)\n        if not scopes:\n            return []\n        return [\n            key_id\n            for key_id in pool\n            if scopes[key_id] not in exclude_quota_scopes\n        ]\n\n    async def _report_provider_429(\n        self,\n        *,\n        ctx: RequestContext,\n        attempt_no: int,\n        retry_after_ms: int | None,\n    ) -> bool:\n        \"\"\"Publish provider-side quota drift to the shared ledger.\n\n        Rotation is unsafe if this report fails: another process could select\n        the same Cloud-project scope immediately.  Callers therefore rotate\n        only after this RPC succeeds.\n        \"\"\"\n\n        if self.supabase is None:\n            return False\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_report_provider_429\",\n                {\n                    \"p_request_uid\": ctx.request_uid,\n                    \"p_attempt_no\": attempt_no,\n                    \"p_retry_after_ms\": retry_after_ms,\n                },\n                log_label=\"provider_429\",\n            )\n            return True\n        except Exception as exc:\n            logger.error(\n                \"google_ai.provider_429_report_failed consumer=%s model=%s scope=%s err=%s\",\n                ctx.consumer,\n                ctx.model,\n                ctx.quota_scope,\n                exc,\n            )\n            return False\n\n    @staticmethod\n    def _rotated_normal_pool(candidate_key_ids: list[str]) -> list[str]:\n        if not candidate_key_ids:\n            return []\n        pool_key = tuple(candidate_key_ids)\n        cursor = int(_NORMAL_POOL_CURSOR.get(pool_key, 0)) % len(candidate_key_ids)\n        _NORMAL_POOL_CURSOR[pool_key] = (cursor + 1) % len(candidate_key_ids)\n        return candidate_key_ids[cursor:] + candidate_key_ids[:cursor]\n\n    @classmethod\n    def _reserve_result_from_data(cls, data: dict[str, Any]) -> \"ReserveResult\":\n        limiter_contract = str(data.get(\"limiter_contract\") or \"\").strip() or None\n        bucket_strategy = str(data.get(\"bucket_strategy\") or \"\").strip() or None\n        quota_scope = str(data.get(\"quota_scope\") or \"\").strip() or None\n        raw_ok = bool(data.get(\"ok\", False))\n        ok = raw_ok\n        blocked_reason = data.get(\"blocked_reason\")\n        if ok and limiter_contract != cls.REQUIRED_LIMITER_CONTRACT:\n            blocked_reason = (\n                \"limiter_contract_missing\"\n                if limiter_contract is None\n                else \"limiter_contract_incompatible\"\n            )\n            logger.critical(\n                \"google_ai.reserve_contract_rejected required=%s received=%s\",\n                cls.REQUIRED_LIMITER_CONTRACT,\n                limiter_contract or \"<missing>\",\n            )\n            ok = False\n        elif ok and bucket_strategy != cls.REQUIRED_BUCKET_STRATEGY:\n            blocked_reason = (\n                \"limiter_bucket_strategy_missing\"\n                if bucket_strategy is None\n                else \"limiter_bucket_strategy_incompatible\"\n            )\n            logger.critical(\n                \"google_ai.reserve_bucket_strategy_rejected required=%s received=%s\",\n                cls.REQUIRED_BUCKET_STRATEGY,\n                bucket_strategy or \"<missing>\",\n            )\n            ok = False\n        elif ok and quota_scope is None:\n            blocked_reason = \"limiter_quota_scope_missing\"\n            logger.critical(\n                \"google_ai.reserve_quota_scope_rejected contract=%s\",\n                limiter_contract,\n            )\n            ok = False\n        expose_key_metadata = ok or not raw_ok\n        return ReserveResult(\n            ok=ok,\n            # Never expose usable key metadata from an unverified successful\n            # response.  This prevents callers from accidentally bypassing the\n            # gate by inspecting fields while ``ok`` is false.\n            api_key_id=data.get(\"api_key_id\") if expose_key_metadata else None,\n            env_var_name=data.get(\"env_var_name\") if expose_key_metadata else None,\n            key_alias=data.get(\"key_alias\") if expose_key_metadata else None,\n            minute_bucket=data.get(\"minute_bucket\"),\n            day_bucket=data.get(\"day_bucket\"),\n            limits=data.get(\"limits\"),\n            used_after=data.get(\"used_after\"),\n            blocked_reason=blocked_reason,\n            retry_after_ms=data.get(\"retry_after_ms\"),\n            quota_scope=quota_scope,\n            limiter_contract=limiter_contract,\n            bucket_strategy=bucket_strategy,\n        )\n\n    async def _run_reserve_rpc(self, payload: dict[str, Any]) -> dict[str, Any]:\n        \"\"\"Call ``google_ai_reserve`` with short transient retries; return parsed row.\"\"\"\n        retry_attempts = self._read_int_env(self.RESERVE_RPC_RETRY_ATTEMPTS_ENV, 2)\n        retry_attempts = max(1, min(retry_attempts, 6))\n        retry_base_delay_ms = self._read_int_env(self.RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV, 350)\n        retry_base_delay_ms = max(50, min(retry_base_delay_ms, 5000))\n        rpc_error: Exception | None = None\n        result = None\n        for rpc_attempt in range(1, retry_attempts + 1):\n            try:\n                result = self.supabase.rpc(\"google_ai_reserve\", payload).execute()\n                rpc_error = None\n                break\n            except Exception as exc:\n                rpc_error = exc\n                transient = self._is_transient_reserve_rpc_error(exc)\n                if not transient or rpc_attempt >= retry_attempts:\n                    raise\n                delay_ms = int(retry_base_delay_ms * (2 ** (rpc_attempt - 1)))\n                delay_ms += random.randint(0, max(30, retry_base_delay_ms // 2))\n                delay_ms = min(delay_ms, 7000)\n                logger.warning(\n                    \"google_ai.reserve_rpc_transient_retry attempt=%s/%s delay_ms=%s err=%s\",\n                    rpc_attempt,\n                    retry_attempts,\n                    delay_ms,\n                    str(exc)[:260],\n                )\n                await asyncio.sleep(delay_ms / 1000.0)\n        if result is None:\n            if rpc_error:\n                raise rpc_error\n            raise RuntimeError(\"google_ai_reserve returned no result\")\n        data = result.data\n        if isinstance(data, list) and data:\n            data = data[0]\n        return data\n\n    async def _call_supabase_rpc_with_retries(\n        self,\n        fn_name: str,\n        payload: dict[str, Any],\n        *,\n        log_label: str,\n    ) -> Any:\n        \"\"\"Call a Supabase RPC with short retries on transient transport errors.\"\"\"\n        retry_attempts = self._read_int_env(self.RESERVE_RPC_RETRY_ATTEMPTS_ENV, 2)\n        retry_attempts = max(1, min(retry_attempts, 6))\n        retry_base_delay_ms = self._read_int_env(\n            self.RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV,\n            350,\n        )\n        retry_base_delay_ms = max(50, min(retry_base_delay_ms, 5000))\n        last_exc: Exception | None = None\n        for rpc_attempt in range(1, retry_attempts + 1):\n            try:\n                return self.supabase.rpc(fn_name, payload).execute()\n            except Exception as exc:\n                last_exc = exc\n                transient = self._is_transient_reserve_rpc_error(exc)\n                if not transient or rpc_attempt >= retry_attempts:\n                    raise\n                delay_ms = int(retry_base_delay_ms * (2 ** (rpc_attempt - 1)))\n                delay_ms += random.randint(0, max(30, retry_base_delay_ms // 2))\n                delay_ms = min(delay_ms, 7000)\n                logger.warning(\n                    \"google_ai.%s_rpc_transient_retry fn=%s attempt=%s/%s delay_ms=%s err=%s\",\n                    log_label,\n                    fn_name,\n                    rpc_attempt,\n                    retry_attempts,\n                    delay_ms,\n                    exc,\n                )\n                await asyncio.sleep(delay_ms / 1000.0)\n        raise last_exc or RuntimeError(f\"RPC failed: {fn_name}\")\n\n    @staticmethod\n    def _is_transient_reserve_rpc_error(exc: Exception) -> bool:\n        cls_name = exc.__class__.__name__.lower()\n        msg = str(exc or \"\").lower()\n        if \"timeout\" in cls_name or \"ssl\" in cls_name or \"connection\" in cls_name:\n            return True\n        markers = (\n            \"timed out\",\n            \"timeout\",\n            \"handshake\",\n            \"server disconnected\",\n            \"connection reset\",\n            \"connection aborted\",\n            \"eof\",\n            \"unexpected eof\",\n            \"temporarily unavailable\",\n            \"tls\",\n            \"ssl\",\n        )\n        return any(token in msg for token in markers)\n\n    def _read_fallback_models(self) -> list[str]:\n        raw = (os.getenv(\"GOOGLE_AI_FALLBACK_MODELS\") or \"\").strip()\n        if not raw:\n            return []\n        out: list[str] = []\n        seen: set[str] = set()\n        for part in raw.split(\",\"):\n            model = part.strip()\n            if not model:\n                continue\n            key = model.lower()\n            if key in seen:\n                continue\n            seen.add(key)\n            out.append(model)\n        return out\n\n    def _build_model_chain(\n        self,\n        requested_model: str,\n        fallback_models: Optional[list[str]] = None,\n    ) -> list[str]:\n        requested = (requested_model or \"\").strip()\n        effective_fallbacks = (\n            self.fallback_models if fallback_models is None else fallback_models\n        )\n        chain: list[str] = []\n        seen: set[str] = set()\n        for model in [requested, *effective_fallbacks]:\n            m = (model or \"\").strip()\n            if not m:\n                continue\n            if self._is_disallowed_text_model(m):\n                logger.warning(\n                    \"google_ai.model_chain_skip model=%s reason=below_%sb_text_policy\",\n                    m,\n                    self.TEXT_MIN_GEMMA_B,\n                )\n                continue\n            key = self._normalize_rate_limit_model(m).lower()\n            if key in seen:\n                continue\n            seen.add(key)\n            chain.append(m)\n        has_gemma = self._is_gemma_model(requested) or any(\n            self._is_gemma_model(m) for m in effective_fallbacks\n        )\n        return chain or [self.TEXT_PRIMARY_MODEL if has_gemma else requested_model]\n\n    async def _notify_incident(\n        self,\n        kind: str,\n        *,\n        ctx: RequestContext | None = None,\n        severity: str = \"critical\",\n        message: str | None = None,\n        details: dict[str, Any] | None = None,\n    ) -> None:\n        if not self.incident_notifier or not self.incident_notifications_enabled:\n            return\n\n        model = (ctx.requested_model if ctx else None) or (ctx.model if ctx else None) or \"\"\n        base = f\"{kind}:{self.consumer}:{model}\"\n        dedupe_key = base\n        if details:\n            code = details.get(\"error_code\") or details.get(\"blocked_reason\") or details.get(\"error_type\")\n            if code:\n                dedupe_key = f\"{base}:{code}\"\n\n        now = _monotonic()\n        last = self._incident_last_sent.get(dedupe_key)\n        if last is not None and (now - last) < float(self.incident_cooldown_seconds):\n            return\n        self._incident_last_sent[dedupe_key] = now\n\n        payload: dict[str, Any] = {\n            \"kind\": kind,\n            \"severity\": severity,\n            \"consumer\": self.consumer,\n            \"account_name\": self.account_name,\n            \"message\": message,\n            \"ts\": datetime.now(timezone.utc).isoformat(),\n        }\n        if ctx:\n            payload.update(\n                {\n                    \"request_uid\": ctx.request_uid,\n                    \"model\": ctx.model,\n                    \"requested_model\": ctx.requested_model or ctx.model,\n                    \"provider_model\": ctx.provider_model,\n                    \"provider_model_name\": ctx.provider_model_name,\n                    \"invoked_model\": ctx.provider_model_name or ctx.requested_model or ctx.model,\n                }\n            )\n        if details:\n            payload.update(details)\n        try:\n            maybe = self.incident_notifier(kind, payload)\n            if asyncio.iscoroutine(maybe):\n                await maybe\n        except Exception as exc:\n            logger.warning(\"google_ai incident notifier failed: %s\", exc)\n    \n    @property\n    def genai(self):\n        \"\"\"Lazy-load legacy google.generativeai module.\"\"\"\n        if self._genai is None:\n            try:\n                import google.generativeai as genai\n                self._genai = genai\n            except ImportError:\n                raise ImportError(\n                    \"google-generativeai package not installed. \"\n                    \"Install with: pip install google-generativeai\"\n                )\n        return self._genai\n\n    @property\n    def genai_new(self):\n        \"\"\"Lazy-load the current google.genai SDK.\"\"\"\n        if self._genai_new is None:\n            try:\n                from google import genai\n\n                self._genai_new = genai\n            except ImportError:\n                return None\n        return self._genai_new\n\n    def _strict_external_candidate_key_ids(\n        self,\n        key_envs: Sequence[str],\n    ) -> list[str]:\n        \"\"\"Resolve an explicit env-name pool without widening or fallback.\n\n        This path is intentionally independent of the permissive GenerateContent\n        compatibility fallbacks.  Managed-agent calls are expensive and must\n        fail closed when the shared ledger or any declared pool member is absent.\n        \"\"\"\n\n        normalized = self._normalize_overflow_envs(key_envs)\n        if not normalized:\n            raise ReservationError(\"external call requires a non-empty key_env pool\")\n        if self.supabase is None:\n            raise ReservationError(\"external call requires the shared Supabase limiter\")\n\n        aliases: list[str] = []\n        for configured in normalized:\n            for alias in self._default_env_aliases(configured):\n                if alias not in aliases:\n                    aliases.append(alias)\n        try:\n            result = (\n                self.supabase.table(\"google_ai_api_keys\")\n                .select(\"id, env_var_name, priority\")\n                .eq(\"is_active\", True)\n                .in_(\"env_var_name\", aliases)\n                .order(\"priority\")\n                .order(\"id\")\n                .execute()\n            )\n            rows = list(result.data or [])\n        except Exception as exc:\n            raise ReservationError(\n                f\"failed to resolve external key pool: {str(exc)[:300]}\"\n            ) from exc\n\n        by_env = {\n            str(row.get(\"env_var_name\") or \"\"): str(row.get(\"id\"))\n            for row in rows\n            if row.get(\"id\")\n        }\n        ids: list[str] = []\n        missing: list[str] = []\n        for configured in normalized:\n            selected = next(\n                (\n                    by_env[alias]\n                    for alias in self._default_env_aliases(configured)\n                    if alias in by_env\n                ),\n                None,\n            )\n            if not selected:\n                missing.append(configured)\n            elif selected not in ids:\n                ids.append(selected)\n        if missing or len(ids) != len(normalized):\n            # Env names are metadata and safe to report; key values are never read\n            # on this failure path.\n            suffix = f\": {','.join(missing)}\" if missing else \"\"\n            raise ReservationError(f\"external key pool is incomplete{suffix}\")\n        return ids\n\n    async def reserve_external_call(\n        self,\n        *,\n        model: str,\n        reserved_tpm: int,\n        key_envs: Sequence[str],\n        request_uid: Optional[str] = None,\n    ) -> ExternalCallLease:\n        \"\"\"Reserve one provider POST against an explicit, fail-closed key pool.\n\n        No process-local limiter, default key, emergency overflow, or direct REST\n        RPC fallback is permitted.  A fresh UUID is generated unless the caller\n        supplies one for crash-safe idempotent replay.\n        \"\"\"\n\n        limit_model = self._normalize_rate_limit_model(model)\n        if not limit_model:\n            raise ValueError(\"model is required\")\n        try:\n            reserved = int(reserved_tpm)\n        except (TypeError, ValueError) as exc:\n            raise ValueError(\"reserved_tpm must be an integer\") from exc\n        if reserved < 1:\n            raise ValueError(\"reserved_tpm must be positive\")\n\n        uid = request_uid or str(uuid.uuid4())\n        try:\n            uid = str(uuid.UUID(uid))\n        except (TypeError, ValueError, AttributeError) as exc:\n            raise ValueError(\"request_uid must be a UUID\") from exc\n\n        candidate_ids = self._strict_external_candidate_key_ids(key_envs)\n        ctx = RequestContext(\n            request_uid=uid,\n            consumer=self.consumer,\n            account_name=self.account_name,\n            model=limit_model,\n            requested_model=limit_model,\n            provider_model=limit_model,\n            provider_model_name=limit_model,\n            reserved_tpm=reserved,\n        )\n        last_result = ReserveResult(ok=False, blocked_reason=\"no_keys\")\n        payload_base = {\n            \"p_request_uid\": uid,\n            \"p_attempt_no\": 1,\n            \"p_consumer\": ctx.consumer,\n            \"p_account_name\": ctx.account_name,\n            \"p_model\": ctx.model,\n            \"p_reserved_tpm\": ctx.reserved_tpm,\n        }\n        try:\n            for key_id in self._rotated_normal_pool(candidate_ids):\n                data = await self._run_reserve_rpc(\n                    {**payload_base, \"p_candidate_key_ids\": [key_id]}\n                )\n                last_result = self._reserve_result_from_data(data)\n                if last_result.ok:\n                    break\n                if (last_result.blocked_reason or \"\").lower() not in {\n                    \"rpm\",\n                    \"tpm\",\n                    \"rpd\",\n                    \"no_keys\",\n                }:\n                    break\n        except Exception as exc:\n            raise ReservationError(\n                f\"strict external reservation failed: {str(exc)[:300]}\"\n            ) from exc\n\n        if not last_result.ok:\n            if (last_result.blocked_reason or \"\").startswith(\n                (\"limiter_contract_\", \"limiter_bucket_strategy_\")\n            ):\n                raise ReservationError(\n                    \"external reservation requires limiter contract/strategy \"\n                    f\"{self.REQUIRED_LIMITER_CONTRACT}/\"\n                    f\"{self.REQUIRED_BUCKET_STRATEGY}; received \"\n                    f\"{last_result.limiter_contract or '<missing>'}/\"\n                    f\"{last_result.bucket_strategy or '<missing>'}\"\n                )\n            raise RateLimitError(\n                blocked_reason=last_result.blocked_reason or \"unknown\",\n                retry_after_ms=last_result.retry_after_ms,\n                model=ctx.model,\n                api_key_id=last_result.api_key_id,\n                minute_bucket=last_result.minute_bucket,\n                day_bucket=last_result.day_bucket,\n            )\n        if not last_result.api_key_id or not last_result.env_var_name:\n            raise ReservationError(\"external reservation returned incomplete key metadata\")\n\n        return ExternalCallLease(\n            request_uid=uid,\n            attempt_no=1,\n            consumer=ctx.consumer,\n            account_name=ctx.account_name,\n            model=ctx.model,\n            reserved_tpm=ctx.reserved_tpm,\n            api_key_id=str(last_result.api_key_id),\n            env_var_name=str(last_result.env_var_name),\n            key_alias=last_result.key_alias,\n            minute_bucket=last_result.minute_bucket,\n            day_bucket=last_result.day_bucket,\n            started_at=ctx.started_at,\n        )\n\n    def get_external_call_api_key(self, lease: ExternalCallLease) -> str:\n        \"\"\"Resolve a leased key at call time without logging or storing it.\"\"\"\n\n        api_key = self._get_api_key(lease.env_var_name)\n        if not api_key:\n            raise ReservationError(\n                f\"API key not found for leased env: {lease.env_var_name}\"\n            )\n        return api_key\n\n    async def mark_external_call_sent(self, lease: ExternalCallLease) -> None:\n        \"\"\"Public sent marker for adapters that own their HTTP transport.\"\"\"\n\n        if self.supabase is None:\n            raise ReservationError(\"external call requires the shared Supabase limiter\")\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_mark_sent\",\n                {\n                    \"p_request_uid\": lease.request_uid,\n                    \"p_attempt_no\": lease.attempt_no,\n                },\n                log_label=\"external_mark_sent\",\n            )\n        except Exception as exc:\n            raise ReservationError(\n                f\"external mark_sent failed: {str(exc)[:300]}\"\n            ) from exc\n\n    async def finalize_external_call(\n        self,\n        lease: ExternalCallLease,\n        *,\n        provider_interaction_id: Optional[str],\n        provider_terminal_status: str,\n        usage: Optional[UsageInfo],\n        duration_ms: int,\n        semantic_status: str = \"not_evaluated\",\n        error: Optional[ProviderError] = None,\n    ) -> None:\n        \"\"\"Finalize provider accounting without claiming semantic success.\n\n        ``completed`` is only a provider terminal state.  The independent\n        ``semantic_status`` remains ``not_evaluated`` until a downstream\n        validator records its verdict with :meth:`record_external_call_semantic_result`.\n        \"\"\"\n\n        terminal = (provider_terminal_status or \"\").strip().lower()\n        semantic = (semantic_status or \"\").strip().lower()\n        if terminal not in self._EXTERNAL_TERMINAL_STATUSES:\n            raise ValueError(f\"not a terminal external status: {terminal or '<empty>'}\")\n        if semantic not in self._EXTERNAL_SEMANTIC_STATUSES:\n            raise ValueError(f\"invalid semantic status: {semantic or '<empty>'}\")\n        if semantic == \"passed\" and terminal != \"completed\":\n            raise ValueError(\"semantic pass requires provider completed status\")\n        if self.supabase is None:\n            raise ReservationError(\"external call requires the shared Supabase limiter\")\n\n        payload = {\n            \"p_request_uid\": lease.request_uid,\n            \"p_attempt_no\": lease.attempt_no,\n            \"p_provider_interaction_id\": provider_interaction_id,\n            \"p_provider_terminal_status\": terminal,\n            \"p_semantic_status\": semantic,\n            \"p_usage_input_tokens\": usage.input_tokens if usage else None,\n            \"p_usage_output_tokens\": usage.output_tokens if usage else None,\n            \"p_usage_total_tokens\": usage.total_tokens if usage else None,\n            \"p_duration_ms\": max(0, int(duration_ms)),\n            \"p_error_type\": error.error_type if error else None,\n            \"p_error_code\": error.error_code if error else None,\n            \"p_error_message\": error.error_message if error else None,\n        }\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_finalize_interaction\",\n                payload,\n                log_label=\"external_finalize\",\n            )\n        except Exception as exc:\n            if self.external_accounting_compat and self._is_missing_rpc_error(\n                exc, \"google_ai_finalize_interaction\"\n            ):\n                # Temporary canary/debug bridge for an older shared ledger. It\n                # still reconciles token usage and request terminality, while\n                # semantic truth remains in FestivalWebResearchLaneRun. The\n                # strict production default is fail-closed until migration 007.\n                await self._call_supabase_rpc_with_retries(\n                    \"google_ai_finalize\",\n                    {\n                        \"p_request_uid\": lease.request_uid,\n                        \"p_attempt_no\": lease.attempt_no,\n                        \"p_usage_input_tokens\": usage.input_tokens if usage else None,\n                        \"p_usage_output_tokens\": usage.output_tokens if usage else None,\n                        \"p_usage_total_tokens\": usage.total_tokens if usage else None,\n                        \"p_duration_ms\": max(0, int(duration_ms)),\n                        \"p_provider_status\": \"succeeded\" if terminal == \"completed\" else \"failed\",\n                        \"p_error_type\": error.error_type if error else None,\n                        \"p_error_code\": error.error_code if error else None,\n                        \"p_error_message\": error.error_message if error else None,\n                    },\n                    log_label=\"external_finalize_compat\",\n                )\n                logger.warning(\n                    \"External interaction used compatibility accounting; apply migration 007 before production\"\n                )\n                return\n            raise ReservationError(\n                f\"external interaction finalize failed: {str(exc)[:300]}\"\n            ) from exc\n\n    async def record_external_call_semantic_result(\n        self,\n        lease: ExternalCallLease,\n        *,\n        semantic_status: str,\n        semantic_error: Optional[str] = None,\n    ) -> None:\n        \"\"\"Record the downstream semantic validator's independent verdict.\"\"\"\n\n        semantic = (semantic_status or \"\").strip().lower()\n        if semantic not in {\"passed\", \"failed\"}:\n            raise ValueError(\"semantic_status must be passed or failed\")\n        if self.supabase is None:\n            raise ReservationError(\"external call requires the shared Supabase limiter\")\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_record_interaction_semantic\",\n                {\n                    \"p_request_uid\": lease.request_uid,\n                    \"p_attempt_no\": lease.attempt_no,\n                    \"p_semantic_status\": semantic,\n                    \"p_semantic_error\": (\n                        str(semantic_error)[:1000] if semantic_error else None\n                    ),\n                },\n                log_label=\"external_semantic\",\n            )\n        except Exception as exc:\n            if self.external_accounting_compat and self._is_missing_rpc_error(\n                exc, \"google_ai_record_interaction_semantic\"\n            ):\n                logger.warning(\n                    \"External semantic verdict retained only in operational DB; migration 007 is missing\"\n                )\n                return\n            raise ReservationError(\n                f\"external semantic result failed: {str(exc)[:300]}\"\n            ) from exc\n    \n    async def generate_content_async(\n        self,\n        model: str,\n        prompt: Any,\n        generation_config: Optional[dict] = None,\n        safety_settings: Optional[list] = None,\n        max_output_tokens: Optional[int] = None,\n        candidate_key_ids: Optional[list[str]] = None,\n        fallback_models: Optional[list[str]] = None,\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Generate content with rate limiting and retries.\n        \n        Args:\n            model: Model name (e.g., \"gemma-3-27b\")\n            prompt: Input prompt or multimodal content parts\n            generation_config: Optional generation config\n            safety_settings: Optional safety settings\n            max_output_tokens: Max output tokens (for TPM reservation)\n            candidate_key_ids: Optional list of API key IDs to try\n            \n        Returns:\n            Tuple of (response_text, usage_info)\n            \n        Raises:\n            RateLimitError: If rate limits exceeded (NO_WAIT)\n            ProviderError: If provider error after max retries\n        \"\"\"\n        request_uid = str(uuid.uuid4())\n        reserved_tpm = self._calculate_reserved_tpm(\n            prompt=prompt,\n            max_output_tokens=max_output_tokens or self.DEFAULT_MAX_OUTPUT_TOKENS,\n        )\n        requested_model = (model or \"\").strip()\n        model_chain = self._build_model_chain(requested_model, fallback_models)\n        attempt_cursor = 0\n\n        last_error: Optional[Exception] = None\n        for model_index, model_name in enumerate(model_chain):\n            limit_model = self._normalize_rate_limit_model(model_name)\n            provider_model, provider_model_name = self._resolve_provider_model(\n                model_name or limit_model\n            )\n            ctx = RequestContext(\n                request_uid=request_uid,\n                consumer=self.consumer,\n                account_name=self.account_name,\n                model=limit_model,\n                requested_model=model_name or limit_model,\n                provider_model=provider_model,\n                provider_model_name=provider_model_name,\n                reserved_tpm=reserved_tpm,\n            )\n\n            local_attempt_no = 0\n            provider_429_excluded_key_ids: set[str] = set()\n            provider_429_excluded_quota_scopes: set[str] = set()\n            attempt_candidate_key_ids = candidate_key_ids\n            while True:\n                local_attempt_no += 1\n                attempt_cursor += 1\n                attempt_no = attempt_cursor\n                try:\n                    response_text, usage = await self._attempt_generate(\n                        ctx=ctx,\n                        attempt_no=attempt_no,\n                        prompt=prompt,\n                        generation_config=generation_config,\n                        safety_settings=safety_settings,\n                        max_output_tokens=max_output_tokens,\n                        candidate_key_ids=attempt_candidate_key_ids,\n                    )\n                    usage.model = limit_model\n                    return response_text, usage\n                except RateLimitError as e:\n                    blocked_reason = (e.blocked_reason or \"\").strip().lower()\n                    has_quota_fallback = (\n                        blocked_reason in {\"rpm\", \"tpm\", \"rpd\", \"provider_429\"}\n                        and model_index < (len(model_chain) - 1)\n                    )\n                    if has_quota_fallback:\n                        last_error = e\n                        next_model = model_chain[model_index + 1]\n                        self._log_event(\n                            \"google_ai.model_quota_fallback\",\n                            ctx,\n                            attempt_no=attempt_no,\n                            blocked_reason=blocked_reason,\n                            retry_after_ms=e.retry_after_ms,\n                            next_model=next_model,\n                        )\n                        await self._notify_incident(\n                            \"rate_limit_model_fallback\",\n                            ctx=ctx,\n                            severity=\"warning\",\n                            message=str(e),\n                            details={\n                                \"blocked_reason\": blocked_reason,\n                                \"retry_after_ms\": e.retry_after_ms,\n                                \"next_model\": next_model,\n                            },\n                        )\n                        break\n                    if blocked_reason in {\"no_keys\", \"model_not_found\"}:\n                        await self._notify_incident(\n                            \"rate_limit_blocked\",\n                            ctx=ctx,\n                            severity=\"critical\",\n                            message=str(e),\n                            details={\n                                \"blocked_reason\": e.blocked_reason,\n                                \"retry_after_ms\": e.retry_after_ms,\n                            },\n                        )\n                    raise\n                except ReservationError as e:\n                    last_error = e\n                    await self._notify_incident(\n                        \"reservation_error\",\n                        ctx=ctx,\n                        severity=\"critical\",\n                        message=str(e),\n                    )\n                    raise\n                except ProviderError as e:\n                    last_error = e\n                    if int(getattr(e, \"status_code\", 0) or 0) == 429:\n                        if not self.allow_provider_429_rotation:\n                            raise\n                        selected_key_id = (ctx.api_key_id or \"\").strip()\n                        selected_quota_scope = (ctx.quota_scope or \"\").strip()\n                        report_ok = await self._report_provider_429(\n                            ctx=ctx,\n                            attempt_no=attempt_no,\n                            retry_after_ms=e.retry_after_ms,\n                        )\n                        if not report_ok:\n                            raise\n                        selected_key_was_already_excluded = (\n                            selected_key_id in provider_429_excluded_key_ids\n                        )\n                        if selected_key_id:\n                            provider_429_excluded_key_ids.add(selected_key_id)\n                        if selected_quota_scope:\n                            provider_429_excluded_quota_scopes.add(selected_quota_scope)\n                        remaining_key_ids = self._provider_429_rotation_candidates(\n                            explicit_candidate_key_ids=candidate_key_ids,\n                            exclude_key_ids=provider_429_excluded_key_ids,\n                            exclude_quota_scopes=provider_429_excluded_quota_scopes,\n                        )\n                        if (\n                            selected_key_id\n                            and not selected_key_was_already_excluded\n                            and remaining_key_ids\n                        ):\n                            self._log_event(\n                                \"google_ai.provider_key_rotation\",\n                                ctx,\n                                attempt_no=attempt_no,\n                                exhausted_api_key_id=selected_key_id,\n                                exhausted_quota_scope=selected_quota_scope,\n                                remaining_pool_members=len(remaining_key_ids),\n                                retry_after_ms=e.retry_after_ms,\n                            )\n                            attempt_candidate_key_ids = remaining_key_ids\n                            continue\n                        # Unpooled consumers and exhausted/explicitly scoped\n                        # pools never send again into the same provider quota\n                        # scope.  A configured model fallback may use its own\n                        # independent model quota; otherwise remain fail-fast.\n                        has_fallback = (\n                            self.allow_provider_model_fallback\n                            and model_index < (len(model_chain) - 1)\n                        )\n                        if has_fallback:\n                            next_model = model_chain[model_index + 1]\n                            self._log_event(\n                                \"google_ai.model_fallback\",\n                                ctx,\n                                attempt_no=attempt_no,\n                                next_model=next_model,\n                                error=e,\n                            )\n                            await self._notify_incident(\n                                \"provider_error_fallback\",\n                                ctx=ctx,\n                                severity=\"warning\",\n                                message=str(e),\n                                details={\n                                    \"error_type\": e.error_type,\n                                    \"status_code\": e.status_code,\n                                    \"attempt_no\": attempt_no,\n                                    \"next_model\": next_model,\n                                    \"exhausted_quota_scope\": selected_quota_scope,\n                                },\n                            )\n                            break\n                        raise\n                    can_retry = bool(e.retryable) and local_attempt_no < self.max_retries\n                    if can_retry:\n                        delay_ms = self.retry_delays_ms[\n                            min(local_attempt_no - 1, len(self.retry_delays_ms) - 1)\n                        ]\n                        if e.retry_after_ms:\n                            delay_ms = max(int(delay_ms), int(e.retry_after_ms))\n                        jitter_ms = random.randint(0, 100)\n                        await asyncio.sleep((delay_ms + jitter_ms) / 1000)\n                        self._log_event(\"google_ai.retry\", ctx, attempt_no=attempt_no, error=str(e))\n                        continue\n\n                    has_fallback = (\n                        self.allow_provider_model_fallback\n                        and model_index < (len(model_chain) - 1)\n                    )\n                    await self._notify_incident(\n                        \"provider_error_fallback\" if has_fallback else \"provider_error\",\n                        ctx=ctx,\n                        severity=\"warning\" if has_fallback else \"critical\",\n                        message=str(e),\n                        details={\n                            \"error_type\": e.error_type,\n                            \"error_code\": e.error_code,\n                            \"status_code\": e.status_code,\n                            \"retryable\": int(bool(e.retryable)),\n                            \"attempt_no\": attempt_no,\n                            \"max_retries\": self.max_retries,\n                            \"next_model\": model_chain[model_index + 1] if has_fallback else None,\n                        },\n                    )\n                    if has_fallback:\n                        self._log_event(\n                            \"google_ai.model_fallback\",\n                            ctx,\n                            attempt_no=attempt_no,\n                            next_model=model_chain[model_index + 1],\n                            error=e,\n                        )\n                        break\n                    raise\n\n        raise last_error or ProviderError(error_type=\"unknown\", error_message=\"Max retries exceeded\")\n    \n\n    async def embed_content_async(\n        self,\n        model: str,\n        text: str,\n        *,\n        output_dimensionality: int = 768,\n        task_type: Optional[str] = None,\n        title: Optional[str] = None,\n        candidate_key_ids: Optional[list[str]] = None,\n    ) -> tuple[tuple[float, ...], UsageInfo]:\n        \"\"\"Generate a text embedding through the same reserve/finalize limiter.\n\n        Embedding provider calls must not bypass ``google_ai_reserve``.  The\n        Google embedding REST response does not currently expose token usage in\n        the shape we use here, so finalize records a conservative reserved-token\n        estimate as best-effort usage and does not reduce the reserved TPM.\n        \"\"\"\n\n        request_uid = str(uuid.uuid4())\n        model_name = (model or \"\").strip()\n        limit_model = self._normalize_rate_limit_model(model_name)\n        provider_model, provider_model_name = self._resolve_provider_model(model_name or limit_model)\n        reserved_tpm = self._calculate_reserved_embedding_tpm(text)\n        ctx = RequestContext(\n            request_uid=request_uid,\n            consumer=self.consumer,\n            account_name=self.account_name,\n            model=limit_model,\n            requested_model=model_name or limit_model,\n            provider_model=provider_model,\n            provider_model_name=provider_model_name,\n            reserved_tpm=reserved_tpm,\n        )\n\n        last_error: Optional[Exception] = None\n        for attempt_no in range(1, self.max_retries + 1):\n            try:\n                return await self._attempt_embed(\n                    ctx=ctx,\n                    attempt_no=attempt_no,\n                    text=text,\n                    output_dimensionality=output_dimensionality,\n                    task_type=task_type,\n                    title=title,\n                    candidate_key_ids=candidate_key_ids,\n                )\n            except RateLimitError:\n                raise\n            except ReservationError:\n                raise\n            except ProviderError as exc:\n                last_error = exc\n                if int(getattr(exc, \"status_code\", 0) or 0) == 429:\n                    raise\n                can_retry = bool(exc.retryable) and attempt_no < self.max_retries\n                if can_retry:\n                    delay_ms = self.retry_delays_ms[min(attempt_no - 1, len(self.retry_delays_ms) - 1)]\n                    if exc.retry_after_ms:\n                        delay_ms = max(int(delay_ms), int(exc.retry_after_ms))\n                    await asyncio.sleep((delay_ms + random.randint(0, 100)) / 1000)\n                    self._log_event(\"google_ai.embedding_retry\", ctx, attempt_no=attempt_no, error=str(exc))\n                    continue\n                raise\n        raise last_error or ProviderError(error_type=\"unknown\", error_message=\"Max embedding retries exceeded\")\n\n    async def _attempt_embed(\n        self,\n        *,\n        ctx: RequestContext,\n        attempt_no: int,\n        text: str,\n        output_dimensionality: int,\n        task_type: Optional[str],\n        title: Optional[str],\n        candidate_key_ids: Optional[list[str]],\n    ) -> tuple[tuple[float, ...], UsageInfo]:\n        reserve_result = await self._reserve(ctx, attempt_no, candidate_key_ids)\n        if not reserve_result.ok:\n            raise RateLimitError(\n                blocked_reason=reserve_result.blocked_reason or \"unknown\",\n                retry_after_ms=reserve_result.retry_after_ms,\n                model=ctx.model,\n                api_key_id=reserve_result.api_key_id,\n                minute_bucket=reserve_result.minute_bucket,\n                day_bucket=reserve_result.day_bucket,\n            )\n\n        ctx.api_key_id = reserve_result.api_key_id\n        ctx.quota_scope = reserve_result.quota_scope\n        self._log_event(\"google_ai.reserve_ok\", ctx, attempt_no=attempt_no, reserve=reserve_result)\n\n        api_key = self._get_api_key(reserve_result.env_var_name)\n        if not api_key:\n            await self._notify_incident(\n                \"missing_api_key\",\n                ctx=ctx,\n                severity=\"critical\",\n                message=f\"API key not found: {reserve_result.env_var_name}\",\n                details={\"env_var_name\": reserve_result.env_var_name},\n            )\n            raise ReservationError(f\"API key not found: {reserve_result.env_var_name}\")\n\n        await self._mark_sent(ctx, attempt_no)\n        start_time = _monotonic()\n        try:\n            if self.dry_run:\n                values = tuple(0.0 for _ in range(max(1, int(output_dimensionality))))\n            else:\n                values = await self._call_embedding_provider(\n                    api_key=api_key,\n                    model=ctx.requested_model or ctx.model,\n                    text=text,\n                    output_dimensionality=output_dimensionality,\n                    task_type=task_type,\n                    title=title,\n                )\n            duration_ms = int((_monotonic() - start_time) * 1000)\n        except Exception as exc:\n            duration_ms = int((_monotonic() - start_time) * 1000)\n            provider_error = self._classify_error(exc)\n            await self._finalize(ctx=ctx, attempt_no=attempt_no, usage=None, duration_ms=duration_ms, error=provider_error)\n            self._log_event(\"google_ai.embedding_call_error\", ctx, attempt_no=attempt_no, duration_ms=duration_ms, error=provider_error)\n            raise provider_error\n\n        usage = UsageInfo(input_tokens=int(ctx.reserved_tpm or 0), output_tokens=0, total_tokens=int(ctx.reserved_tpm or 0))\n        await self._finalize(ctx=ctx, attempt_no=attempt_no, usage=usage, duration_ms=duration_ms)\n        self._log_event(\"google_ai.embedding_call_ok\", ctx, attempt_no=attempt_no, duration_ms=duration_ms, usage=usage)\n        return values, usage\n\n    async def _call_embedding_provider(\n        self,\n        *,\n        api_key: str,\n        model: str,\n        text: str,\n        output_dimensionality: int,\n        task_type: Optional[str],\n        title: Optional[str],\n    ) -> tuple[float, ...]:\n        def _sync_call() -> tuple[float, ...]:\n            provider_model, provider_model_name = self._resolve_provider_model(model)\n            endpoint = f\"https://generativelanguage.googleapis.com/v1beta/{provider_model_name}:embedContent\"\n            payload: dict[str, Any] = {\n                \"model\": provider_model_name,\n                \"content\": {\"parts\": [{\"text\": text}]},\n                \"outputDimensionality\": int(output_dimensionality),\n            }\n            if task_type:\n                payload[\"taskType\"] = str(task_type)\n            if title:\n                payload[\"title\"] = str(title)\n            request = urllib.request.Request(\n                endpoint,\n                data=json.dumps(payload, ensure_ascii=False).encode(\"utf-8\"),\n                method=\"POST\",\n                headers={\"Content-Type\": \"application/json\", \"x-goog-api-key\": api_key},\n            )\n            timeout_sec = float(self.provider_timeout_seconds or 0.0) or None\n            with urllib.request.urlopen(request, timeout=timeout_sec) as response:\n                response_payload = json.loads(response.read().decode(\"utf-8\") or \"{}\")\n            values = response_payload.get(\"embedding\", {}).get(\"values\")\n            if not isinstance(values, list) or len(values) != int(output_dimensionality):\n                got = len(values) if isinstance(values, list) else \"missing\"\n                raise RuntimeError(f\"Gemini embedding returned unexpected dimension: {got}\")\n            return tuple(float(value) for value in values)\n\n        return await asyncio.to_thread(_sync_call)\n\n    async def _attempt_generate(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        prompt: Any,\n        generation_config: Optional[dict],\n        safety_settings: Optional[list],\n        max_output_tokens: Optional[int],\n        candidate_key_ids: Optional[list[str]],\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Single attempt to generate content.\"\"\"\n        \n        # 1. Reserve rate limit slot\n        reserve_result = await self._reserve(ctx, attempt_no, candidate_key_ids)\n        \n        if not reserve_result.ok:\n            raise RateLimitError(\n                blocked_reason=reserve_result.blocked_reason or \"unknown\",\n                retry_after_ms=reserve_result.retry_after_ms,\n                model=ctx.model,\n                api_key_id=reserve_result.api_key_id,\n                minute_bucket=reserve_result.minute_bucket,\n                day_bucket=reserve_result.day_bucket,\n            )\n\n        ctx.api_key_id = reserve_result.api_key_id\n        ctx.quota_scope = reserve_result.quota_scope\n        self._log_event(\"google_ai.reserve_ok\", ctx, attempt_no=attempt_no, reserve=reserve_result)\n        \n        # 2. Get API key\n        api_key = self._get_api_key(reserve_result.env_var_name)\n        if not api_key:\n            await self._notify_incident(\n                \"missing_api_key\",\n                ctx=ctx,\n                severity=\"critical\",\n                message=f\"API key not found: {reserve_result.env_var_name}\",\n                details={\"env_var_name\": reserve_result.env_var_name},\n            )\n            raise ReservationError(f\"API key not found: {reserve_result.env_var_name}\")\n        \n        # 3. Mark as sent (before actual call)\n        await self._mark_sent(ctx, attempt_no)\n        \n        # 4. Call provider\n        start_time = _monotonic()\n        try:\n            if self.dry_run:\n                # Dry run mode for testing\n                prompt_preview = self._prompt_text_for_estimate(prompt)[:50]\n                response_text = f\"[DRY RUN] Response for: {prompt_preview}...\"\n                usage = UsageInfo(input_tokens=100, output_tokens=50, total_tokens=150)\n            else:\n                response_text, usage = await self._call_provider(\n                    api_key=api_key,\n                    model=ctx.requested_model or ctx.model,\n                    prompt=prompt,\n                    generation_config=generation_config,\n                    safety_settings=safety_settings,\n                    max_output_tokens=max_output_tokens,\n                )\n            \n            duration_ms = int((_monotonic() - start_time) * 1000)\n            \n        except asyncio.CancelledError:\n            duration_ms = int((_monotonic() - start_time) * 1000)\n            cancelled_error = ProviderError(\n                error_type=\"cancelled\",\n                error_message=\"Provider call cancelled after shared reservation was sent\",\n                retryable=False,\n            )\n            # asyncio cancellation derives from BaseException, so the ordinary\n            # provider-error branch below does not run. Finalize explicitly and\n            # shield the accounting RPC: sent attempts must not remain\n            # indefinitely ambiguous when an outer stage wall clock expires.\n            try:\n                await asyncio.shield(\n                    self._finalize(\n                        ctx=ctx,\n                        attempt_no=attempt_no,\n                        usage=None,\n                        duration_ms=duration_ms,\n                        error=cancelled_error,\n                    )\n                )\n            except Exception as finalize_error:\n                logger.warning(\n                    \"Failed to finalize cancelled Google AI attempt: %s\",\n                    finalize_error,\n                )\n            self._log_event(\n                \"google_ai.call_cancelled\",\n                ctx,\n                attempt_no=attempt_no,\n                duration_ms=duration_ms,\n                error=cancelled_error,\n            )\n            raise\n        except Exception as e:\n            duration_ms = int((_monotonic() - start_time) * 1000)\n            \n            # Classify error\n            provider_error = self._classify_error(e)\n            \n            # Finalize with error\n            await self._finalize(\n                ctx=ctx,\n                attempt_no=attempt_no,\n                usage=None,\n                duration_ms=duration_ms,\n                error=provider_error,\n            )\n            \n            self._log_event(\n                \"google_ai.call_error\",\n                ctx,\n                attempt_no=attempt_no,\n                duration_ms=duration_ms,\n                error=provider_error,\n            )\n            \n            raise provider_error\n        \n        # 5. Finalize (update usage, reconcile TPM)\n        await self._finalize(\n            ctx=ctx,\n            attempt_no=attempt_no,\n            usage=usage,\n            duration_ms=duration_ms,\n        )\n        \n        self._log_event(\n            \"google_ai.call_ok\",\n            ctx,\n            attempt_no=attempt_no,\n            duration_ms=duration_ms,\n            usage=usage,\n        )\n        \n        return response_text, usage\n    \n    async def _reserve(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        candidate_key_ids: Optional[list[str]],\n    ) -> ReserveResult:\n        \"\"\"Reserve rate limit slot via Supabase RPC.\"\"\"\n        if not self.supabase:\n            if self.reserve_key_envs:\n                logger.error(\n                    \"google_ai.normal_pool_limiter_unavailable consumer=%s envs=%s\",\n                    ctx.consumer,\n                    \",\".join(self.reserve_key_envs),\n                )\n                return ReserveResult(\n                    ok=False,\n                    blocked_reason=\"normal_pool_limiter_unavailable\",\n                )\n            # No Supabase still must not become an unlimited production bypass:\n            # use the same process-local fail-fast limiter as the RPC-missing\n            # fallback unless a local caller explicitly disables it.\n            logger.warning(\"No Supabase client, using local rate limit reservation\")\n            if self.allow_local_limiter_fallback:\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-no-supabase\",\n                    blocked_reason=\"supabase_unavailable\",\n                )\n            return ReserveResult(ok=False, blocked_reason=\"shared_limiter_unavailable\")\n        was_cached_missing = self._reserve_rpc_missing\n        if was_cached_missing:\n            now = _monotonic()\n            age = (\n                now - self._reserve_rpc_missing_since\n                if self._reserve_rpc_missing_since > 0\n                else 0.0\n            )\n            if age < self.reserve_rpc_recheck_seconds:\n                if self.allow_local_limiter_fallback:\n                    return await self._local_reserve(\n                        ctx,\n                        attempt_no=attempt_no,\n                        key_alias=\"local-fallback-no-rpc-cached\",\n                        blocked_reason=\"reserve_rpc_missing\",\n                    )\n                return ReserveResult(ok=False, blocked_reason=\"reserve_rpc_missing\")\n            # Cooldown elapsed: retry RPC once instead of staying in cached fallback forever.\n            self._reserve_rpc_missing = False\n            logger.warning(\n                \"google_ai.reserve_rpc_recheck consumer=%s model=%s age_s=%.1f\",\n                ctx.consumer,\n                ctx.model,\n                age,\n            )\n\n        if self._reserve_rpc_missing:\n            if self.allow_local_limiter_fallback:\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-no-rpc-cached\",\n                    blocked_reason=\"reserve_rpc_missing\",\n                )\n            return ReserveResult(ok=False, blocked_reason=\"reserve_rpc_missing\")\n        \n        caller_candidate_scope_explicit = candidate_key_ids is not None\n        scoped_candidate_key_ids = candidate_key_ids\n        normal_pool_active = scoped_candidate_key_ids is None and bool(self.reserve_key_envs)\n        if normal_pool_active:\n            scoped_candidate_key_ids = self._resolve_normal_pool_candidate_key_ids()\n            if not scoped_candidate_key_ids:\n                return ReserveResult(\n                    ok=False,\n                    blocked_reason=\"normal_pool_candidates_missing\",\n                )\n        elif scoped_candidate_key_ids is None:\n            scoped_candidate_key_ids = self._resolve_default_env_candidate_key_ids(\n                consumer=ctx.consumer,\n            )\n        if scoped_candidate_key_ids == []:\n            logger.warning(\n                \"google_ai.reserve_default_env_candidates_missing_fallback \"\n                \"consumer=%s env=%s\",\n                ctx.consumer,\n                self.default_env_var_name,\n            )\n            if self.allow_local_limiter_fallback:\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-default-env-missing\",\n                    blocked_reason=\"default_env_candidates_missing\",\n                )\n            return ReserveResult(ok=False, blocked_reason=\"default_env_candidates_missing\")\n\n        payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_attempt_no\": attempt_no,\n            \"p_consumer\": ctx.consumer,\n            \"p_account_name\": ctx.account_name,\n            \"p_model\": ctx.model,\n            \"p_reserved_tpm\": ctx.reserved_tpm,\n            \"p_candidate_key_ids\": scoped_candidate_key_ids,\n        }\n\n        try:\n            if normal_pool_active and isinstance(scoped_candidate_key_ids, list):\n                last_result = ReserveResult(ok=False, blocked_reason=\"no_keys\")\n                for key_id in self._rotated_normal_pool(scoped_candidate_key_ids):\n                    pool_payload = dict(payload)\n                    pool_payload[\"p_candidate_key_ids\"] = [key_id]\n                    pool_data = await self._run_reserve_rpc(pool_payload)\n                    pool_result = self._reserve_result_from_data(pool_data)\n                    last_result = pool_result\n                    if pool_result.ok:\n                        self._log_event(\n                            \"google_ai.reserve_normal_pool_used\",\n                            ctx,\n                            attempt_no=attempt_no,\n                            reserve=pool_result,\n                        )\n                        return pool_result\n                    if (pool_result.blocked_reason or \"\").strip().lower() not in {\n                        \"rpm\",\n                        \"tpm\",\n                        \"rpd\",\n                        \"no_keys\",\n                    }:\n                        return pool_result\n                return last_result\n\n            data = await self._run_reserve_rpc(payload)\n            result = self._reserve_result_from_data(data)\n\n            if result.ok:\n                if was_cached_missing:\n                    logger.info(\n                        \"google_ai.reserve_rpc_recovered consumer=%s model=%s\",\n                        ctx.consumer,\n                        ctx.model,\n                    )\n                    self._reserve_rpc_missing_since = 0.0\n                    self._missing_rpc_logged.discard(\"google_ai_reserve\")\n                return result\n\n            # Scoped lane refused. If it is out of *daily* budget (rpd/no_keys)\n            # and an emergency overflow pool is configured, retry the reservation\n            # once with the scoped + spare keys merged. The RPC orders by priority\n            # and skips the exhausted scoped key, so it borrows the cheapest spare\n            # key only when the lane is genuinely out of daily budget. The same\n            # request_uid/attempt_no is safe to reuse: a blocked reservation never\n            # writes a request_attempts row, so there is no idempotency conflict.\n            blocked = (result.blocked_reason or \"\").strip().lower()\n            if (\n                not caller_candidate_scope_explicit\n                and isinstance(scoped_candidate_key_ids, list)\n                and scoped_candidate_key_ids\n                and blocked in self._RESERVE_OVERFLOW_TRIGGER_REASONS\n            ):\n                overflow_ids = self._resolve_overflow_candidate_key_ids(\n                    exclude=scoped_candidate_key_ids\n                )\n                if overflow_ids:\n                    payload[\"p_candidate_key_ids\"] = list(scoped_candidate_key_ids) + overflow_ids\n                    overflow_data = await self._run_reserve_rpc(payload)\n                    overflow_result = self._reserve_result_from_data(overflow_data)\n                    if overflow_result.ok:\n                        if was_cached_missing:\n                            self._reserve_rpc_missing_since = 0.0\n                            self._missing_rpc_logged.discard(\"google_ai_reserve\")\n                        self._log_event(\n                            \"google_ai.reserve_overflow_used\",\n                            ctx,\n                            attempt_no=attempt_no,\n                            reserve=overflow_result,\n                        )\n                        return overflow_result\n                    # Overflow pool also exhausted: surface the overflow verdict.\n                    return overflow_result\n\n            return result\n\n        except Exception as e:\n            if (\n                self.allow_reserve_fallback\n                and self._is_missing_reserve_rpc_error(e)\n                and (not self.dry_run)\n                and (os.getenv(self.RESERVE_DIRECT_RETRY_ENV, \"1\").strip().lower() in {\"1\", \"true\", \"yes\", \"on\"})\n            ):\n                # Retry via direct REST call with explicit schema headers.\n                direct = await self._reserve_via_direct_rest(ctx, attempt_no=attempt_no, payload=payload)\n                if direct is not None:\n                    if was_cached_missing:\n                        self._reserve_rpc_missing_since = 0.0\n                        self._missing_rpc_logged.discard(\"google_ai_reserve\")\n                    return direct\n\n            if self.allow_reserve_fallback and self._is_missing_reserve_rpc_error(e):\n                msg = str(e)\n                self._reserve_rpc_missing = True\n                self._reserve_rpc_missing_since = _monotonic()\n                if \"google_ai_reserve\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"google_ai_reserve\")\n                    logger.error(\n                        \"Supabase RPC google_ai_reserve is missing in this Supabase project \"\n                        \"(PGRST202). Rate limiting via Supabase is disabled; using direct env key \"\n                        \"%s instead. Set %s=0 to fail hard. error=%s\",\n                        self.default_env_var_name,\n                        self.RESERVE_FALLBACK_ENV,\n                        msg,\n                    )\n                self._log_event(\n                    \"google_ai.reserve_fallback_no_rpc\",\n                    ctx,\n                    attempt_no=attempt_no,\n                    error=msg[:500],\n                )\n                await self._notify_incident(\n                    \"reserve_rpc_missing\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=(\n                        \"Supabase RPC google_ai_reserve missing; \"\n                        \"switched to process-local limiter fallback (direct API key).\"\n                    ),\n                    details={\"error\": msg[:500]},\n                )\n                if self.allow_local_limiter_fallback:\n                    return await self._local_reserve(\n                        ctx,\n                        attempt_no=attempt_no,\n                        key_alias=\"local-fallback\",\n                        blocked_reason=\"reserve_rpc_missing\",\n                    )\n                return ReserveResult(ok=False, blocked_reason=\"reserve_rpc_missing\")\n            if (\n                self.allow_reserve_fallback\n                and self.allow_local_limiter_on_reserve_error\n                and self.allow_local_limiter_fallback\n            ):\n                msg = str(e)\n                logger.warning(\"Reserve RPC failed; using local limiter fallback. error=%s\", msg)\n                await self._notify_incident(\n                    \"reserve_rpc_error_fallback\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=f\"Reserve RPC failed; using local limiter fallback: {e}\",\n                    details={\"error\": msg[:500]},\n                )\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-reserve-error\",\n                    blocked_reason=\"reserve_rpc_error\",\n                )\n            logger.error(\"Failed to call google_ai_reserve: %s\", e)\n            await self._notify_incident(\n                \"reserve_rpc_error\",\n                ctx=ctx,\n                severity=\"critical\",\n                message=f\"Reserve RPC failed: {e}\",\n                details={\"error\": str(e)[:500]},\n            )\n            raise ReservationError(f\"Reserve RPC failed: {e}\")\n\n    async def _reserve_via_direct_rest(\n        self,\n        ctx: RequestContext,\n        *,\n        attempt_no: int,\n        payload: dict[str, Any],\n    ) -> ReserveResult | None:\n        \"\"\"Call google_ai_reserve via REST endpoint, forcing schema headers.\n\n        This is a fallback for environments where Supabase client is configured with\n        a different schema (e.g. 'private') and PostgREST returns 404 for an RPC\n        that exists in 'public'.\n        \"\"\"\n        # The limiter ledger is an independent security boundary. Never retry\n        # its RPC against a general/storage/personalization Supabase project.\n        base_url = (\n            os.getenv(GOOGLE_AI_LIMITER_SUPABASE_URL_ENV) or \"\"\n        ).strip().rstrip(\"/\")\n        key = (\n            os.getenv(GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV) or \"\"\n        ).strip()\n        if not base_url or not key:\n            return None\n        schema = (os.getenv(self.RESERVE_DIRECT_SCHEMA_ENV) or \"public\").strip() or \"public\"\n        endpoint = f\"{base_url}/rest/v1/rpc/google_ai_reserve\"\n        headers = {\n            \"apikey\": key,\n            \"Authorization\": f\"Bearer {key}\",\n            \"Content-Type\": \"application/json\",\n            \"Accept\": \"application/json\",\n            \"Accept-Profile\": schema,\n            \"Content-Profile\": schema,\n        }\n\n        def _do() -> tuple[int, str]:\n            import requests\n\n            resp = requests.post(endpoint, headers=headers, json=payload, timeout=20)\n            return int(resp.status_code), resp.text or \"\"\n\n        try:\n            status, body = await asyncio.to_thread(_do)\n        except Exception as exc:\n            self._log_event(\n                \"google_ai.reserve_direct_error\",\n                ctx,\n                attempt_no=attempt_no,\n                error=str(exc)[:300],\n            )\n            return None\n\n        if status == 404:\n            # Still missing in this schema/project.\n            return None\n        if status >= 400:\n            self._log_event(\n                \"google_ai.reserve_direct_http_error\",\n                ctx,\n                attempt_no=attempt_no,\n                status=status,\n                body_head=(body or \"\").replace(\"\\n\", \" \")[:240],\n            )\n            return None\n\n        try:\n            data = json.loads(body) if body else {}\n            if isinstance(data, list) and data:\n                data = data[0]\n            if not isinstance(data, dict):\n                return None\n        except Exception:\n            return None\n\n        self._log_event(\n            \"google_ai.reserve_direct_ok\",\n            ctx,\n            attempt_no=attempt_no,\n            status=status,\n            schema=schema,\n        )\n        if data.get(\"ok\") and not data.get(\"key_alias\"):\n            data = {**data, \"key_alias\": f\"direct:{schema}\"}\n        return self._reserve_result_from_data(data)\n\n    def _read_local_limit(self, name: str, default: int) -> int:\n        raw = (os.getenv(name) or \"\").strip()\n        if not raw:\n            return default\n        try:\n            return max(1, int(float(raw)))\n        except Exception:\n            return default\n\n    async def _local_reserve(\n        self,\n        ctx: RequestContext,\n        *,\n        attempt_no: int,\n        key_alias: str,\n        blocked_reason: str,\n    ) -> ReserveResult:\n        \"\"\"Process-local limiter used when Supabase reserve RPC is missing/flaky.\"\"\"\n        rpm_limit = self._read_local_limit(self.LOCAL_RPM_ENV, 15)\n        tpm_limit = self._read_local_limit(self.LOCAL_TPM_ENV, 12000)\n        rpd_limit = self._read_local_limit(self.LOCAL_RPD_ENV, 5000)\n\n        now = time.time()\n        minute_bucket = int(now // 60) * 60\n        day_bucket = datetime.fromtimestamp(now, timezone.utc).date().isoformat()\n        required_tpm = max(1, int(ctx.reserved_tpm))\n\n        async with self._local_limiter_lock:\n            if self._local_limiter_minute_bucket != minute_bucket:\n                self._local_limiter_minute_bucket = minute_bucket\n                self._local_limiter_used_rpm = 0\n                self._local_limiter_used_tpm = 0\n            if self._local_limiter_day_bucket != day_bucket:\n                self._local_limiter_day_bucket = day_bucket\n                self._local_limiter_used_rpd = 0\n\n            limits = {\"rpd\": rpd_limit, \"rpm\": rpm_limit, \"tpm\": tpm_limit}\n            used_before = {\n                \"rpd\": self._local_limiter_used_rpd,\n                \"rpm\": self._local_limiter_used_rpm,\n                \"tpm\": self._local_limiter_used_tpm,\n            }\n\n            if self._local_limiter_used_rpd + 1 > rpd_limit:\n                midnight = (\n                    datetime.fromtimestamp(now, timezone.utc)\n                    .replace(hour=0, minute=0, second=0, microsecond=0)\n                    + timedelta(days=1)\n                )\n                retry_after_ms = max(1000, int((midnight.timestamp() - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=self.default_env_var_name,\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"rpd\",\n                    retry_after_ms=retry_after_ms,\n                )\n            if self._local_limiter_used_rpm + 1 > rpm_limit:\n                retry_after_ms = max(250, int((minute_bucket + 60 - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=self.default_env_var_name,\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"rpm\",\n                    retry_after_ms=retry_after_ms,\n                )\n            if self._local_limiter_used_tpm + required_tpm > tpm_limit:\n                retry_after_ms = max(250, int((minute_bucket + 60 - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=self.default_env_var_name,\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"tpm\",\n                    retry_after_ms=retry_after_ms,\n                )\n\n            self._local_limiter_used_rpd += 1\n            self._local_limiter_used_rpm += 1\n            self._local_limiter_used_tpm += required_tpm\n            used_after = {\n                \"rpd\": self._local_limiter_used_rpd,\n                \"rpm\": self._local_limiter_used_rpm,\n                \"tpm\": self._local_limiter_used_tpm,\n            }\n\n            reserve = ReserveResult(\n                ok=True,\n                env_var_name=self.default_env_var_name,\n                key_alias=key_alias,\n                minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                day_bucket=day_bucket,\n                limits=limits,\n                used_after=used_after,\n                blocked_reason=blocked_reason,\n                retry_after_ms=None,\n            )\n\n        self._log_event(\"google_ai.reserve_local_fallback_ok\", ctx, attempt_no=attempt_no, reserve=reserve)\n        return reserve\n\n    @staticmethod\n    def _is_missing_reserve_rpc_error(error: Exception) -> bool:\n        return GoogleAIClient._is_missing_rpc_error(error, \"google_ai_reserve\")\n\n    @staticmethod\n    def _is_missing_rpc_error(error: Exception, rpc_name: str) -> bool:\n        message = str(error).lower()\n        rpc_name_l = rpc_name.lower()\n        if rpc_name_l not in message:\n            return False\n        markers = (\n            \"pgrst202\",\n            \"route post:/rpc/\",\n            \"not found\",\n            \"schema cache\",\n        )\n        return any(marker in message for marker in markers)\n    \n    async def _mark_sent(self, ctx: RequestContext, attempt_no: int) -> None:\n        \"\"\"Mark request as sent (before calling provider).\"\"\"\n        if not self.supabase:\n            return\n        if self._mark_sent_rpc_missing:\n            return\n        request_uid = ctx.request_uid\n\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_mark_sent\",\n                {\n                    \"p_request_uid\": request_uid,\n                    \"p_attempt_no\": attempt_no,\n                },\n                log_label=\"mark_sent\",\n            )\n        except Exception as e:\n            message = str(e)\n            if self._is_missing_rpc_error(e, \"google_ai_mark_sent\"):\n                self._mark_sent_rpc_missing = True\n                if \"google_ai_mark_sent\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"google_ai_mark_sent\")\n                    logger.warning(\n                        \"Supabase RPC google_ai_mark_sent is missing (PGRST202). \"\n                        \"Will skip it for the rest of the process. error=%s\",\n                        message,\n                    )\n                return\n            logger.warning(\"Failed to mark_sent: %s\", e)\n            await self._notify_incident(\n                \"mark_sent_rpc_error\",\n                ctx=ctx,\n                severity=\"warning\",\n                message=message,\n                details={\"attempt_no\": attempt_no, \"rpc\": \"google_ai_mark_sent\"},\n            )\n    \n    async def _finalize(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        usage: Optional[UsageInfo],\n        duration_ms: int,\n        error: Optional[ProviderError] = None,\n    ) -> None:\n        \"\"\"Finalize request (record usage, reconcile TPM).\"\"\"\n        if not self.supabase:\n            return\n\n        legacy_payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_api_key_id\": ctx.api_key_id,\n            \"p_model\": ctx.model,\n            \"p_actual_input_tokens\": usage.input_tokens if usage else None,\n            \"p_actual_output_tokens\": usage.output_tokens if usage else None,\n            \"p_status\": \"success\" if not error else \"failed\",\n        }\n\n        if self._finalize_rpc_missing:\n            await self._finalize_legacy(legacy_payload)\n            return\n\n        payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_attempt_no\": attempt_no,\n            \"p_usage_input_tokens\": usage.input_tokens if usage else None,\n            \"p_usage_output_tokens\": usage.output_tokens if usage else None,\n            \"p_usage_total_tokens\": usage.total_tokens if usage else None,\n            \"p_duration_ms\": duration_ms,\n            \"p_provider_status\": \"succeeded\" if not error else \"failed\",\n            \"p_error_type\": error.error_type if error else None,\n            \"p_error_code\": error.error_code if error else None,\n            \"p_error_message\": error.error_message if error else None,\n        }\n\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_finalize\",\n                payload,\n                log_label=\"finalize\",\n            )\n            return\n        except Exception as e:\n            if not self._is_missing_rpc_error(e, \"google_ai_finalize\"):\n                logger.warning(\"Failed to finalize: %s\", e)\n                await self._notify_incident(\n                    \"finalize_rpc_error\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=str(e),\n                    details={\"attempt_no\": attempt_no, \"rpc\": \"google_ai_finalize\"},\n                )\n                return\n            logger.info(\"google_ai_finalize missing, falling back to finalize_google_ai_usage\")\n            # Don't try google_ai_finalize again in this process.\n            self._finalize_rpc_missing = True\n\n        await self._finalize_legacy(legacy_payload)\n\n    async def _finalize_legacy(self, legacy_payload: dict[str, Any]) -> None:\n        \"\"\"Legacy finalize fallback for Supabase projects without google_ai_finalize.\"\"\"\n        if self._legacy_finalize_rpc_missing:\n            return\n\n        try:\n            self.supabase.rpc(\"finalize_google_ai_usage\", legacy_payload).execute()\n        except Exception as legacy_error:\n            if self._is_missing_rpc_error(legacy_error, \"finalize_google_ai_usage\"):\n                self._legacy_finalize_rpc_missing = True\n                if \"finalize_google_ai_usage\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"finalize_google_ai_usage\")\n                    logger.warning(\n                        \"Supabase RPC finalize_google_ai_usage is missing (PGRST202). \"\n                        \"Finalize is disabled for the rest of the process. error=%s\",\n                        legacy_error,\n                    )\n                return\n            logger.warning(\"Failed to finalize_google_ai_usage: %s\", legacy_error)\n    \n    async def _call_provider(\n        self,\n        api_key: str,\n        model: str,\n        prompt: Any,\n        generation_config: Optional[dict],\n        safety_settings: Optional[list],\n        max_output_tokens: Optional[int],\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Call Google AI provider.\"\"\"\n        # Build generation config\n        config = dict(generation_config or {})\n        if max_output_tokens and \"max_output_tokens\" not in config:\n            config[\"max_output_tokens\"] = max_output_tokens\n        \n        # Create model:\n        # - google.generativeai expects model_name like \"models/gemma-3-27b-it\"\n        # - For Gemma, \"-it\" is the tested interactive-tuned variant in this project.\n        # - For Gemini, use the model name as-is (no \"-it\" suffix).\n        _provider_model, model_name = self._resolve_provider_model(model)\n\n        # Hosted Gemma 4 thinking is enabled unless callers opt out through the\n        # API. Most repository consumers are bounded extraction, validation,\n        # and rewrite stages; private thoughts can otherwise consume their\n        # complete output allowance and yield an empty MAX_TOKENS response.\n        # Preserve an explicit caller choice, but default to Google's documented\n        # Gemma 4 off switch.\n        if self._is_gemma4_model(model_name) or self._is_gemma4_model(model):\n            config.setdefault(\n                \"thinking_config\",\n                {\"thinking_level\": \"minimal\"},\n            )\n\n        # Gemma 3 frequently rejects native JSON-mode knobs, while Gemma 4\n        # benefits from native structured output contracts. Keep the old guard\n        # for pre-Gemma-4 models, but allow `response_mime_type` /\n        # `response_schema` through for Gemma 4.\n        if \"response_schema_name\" in config:\n            config.pop(\"response_schema_name\", None)\n        if self._is_gemma_model(model_name) or self._is_gemma_model(model):\n            stripped = []\n            if self._is_gemma4_model(model_name) or self._is_gemma4_model(model):\n                pass\n            else:\n                for key in (\"response_mime_type\", \"response_schema\"):\n                    if key in config:\n                        stripped.append(key)\n                        config.pop(key, None)\n            if stripped:\n                logger.info(\n                    \"google_ai: stripped_generation_config model=%s provider_model=%s stripped=%s\",\n                    model,\n                    model_name,\n                    \",\".join(stripped),\n                )\n\n        # Generate content. Prefer the current google.genai SDK; fall back to\n        # deprecated google.generativeai only if the new SDK is unavailable in a\n        # local/dev environment.\n        # Unit tests and a few local probes inject `client._genai` with a fake\n        # legacy module. Respect that injection instead of preferring the real\n        # new SDK and accidentally making network calls with test keys.\n        new_sdk = None if self._genai is not None else self.genai_new\n        if new_sdk is not None:\n            new_config = dict(config)\n            if safety_settings and \"safety_settings\" not in new_config:\n                new_config[\"safety_settings\"] = safety_settings\n            client_kwargs: dict[str, Any] = {}\n            if self.hard_single_provider_attempt:\n                # HttpRetryOptions.attempts includes the original request; 1\n                # therefore disables SDK-internal retries and makes the\n                # feature-level physical-send counter authoritative.\n                client_kwargs[\"http_options\"] = {\n                    \"retry_options\": {\"attempts\": 1},\n                }\n            gen_client = new_sdk.Client(api_key=api_key, **client_kwargs)\n            provider_call = gen_client.aio.models.generate_content(\n                model=model_name,\n                contents=prompt,\n                config=new_config or None,\n            )\n        else:\n            if self.hard_single_provider_attempt:\n                raise ProviderError(\n                    error_type=\"provider_sdk_unavailable\",\n                    error_message=(\n                        \"google.genai is required for a hard single-attempt \"\n                        \"provider budget; legacy SDK fallback is disabled\"\n                    ),\n                    retryable=False,\n                )\n            self.genai.configure(api_key=api_key)\n            gen_model = self.genai.GenerativeModel(model_name)\n            provider_call = gen_model.generate_content_async(\n                prompt,\n                generation_config=config,\n                safety_settings=safety_settings,\n            )\n        # Capture the timeout locally so the error message reports the value\n        # that was actually in effect when ``wait_for`` was armed. Smart Update\n        # mutates ``self.provider_timeout_seconds`` per-stage (set in a try /\n        # finally pair around each call); under concurrent or rapidly\n        # successive stages a different caller's ``finally`` can reset the\n        # attribute back to ``0.0`` while this call is still inside\n        # ``wait_for``. Reading the attribute in the ``except`` branch then\n        # surfaced as ``timed out after 0.0s`` even when the real cap was,\n        # say, 70s — observed on label ``telegraph_render_remove_logistics``\n        # on 2026-05-08, INC-2026-05-08.\n        timeout_sec = float(self.provider_timeout_seconds or 0.0)\n        if timeout_sec > 0:\n            try:\n                response = await asyncio.wait_for(\n                    provider_call,\n                    timeout=timeout_sec,\n                )\n            except asyncio.TimeoutError as exc:\n                raise TimeoutError(\n                    f\"Google AI provider call timed out after {timeout_sec:.1f}s\"\n                ) from exc\n        else:\n            response = await provider_call\n        \n        def _get_usage(resp: Any) -> UsageInfo:\n            usage = UsageInfo()\n            meta = getattr(resp, \"usage_metadata\", None)\n            if not meta:\n                return usage\n            try:\n                if isinstance(meta, dict):\n                    usage.input_tokens = int(meta.get(\"prompt_token_count\") or 0)\n                    usage.output_tokens = int(meta.get(\"candidates_token_count\") or 0)\n                    usage.total_tokens = int(meta.get(\"total_token_count\") or 0)\n                else:\n                    usage.input_tokens = int(getattr(meta, \"prompt_token_count\", 0) or 0)\n                    usage.output_tokens = int(getattr(meta, \"candidates_token_count\", 0) or 0)\n                    usage.total_tokens = int(getattr(meta, \"total_token_count\", 0) or 0)\n            except Exception:\n                # Best-effort only; token accounting must not break requests.\n                pass\n            return usage\n\n        def _extract_text(resp: Any) -> str:\n            # Newer responses often store content in candidates[].content.parts[].text.\n            # Gemma 4 may emit thought-channel parts; those must not leak into\n            # parsed JSON, persisted history, or public operator paths.\n            parts: list[str] = []\n            cands = getattr(resp, \"candidates\", None)\n            if cands:\n                for cand in list(cands):\n                    content = getattr(cand, \"content\", None)\n                    if content is None and isinstance(cand, dict):\n                        content = cand.get(\"content\")\n                    if content is None:\n                        continue\n                    cand_parts = getattr(content, \"parts\", None)\n                    if cand_parts is None and isinstance(content, dict):\n                        cand_parts = content.get(\"parts\")\n                    if cand_parts:\n                        for part in list(cand_parts):\n                            thought = getattr(part, \"thought\", None)\n                            if thought is None and isinstance(part, dict):\n                                thought = part.get(\"thought\")\n                            if thought:\n                                continue\n                            t = getattr(part, \"text\", None)\n                            if t is None and isinstance(part, dict):\n                                t = part.get(\"text\")\n                            if isinstance(t, str) and t.strip():\n                                parts.append(t.strip())\n                    else:\n                        t = getattr(content, \"text\", None)\n                        if isinstance(t, str) and t.strip():\n                            parts.append(t.strip())\n            if parts:\n                return \"\\n\".join(parts).strip()\n\n            # Old `google.generativeai`: response.text\n            try:\n                text = getattr(resp, \"text\", None)\n                if isinstance(text, str) and text.strip():\n                    return text.strip()\n            except Exception:\n                pass\n\n            # No usable answer text. This happens when the model returns only\n            # thought-channel parts (e.g. Gemma 4 spent the whole output-token\n            # budget \"thinking\" and never emitted an answer part) or an otherwise\n            # empty candidate. NEVER stringify the raw SDK response here: str(resp)\n            # dumps the entire GenerateContentResponse repr (thought text, token\n            # counts, http headers) and callers treat that as model output, which\n            # is exactly how the SDK repr leaked into public posts. It also\n            # silently defeats the empty_response guard below. Return \"\" so the\n            # caller raises ProviderError and retries / falls back.\n            return \"\"\n\n        def _diagnose_empty(resp: Any) -> str:\n            \"\"\"Best-effort, repr-safe summary of why extraction yielded no text.\n\n            Must not embed the raw response (that is the leak we are fixing): only\n            small scalar signals (finish reasons, whether any thought-only part was\n            present, token counts) so the failure is visible in logs/metrics.\n            \"\"\"\n            finish_reasons: list[str] = []\n            had_thought_part = False\n            had_any_part = False\n            try:\n                for cand in list(getattr(resp, \"candidates\", None) or []):\n                    fr = getattr(cand, \"finish_reason\", None)\n                    if fr is None and isinstance(cand, dict):\n                        fr = cand.get(\"finish_reason\")\n                    if fr is not None:\n                        finish_reasons.append(getattr(fr, \"name\", None) or str(fr))\n                    content = getattr(cand, \"content\", None)\n                    if content is None and isinstance(cand, dict):\n                        content = cand.get(\"content\")\n                    cand_parts = getattr(content, \"parts\", None)\n                    if cand_parts is None and isinstance(content, dict):\n                        cand_parts = content.get(\"parts\")\n                    for part in list(cand_parts or []):\n                        had_any_part = True\n                        th = getattr(part, \"thought\", None)\n                        if th is None and isinstance(part, dict):\n                            th = part.get(\"thought\")\n                        if th:\n                            had_thought_part = True\n            except Exception:\n                pass\n            meta = getattr(resp, \"usage_metadata\", None)\n            thoughts_tokens = getattr(meta, \"thoughts_token_count\", None) if meta else None\n            return (\n                f\"finish_reasons={finish_reasons or None} \"\n                f\"thought_only={had_thought_part and not response_text and had_any_part} \"\n                f\"thoughts_token_count={thoughts_tokens}\"\n            )\n\n        usage = _get_usage(response)\n        response_text = _extract_text(response)\n        if not response_text:\n            diag = _diagnose_empty(response)\n            logger.warning(\n                \"google_ai.empty_response requested_model=%s provider_model_name=%s %s\",\n                model,\n                model_name,\n                diag,\n            )\n            raise ProviderError(\n                error_type=\"empty_response\",\n                error_message=(\n                    \"Provider returned empty text \"\n                    f\"(requested_model={model}, provider_model_name={model_name}; {diag})\"\n                ),\n                retryable=True,\n            )\n        return response_text, usage\n    \n    def _get_api_key(self, env_var_name: Optional[str]) -> Optional[str]:\n        \"\"\"Get API key from environment or secrets provider.\"\"\"\n        name = env_var_name or self.default_env_var_name or \"GOOGLE_API_KEY\"\n\n        aliases = self._default_env_aliases(name) or [name]\n        if self.secrets_provider:\n            for alias in aliases:\n                value = self.secrets_provider.get_secret(alias)\n                if value:\n                    return value\n\n        for alias in aliases:\n            value = os.getenv(alias)\n            if value:\n                return value\n        return None\n\n    def _prompt_estimate_components(self, prompt: Any) -> tuple[str, int]:\n        if isinstance(prompt, str):\n            return prompt, 0\n        if isinstance(prompt, (list, tuple)):\n            text_parts: list[str] = []\n            blob_count = 0\n            for item in prompt:\n                extracted, item_blob_count = self._prompt_estimate_components(item)\n                if extracted:\n                    text_parts.append(extracted)\n                blob_count += item_blob_count\n            return \"\\n\".join(text_parts), blob_count\n        if isinstance(prompt, dict):\n            text_parts: list[str] = []\n            blob_count = 0\n            parts_value = prompt.get(\"parts\")\n            if isinstance(parts_value, (list, tuple)):\n                extracted, nested_blob_count = self._prompt_estimate_components(parts_value)\n                if extracted:\n                    text_parts.append(extracted)\n                blob_count += nested_blob_count\n            for key in (\"text\", \"prompt\", \"content\"):\n                value = prompt.get(key)\n                if isinstance(value, str):\n                    if value.strip():\n                        text_parts.append(value.strip())\n                    continue\n                if isinstance(value, (list, tuple)):\n                    extracted, nested_blob_count = self._prompt_estimate_components(value)\n                    if extracted:\n                        text_parts.append(extracted)\n                    blob_count += nested_blob_count\n            if \"inline_data\" in prompt or (\n                isinstance(prompt.get(\"mime_type\"), str) and prompt.get(\"data\") is not None\n            ):\n                blob_count += 1\n            if text_parts:\n                return \"\\n\".join(text_parts), blob_count\n            return \"\", blob_count\n        return str(prompt or \"\"), 0\n\n    def _prompt_text_for_estimate(self, prompt: Any) -> str:\n        text, _blob_count = self._prompt_estimate_components(prompt)\n        return text\n\n    def _estimate_prompt_tokens(self, prompt: Any) -> int:\n        \"\"\"Best-effort token estimate for prompts.\n\n        We can't depend on provider-side countTokens here (it would also require\n        an API call). We use conservative byte/char heuristics because long\n        Cyrillic/OCR prompts can tokenize much denser than a simple bytes/4\n        estimate and otherwise slip past reserve() only to hit provider 429.\n        \"\"\"\n        prompt_text, blob_count = self._prompt_estimate_components(prompt)\n        if not prompt_text and blob_count <= 0:\n            return 1\n        try:\n            size = len(prompt_text.encode(\"utf-8\", errors=\"ignore\"))\n        except Exception:\n            size = len(prompt_text)\n        chars = len(prompt_text)\n        non_ascii = sum(1 for ch in prompt_text if ord(ch) > 127)\n        non_ascii_ratio = (non_ascii / chars) if chars > 0 else 0.0\n\n        bytes_est = size / float(self._BYTES_PER_TOKEN_ESTIMATE)\n        if non_ascii_ratio >= 0.30:\n            chars_est = chars * 0.72\n            bytes_est = size / 2.6\n        else:\n            chars_est = chars * 0.30\n\n        est = int(max(bytes_est, chars_est))\n        # Add overhead for JSON, escaping, and tokenization variance.\n        est = int(est * 1.15) + 50\n        if blob_count > 0:\n            est += blob_count * int(self.DEFAULT_MULTIMODAL_IMAGE_TOKENS)\n        return max(1, est)\n\n\n    def _calculate_reserved_embedding_tpm(self, text: Any) -> int:\n        input_est = self._estimate_prompt_tokens(text)\n        extra = self._read_int_env(\"GOOGLE_AI_EMBEDDING_TPM_RESERVE_EXTRA\", self.DEFAULT_EMBEDDING_TPM_RESERVE_EXTRA)\n        return max(1, int(input_est) + int(extra))\n\n    def _calculate_reserved_tpm(self, *, prompt: Any, max_output_tokens: int) -> int:\n        \"\"\"Calculate tokens to reserve for TPM check.\n\n        Supabase reservation must cover BOTH prompt (input) and output tokens.\n        Under-reserving here can lead to provider 429 (ResourceExhausted) even\n        when Supabase reserve() returned ok=true.\n        \"\"\"\n        input_est = self._estimate_prompt_tokens(prompt)\n        output_budget = max(1, int(max_output_tokens))\n        return input_est + output_budget + int(self.DEFAULT_TPM_RESERVE_EXTRA)\n    \n    def _classify_error(self, error: Exception) -> ProviderError:\n        \"\"\"Classify exception into ProviderError.\"\"\"\n        if isinstance(error, ProviderError):\n            return error\n        error_str = str(error)\n        error_lower = error_str.lower()\n        error_type = type(error).__name__\n\n        retry_after_ms: Optional[int] = None\n        # Gemini/Gemma errors often include \"Please retry in <seconds>s.\"\n        m_retry = re.search(r\"retry in\\s+(\\d+(?:\\.\\d+)?)\\s*s\", error_lower)\n        if m_retry:\n            try:\n                retry_after_ms = int(float(m_retry.group(1)) * 1000)\n            except Exception:\n                retry_after_ms = None\n        \n        # Check for retryable errors\n        retryable = any(x in error_lower for x in [\n            \"timeout\",\n            \"connection\",\n            \"temporary\",\n            \"rate limit\",\n            \"503\",\n            \"502\",\n            \"504\",\n            \"resource_exhausted\",\n            \"unavailable\",\n            \"deadline exceeded\",\n            \"internal\",\n            \"try again\",\n            \"econnreset\",\n            \"connection reset\",\n            \"socket\",\n        ])\n        status_code: Optional[int] = None\n        for code in (\"429\", \"500\", \"502\", \"503\", \"504\"):\n            if code in error_lower:\n                try:\n                    status_code = int(code)\n                except Exception:\n                    status_code = None\n                break\n        # Some exceptions don't include \"resource_exhausted\" in the string, but\n        # the type name is still informative.\n        if not retryable and error_type.lower() in {\"resourceexhausted\", \"unavailable\"}:\n            retryable = True\n        if not retryable and status_code == 429:\n            retryable = True\n        \n        return ProviderError(\n            error_type=error_type,\n            error_message=error_str[:500],  # Limit message length\n            retryable=retryable,\n            status_code=status_code,\n            retry_after_ms=retry_after_ms,\n        )\n    \n    def _log_event(\n        self,\n        event: str,\n        ctx: RequestContext,\n        attempt_no: int = 1,\n        **kwargs,\n    ) -> None:\n        \"\"\"Log structured event (JSON lines format).\"\"\"\n        log_data = {\n            \"ts\": datetime.now(timezone.utc).isoformat(),\n            \"event\": event,\n            \"request_uid\": ctx.request_uid,\n            \"attempt_no\": attempt_no,\n            \"consumer\": ctx.consumer,\n            \"account_name\": ctx.account_name,\n            \"model\": ctx.model,\n            \"requested_model\": ctx.requested_model or ctx.model,\n            \"provider_model\": ctx.provider_model,\n            \"provider_model_name\": ctx.provider_model_name,\n            \"invoked_model\": ctx.provider_model_name or ctx.requested_model or ctx.model,\n            \"api_key_id\": ctx.api_key_id,\n            \"quota_scope\": ctx.quota_scope,\n            \"reserved_tpm\": ctx.reserved_tpm,\n        }\n        \n        # Add optional fields\n        for key, value in kwargs.items():\n            if value is not None:\n                if hasattr(value, \"__dict__\"):\n                    log_data[key] = value.__dict__\n                else:\n                    log_data[key] = value\n        \n        logger.info(json.dumps(log_data, ensure_ascii=False, default=str))\n", "exceptions.py": "\"\"\"Custom exceptions for Google AI SDK.\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\n\n@dataclass\nclass RateLimitError(Exception):\n    \"\"\"Raised when rate limits are exceeded.\n    \n    NO_WAIT policy: this error is raised immediately without waiting.\n    \"\"\"\n    blocked_reason: str  # 'rpm' | 'tpm' | 'rpd'\n    retry_after_ms: Optional[int] = None\n    model: Optional[str] = None\n    api_key_id: Optional[str] = None\n    minute_bucket: Optional[str] = None\n    day_bucket: Optional[str] = None\n    \n    def __str__(self) -> str:\n        msg = f\"Rate limit exceeded: {self.blocked_reason}\"\n        if self.retry_after_ms:\n            msg += f\" (retry after {self.retry_after_ms}ms)\"\n        return msg\n\n\n@dataclass\nclass ProviderError(Exception):\n    \"\"\"Raised when Google AI provider returns an error.\n    \n    Retryable errors will be retried up to 3 times.\n    \"\"\"\n    error_type: str\n    error_code: Optional[str] = None\n    error_message: Optional[str] = None\n    retryable: bool = False\n    status_code: Optional[int] = None\n    retry_after_ms: Optional[int] = None\n    \n    def __str__(self) -> str:\n        msg = f\"Provider error: {self.error_type}\"\n        if self.error_code:\n            msg += f\" ({self.error_code})\"\n        if self.error_message:\n            msg += f\": {self.error_message}\"\n        if self.retry_after_ms:\n            msg += f\" (retry after {self.retry_after_ms}ms)\"\n        return msg\n\n\nclass SecretsError(Exception):\n    \"\"\"Raised when secrets cannot be retrieved or decrypted.\"\"\"\n    pass\n\n\nclass ReservationError(Exception):\n    \"\"\"Raised when rate limit reservation fails unexpectedly.\"\"\"\n    pass\n", "interactions.py": "\"\"\"Strict async REST client for Google's managed-agent Interactions API.\n\nThe adapter intentionally does not use the GenerateContent SDK path.  Every\ninteraction-creating POST owns a distinct shared-ledger lease; GET polling and\nenvironment snapshot downloads do not consume an interaction RPD reservation.\nProvider terminal state and downstream semantic validity are separate concepts.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport io\nimport json\nimport os\nimport re\nimport ssl\nimport tarfile\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\nfrom dataclasses import dataclass, field\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any, Mapping, Optional, Protocol, Sequence\n\nfrom google_ai.client import ExternalCallLease, GoogleAIClient, UsageInfo\nfrom google_ai.exceptions import ProviderError, ReservationError\n\n\nANTIGRAVITY_AGENT = \"antigravity-preview-05-2026\"\nINTERACTIONS_API_REVISION = \"2026-05-20\"\nDEFAULT_BASE_URL = \"https://generativelanguage.googleapis.com/v1beta\"\n\nINTERACTION_STATUSES = frozenset(\n    {\n        \"queued\",\n        \"in_progress\",\n        \"requires_action\",\n        \"completed\",\n        \"failed\",\n        \"cancelled\",\n        \"incomplete\",\n        \"budget_exceeded\",\n    }\n)\nACTIVE_INTERACTION_STATUSES = frozenset({\"queued\", \"in_progress\"})\nTERMINAL_INTERACTION_STATUSES = INTERACTION_STATUSES - ACTIVE_INTERACTION_STATUSES\n\n_RESOURCE_ID_RE = re.compile(r\"^[A-Za-z0-9._-]{1,512}$\")\n\n\nclass InteractionsProtocolError(ProviderError):\n    \"\"\"The provider response did not satisfy the documented schema.\"\"\"\n\n\nclass InteractionDeadlineExceeded(TimeoutError):\n    \"\"\"A background interaction did not stop before the caller deadline.\"\"\"\n\n    def __init__(\n        self,\n        interaction_id: str,\n        *,\n        cancel_result: Optional[\"ProviderInteraction\"] = None,\n        cancel_error: Optional[Exception] = None,\n    ) -> None:\n        super().__init__(f\"interaction deadline exceeded: {interaction_id}\")\n        self.interaction_id = interaction_id\n        self.cancel_result = cancel_result\n        self.cancel_error = cancel_error\n\n\n@dataclass(frozen=True)\nclass HTTPResponse:\n    status: int\n    headers: Mapping[str, str]\n    body: bytes\n\n\nclass AsyncHTTPTransport(Protocol):\n    async def request(\n        self,\n        method: str,\n        url: str,\n        *,\n        headers: Mapping[str, str],\n        body: Optional[bytes],\n        timeout_seconds: float,\n        max_response_bytes: int,\n    ) -> HTTPResponse: ...\n\n\nclass _SafeGoogleRedirectHandler(urllib.request.HTTPRedirectHandler):\n    \"\"\"Follow only HTTPS Google download redirects and never forward the key.\"\"\"\n\n    _ALLOWED_SUFFIXES = (\".googleapis.com\", \".googleusercontent.com\")\n\n    def redirect_request(self, req, fp, code, msg, headers, newurl):  # type: ignore[no-untyped-def]\n        old_host = (urllib.parse.urlsplit(req.full_url).hostname or \"\").lower()\n        parsed = urllib.parse.urlsplit(newurl)\n        new_host = (parsed.hostname or \"\").lower()\n        if parsed.scheme != \"https\" or not any(\n            new_host == suffix[1:] or new_host.endswith(suffix)\n            for suffix in self._ALLOWED_SUFFIXES\n        ):\n            raise urllib.error.HTTPError(\n                newurl,\n                code,\n                \"unsafe redirect target\",\n                headers,\n                fp,\n            )\n        redirected = super().redirect_request(req, fp, code, msg, headers, newurl)\n        if redirected is not None and new_host != old_host:\n            redirected.headers.pop(\"X-goog-api-key\", None)\n            redirected.unredirected_hdrs.pop(\"X-goog-api-key\", None)\n        return redirected\n\n\nclass UrllibAsyncHTTPTransport:\n    \"\"\"Dependency-free async transport backed by ``asyncio.to_thread``.\"\"\"\n\n    def __init__(self) -> None:\n        context = ssl.create_default_context()\n        self._opener = urllib.request.build_opener(\n            urllib.request.HTTPSHandler(context=context),\n            _SafeGoogleRedirectHandler(),\n        )\n\n    async def request(\n        self,\n        method: str,\n        url: str,\n        *,\n        headers: Mapping[str, str],\n        body: Optional[bytes],\n        timeout_seconds: float,\n        max_response_bytes: int,\n    ) -> HTTPResponse:\n        return await asyncio.to_thread(\n            self._request_sync,\n            method,\n            url,\n            dict(headers),\n            body,\n            timeout_seconds,\n            max_response_bytes,\n        )\n\n    def _request_sync(\n        self,\n        method: str,\n        url: str,\n        headers: dict[str, str],\n        body: Optional[bytes],\n        timeout_seconds: float,\n        max_response_bytes: int,\n    ) -> HTTPResponse:\n        request = urllib.request.Request(\n            url,\n            data=body,\n            headers=headers,\n            method=method,\n        )\n        try:\n            response = self._opener.open(request, timeout=timeout_seconds)\n            with response:\n                payload = response.read(max_response_bytes + 1)\n                if len(payload) > max_response_bytes:\n                    raise ProviderError(\n                        error_type=\"response_too_large\",\n                        error_message=\"provider response exceeded the configured byte limit\",\n                    )\n                return HTTPResponse(\n                    status=int(response.status),\n                    headers={str(k): str(v) for k, v in response.headers.items()},\n                    body=payload,\n                )\n        except urllib.error.HTTPError as exc:\n            payload = exc.read(min(max_response_bytes, 64 * 1024))\n            return HTTPResponse(\n                status=int(exc.code),\n                headers={str(k): str(v) for k, v in (exc.headers or {}).items()},\n                body=payload,\n            )\n\n\n@dataclass(frozen=True)\nclass ProviderInteraction:\n    \"\"\"One provider response; this object makes no semantic-quality claim.\"\"\"\n\n    id: str\n    provider_status: str\n    environment_id: Optional[str]\n    steps: tuple[dict[str, Any], ...]\n    usage: UsageInfo\n    raw: dict[str, Any] = field(repr=False, compare=False)\n    lease: ExternalCallLease = field(repr=False, compare=False)\n\n    @property\n    def status(self) -> str:\n        \"\"\"Compatibility alias that still names the provider status only.\"\"\"\n\n        return self.provider_status\n\n    @property\n    def is_active(self) -> bool:\n        return self.provider_status in ACTIVE_INTERACTION_STATUSES\n\n    @property\n    def is_terminal(self) -> bool:\n        return self.provider_status in TERMINAL_INTERACTION_STATUSES\n\n    @property\n    def output_text(self) -> str:\n        pieces: list[str] = []\n        for step in self.steps:\n            if step.get(\"type\") != \"model_output\":\n                continue\n            for content in step.get(\"content\") or []:\n                if isinstance(content, dict) and content.get(\"type\") == \"text\":\n                    text = content.get(\"text\")\n                    if isinstance(text, str):\n                        pieces.append(text)\n        return \"\".join(pieces)\n\n    def to_checkpoint(self) -> dict[str, Any]:\n        \"\"\"Minimal JSON-safe state needed to resume polling after a restart.\"\"\"\n\n        return {\n            \"id\": self.id,\n            \"provider_status\": self.provider_status,\n            \"environment_id\": self.environment_id,\n            \"lease\": self.lease.to_dict(),\n        }\n\n    @classmethod\n    def from_checkpoint(cls, value: Mapping[str, Any]) -> \"ProviderInteraction\":\n        \"\"\"Restore a pollable handle; the next GET refreshes steps and usage.\"\"\"\n\n        status = str(value[\"provider_status\"])\n        if status not in INTERACTION_STATUSES:\n            raise ValueError(\"checkpoint contains an unknown provider status\")\n        lease_raw = value.get(\"lease\")\n        if not isinstance(lease_raw, dict):\n            raise ValueError(\"checkpoint lease must be an object\")\n        environment_id = value.get(\"environment_id\")\n        interaction_id = str(value[\"id\"])\n        if not _RESOURCE_ID_RE.fullmatch(interaction_id):\n            raise ValueError(\"checkpoint contains an invalid interaction id\")\n        if environment_id is not None and not _RESOURCE_ID_RE.fullmatch(\n            str(environment_id)\n        ):\n            raise ValueError(\"checkpoint contains an invalid environment id\")\n        return cls(\n            id=interaction_id,\n            provider_status=status,\n            environment_id=(\n                str(environment_id) if environment_id is not None else None\n            ),\n            steps=(),\n            usage=UsageInfo(),\n            raw={},\n            lease=ExternalCallLease.from_dict(lease_raw),\n        )\n\n\nclass AntigravityInteractionsClient:\n    \"\"\"Quota-accounted client for ``antigravity-preview-05-2026``.\n\n    ``key_envs`` is mandatory.  The pool is never widened to a default key and\n    never spills into the generic overflow/fallback paths in ``GoogleAIClient``.\n    \"\"\"\n\n    def __init__(\n        self,\n        rate_limiter: GoogleAIClient,\n        *,\n        key_envs: Sequence[str],\n        transport: Optional[AsyncHTTPTransport] = None,\n        base_url: str = DEFAULT_BASE_URL,\n        request_timeout_seconds: float = 60.0,\n        poll_interval_seconds: float = 5.0,\n        max_json_response_bytes: int = 16 * 1024 * 1024,\n        max_snapshot_bytes: int = 256 * 1024 * 1024,\n        max_snapshot_unpacked_bytes: int = 512 * 1024 * 1024,\n        max_snapshot_members: int = 20_000,\n        cancel_path_style: str = \"path\",\n    ) -> None:\n        normalized = GoogleAIClient._normalize_overflow_envs(key_envs)\n        if not normalized:\n            raise ValueError(\"key_envs must be an explicit non-empty pool\")\n        if not base_url.startswith(\"https://\"):\n            raise ValueError(\"base_url must use HTTPS\")\n        if cancel_path_style not in {\"path\", \"colon\"}:\n            raise ValueError(\"cancel_path_style must be path or colon\")\n        self.rate_limiter = rate_limiter\n        self.key_envs = tuple(normalized)\n        self.transport = transport or UrllibAsyncHTTPTransport()\n        self.base_url = base_url.rstrip(\"/\")\n        self.request_timeout_seconds = max(0.1, float(request_timeout_seconds))\n        self.poll_interval_seconds = max(0.0, float(poll_interval_seconds))\n        self.max_json_response_bytes = max(1024, int(max_json_response_bytes))\n        self.max_snapshot_bytes = max(1024, int(max_snapshot_bytes))\n        self.max_snapshot_unpacked_bytes = max(\n            self.max_snapshot_bytes,\n            int(max_snapshot_unpacked_bytes),\n        )\n        self.max_snapshot_members = max(1, int(max_snapshot_members))\n        self.cancel_path_style = cancel_path_style\n\n    async def create(\n        self,\n        input: Any,\n        *,\n        max_total_tokens: int,\n        environment: Any = \"remote\",\n        tools: Optional[Sequence[Mapping[str, Any]]] = None,\n        system_instruction: Optional[str] = None,\n    ) -> ProviderInteraction:\n        \"\"\"Create a stored background interaction using one new RPD lease.\"\"\"\n\n        body = self._create_body(\n            input=input,\n            max_total_tokens=max_total_tokens,\n            environment=environment,\n            tools=tools,\n            system_instruction=system_instruction,\n        )\n        return await self._post_interaction(body, key_envs=self.key_envs)\n\n    async def continue_interaction(\n        self,\n        previous: ProviderInteraction,\n        input: Any = \"continue\",\n        *,\n        max_total_tokens: int,\n        tools: Optional[Sequence[Mapping[str, Any]]] = None,\n        system_instruction: Optional[str] = None,\n    ) -> ProviderInteraction:\n        \"\"\"Continue in the exact previous interaction and sandbox environment.\"\"\"\n\n        if previous.is_active:\n            raise ValueError(\"cannot continue an active interaction\")\n        if not previous.environment_id:\n            raise ValueError(\"previous interaction has no environment_id\")\n        body = self._create_body(\n            input=input,\n            max_total_tokens=max_total_tokens,\n            environment=previous.environment_id,\n            tools=tools,\n            system_instruction=system_instruction,\n        )\n        body[\"previous_interaction_id\"] = previous.id\n        # An environment belongs to the key/project that provisioned it.  Do not\n        # rotate a continuation onto another pool member.\n        return await self._post_interaction(\n            body,\n            key_envs=(previous.lease.env_var_name,),\n        )\n\n    async def get(self, interaction: ProviderInteraction) -> ProviderInteraction:\n        \"\"\"Poll one interaction without reserving RPM/TPM/RPD in the model ledger.\"\"\"\n\n        interaction_id = self._resource_id(interaction.id, \"interaction_id\")\n        api_key = self.rate_limiter.get_external_call_api_key(interaction.lease)\n        try:\n            payload = await self._request_json(\n                \"GET\",\n                f\"{self.base_url}/interactions/{interaction_id}\",\n                api_key=api_key,\n            )\n        except ProviderError as exc:\n            # A non-retryable poll rejection (notably an account/project 403)\n            # is terminal for this handle. Reconcile the original reservation\n            # rather than leaving it indefinitely in sent/in_progress state.\n            if not exc.retryable:\n                await self.rate_limiter.finalize_external_call(\n                    interaction.lease,\n                    provider_interaction_id=interaction.id,\n                    provider_terminal_status=\"failed\",\n                    usage=None,\n                    duration_ms=max(\n                        0,\n                        int((time.time() - interaction.lease.started_at.timestamp()) * 1000),\n                    ),\n                    semantic_status=\"not_evaluated\",\n                    error=exc,\n                )\n            raise\n        result = self._parse_interaction(payload, interaction.lease)\n        await self._finalize_if_terminal(result)\n        return result\n\n    async def wait(\n        self,\n        interaction: ProviderInteraction,\n        *,\n        deadline_seconds: float,\n        cancel_on_deadline: bool = True,\n    ) -> ProviderInteraction:\n        \"\"\"Poll to a terminal/action state and optionally cancel at the deadline.\"\"\"\n\n        if deadline_seconds < 0:\n            raise ValueError(\"deadline_seconds must be non-negative\")\n        current = interaction\n        if current.is_terminal:\n            return current\n        deadline = time.monotonic() + float(deadline_seconds)\n        while current.is_active:\n            remaining = deadline - time.monotonic()\n            if remaining <= 0:\n                cancel_result: Optional[ProviderInteraction] = None\n                cancel_error: Optional[Exception] = None\n                if cancel_on_deadline:\n                    try:\n                        cancel_result = await self.cancel(current)\n                    except Exception as exc:  # deadline remains the primary error\n                        cancel_error = exc\n                raise InteractionDeadlineExceeded(\n                    current.id,\n                    cancel_result=cancel_result,\n                    cancel_error=cancel_error,\n                )\n            await asyncio.sleep(min(self.poll_interval_seconds, remaining))\n            current = await self.get(current)\n        return current\n\n    async def cancel(self, interaction: ProviderInteraction) -> ProviderInteraction:\n        \"\"\"Cancel a running interaction without consuming an interaction RPD.\n\n        The current reference uses ``/{id}/cancel``.  A 404/405 is retried once\n        with the preview guide's older ``/{id}:cancel`` spelling (or vice versa).\n        Both control-plane POSTs receive distinct request IDs.\n        \"\"\"\n\n        interaction_id = self._resource_id(interaction.id, \"interaction_id\")\n        api_key = self.rate_limiter.get_external_call_api_key(interaction.lease)\n        styles = (\n            (\"path\", \"colon\")\n            if self.cancel_path_style == \"path\"\n            else (\"colon\", \"path\")\n        )\n        last_error: Optional[ProviderError] = None\n        for index, style in enumerate(styles):\n            suffix = f\"/{interaction_id}/cancel\" if style == \"path\" else f\"/{interaction_id}:cancel\"\n            try:\n                payload = await self._request_json(\n                    \"POST\",\n                    f\"{self.base_url}/interactions{suffix}\",\n                    api_key=api_key,\n                    request_uid=str(uuid.uuid4()),\n                )\n                result = self._parse_interaction(payload, interaction.lease)\n                await self._finalize_if_terminal(result)\n                return result\n            except ProviderError as exc:\n                last_error = exc\n                if index == 0 and exc.status_code in {404, 405}:\n                    continue\n                raise\n        raise last_error or ProviderError(error_type=\"cancel_failed\")\n\n    async def download_environment(\n        self,\n        interaction: ProviderInteraction,\n        destination_tar: os.PathLike[str] | str,\n        *,\n        extract_to: Optional[os.PathLike[str] | str] = None,\n    ) -> Path:\n        \"\"\"Download an environment snapshot and optionally extract it safely.\n\n        Downloads do not reserve an interaction request.  Extraction rejects\n        absolute/traversal paths, links, devices, oversized archives, and member\n        explosions before writing any archive member.\n        \"\"\"\n\n        if not interaction.environment_id:\n            raise ValueError(\"interaction has no environment_id\")\n        environment_id = self._resource_id(\n            interaction.environment_id,\n            \"environment_id\",\n        )\n        api_key = self.rate_limiter.get_external_call_api_key(interaction.lease)\n        url = (\n            f\"{self.base_url}/files/environment-{environment_id}:download?alt=media\"\n        )\n        response = await self.transport.request(\n            \"GET\",\n            url,\n            headers=self._headers(api_key),\n            body=None,\n            timeout_seconds=self.request_timeout_seconds,\n            max_response_bytes=self.max_snapshot_bytes,\n        )\n        self._raise_for_status(response, secret=api_key)\n        if len(response.body) > self.max_snapshot_bytes:\n            raise ProviderError(\n                error_type=\"response_too_large\",\n                error_message=\"environment snapshot exceeded the configured byte limit\",\n            )\n        destination = Path(destination_tar)\n        await asyncio.to_thread(self._atomic_write, destination, response.body)\n        if extract_to is not None:\n            await asyncio.to_thread(\n                self._safe_extract_tar,\n                response.body,\n                Path(extract_to),\n            )\n        return destination\n\n    def _create_body(\n        self,\n        *,\n        input: Any,\n        max_total_tokens: int,\n        environment: Any,\n        tools: Optional[Sequence[Mapping[str, Any]]],\n        system_instruction: Optional[str],\n    ) -> dict[str, Any]:\n        try:\n            token_budget = int(max_total_tokens)\n        except (TypeError, ValueError) as exc:\n            raise ValueError(\"max_total_tokens must be an integer\") from exc\n        if not 1 <= token_budget <= 100_000:\n            raise ValueError(\"max_total_tokens must be between 1 and 100000\")\n        if input is None or input == \"\":\n            raise ValueError(\"input is required\")\n        if environment is None or environment == \"\":\n            raise ValueError(\"environment is required\")\n        body: dict[str, Any] = {\n            \"agent\": ANTIGRAVITY_AGENT,\n            \"input\": input,\n            \"environment\": environment,\n            \"background\": True,\n            \"store\": True,\n            \"agent_config\": {\n                \"type\": \"antigravity\",\n                \"max_total_tokens\": token_budget,\n            },\n        }\n        if tools is not None:\n            body[\"tools\"] = [dict(tool) for tool in tools]\n        if system_instruction is not None:\n            body[\"system_instruction\"] = str(system_instruction)\n        return body\n\n    async def _post_interaction(\n        self,\n        body: dict[str, Any],\n        *,\n        key_envs: Sequence[str],\n    ) -> ProviderInteraction:\n        request_uid = str(uuid.uuid4())\n        reserved_tpm = int(body[\"agent_config\"][\"max_total_tokens\"])\n        lease = await self.rate_limiter.reserve_external_call(\n            model=ANTIGRAVITY_AGENT,\n            reserved_tpm=reserved_tpm,\n            key_envs=key_envs,\n            request_uid=request_uid,\n        )\n        started = time.monotonic()\n        try:\n            api_key = self.rate_limiter.get_external_call_api_key(lease)\n            await self.rate_limiter.mark_external_call_sent(lease)\n            payload = await self._request_json(\n                \"POST\",\n                f\"{self.base_url}/interactions\",\n                api_key=api_key,\n                request_uid=request_uid,\n                json_body=body,\n            )\n            interaction = self._parse_interaction(payload, lease)\n        except Exception as exc:\n            provider_error = self._as_provider_error(exc)\n            await self._finalize_transport_failure(\n                lease,\n                provider_error,\n                duration_ms=int((time.monotonic() - started) * 1000),\n            )\n            raise provider_error from exc\n\n        if interaction.is_terminal:\n            await self._finalize_if_terminal(\n                interaction,\n                duration_ms=int((time.monotonic() - started) * 1000),\n            )\n        return interaction\n\n    async def _finalize_transport_failure(\n        self,\n        lease: ExternalCallLease,\n        error: ProviderError,\n        *,\n        duration_ms: int,\n    ) -> None:\n        try:\n            await self.rate_limiter.finalize_external_call(\n                lease,\n                provider_interaction_id=None,\n                provider_terminal_status=\"failed\",\n                usage=None,\n                duration_ms=duration_ms,\n                semantic_status=\"not_evaluated\",\n                error=error,\n            )\n        except Exception as finalize_error:\n            raise ReservationError(\n                f\"provider call failed and accounting finalization also failed: \"\n                f\"{str(finalize_error)[:300]}\"\n            ) from error\n\n    async def _finalize_if_terminal(\n        self,\n        interaction: ProviderInteraction,\n        *,\n        duration_ms: Optional[int] = None,\n    ) -> None:\n        if not interaction.is_terminal:\n            return\n        elapsed = duration_ms\n        if elapsed is None:\n            elapsed = max(\n                0,\n                int(\n                    (\n                        time.time()\n                        - interaction.lease.started_at.timestamp()\n                    )\n                    * 1000\n                ),\n            )\n        error: Optional[ProviderError] = None\n        if interaction.provider_status in {\"failed\", \"cancelled\"}:\n            error = ProviderError(\n                error_type=f\"interaction_{interaction.provider_status}\",\n                error_message=self._interaction_error_message(interaction.raw),\n            )\n        await self.rate_limiter.finalize_external_call(\n            interaction.lease,\n            provider_interaction_id=interaction.id,\n            provider_terminal_status=interaction.provider_status,\n            usage=interaction.usage,\n            duration_ms=elapsed,\n            semantic_status=\"not_evaluated\",\n            error=error,\n        )\n\n    async def _request_json(\n        self,\n        method: str,\n        url: str,\n        *,\n        api_key: str,\n        request_uid: Optional[str] = None,\n        json_body: Optional[dict[str, Any]] = None,\n    ) -> dict[str, Any]:\n        body = None\n        if json_body is not None:\n            body = json.dumps(\n                json_body,\n                ensure_ascii=False,\n                separators=(\",\", \":\"),\n            ).encode(\"utf-8\")\n        response = await self.transport.request(\n            method,\n            url,\n            headers=self._headers(api_key, request_uid=request_uid),\n            body=body,\n            timeout_seconds=self.request_timeout_seconds,\n            max_response_bytes=self.max_json_response_bytes,\n        )\n        self._raise_for_status(response, secret=api_key)\n        if len(response.body) > self.max_json_response_bytes:\n            raise ProviderError(\n                error_type=\"response_too_large\",\n                error_message=\"Interactions API response exceeded the configured byte limit\",\n            )\n        try:\n            payload = json.loads(response.body.decode(\"utf-8\"))\n        except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n            raise InteractionsProtocolError(\n                error_type=\"invalid_json\",\n                error_message=\"Interactions API returned invalid JSON\",\n                status_code=response.status,\n            ) from exc\n        if not isinstance(payload, dict):\n            raise InteractionsProtocolError(\n                error_type=\"invalid_response\",\n                error_message=\"Interactions API response must be an object\",\n                status_code=response.status,\n            )\n        return payload\n\n    @staticmethod\n    def _headers(api_key: str, *, request_uid: Optional[str] = None) -> dict[str, str]:\n        headers = {\n            \"Accept\": \"application/json\",\n            \"Content-Type\": \"application/json\",\n            \"x-goog-api-key\": api_key,\n            \"Api-Revision\": INTERACTIONS_API_REVISION,\n        }\n        if request_uid:\n            headers[\"X-Request-Id\"] = request_uid\n        return headers\n\n    def _parse_interaction(\n        self,\n        payload: dict[str, Any],\n        lease: ExternalCallLease,\n    ) -> ProviderInteraction:\n        interaction_id = payload.get(\"id\")\n        status = str(payload.get(\"status\") or \"\").strip().lower()\n        if not isinstance(interaction_id, str) or not interaction_id:\n            raise InteractionsProtocolError(\n                error_type=\"missing_interaction_id\",\n                error_message=\"Interactions API response has no id\",\n            )\n        self._resource_id(interaction_id, \"interaction_id\")\n        if status not in INTERACTION_STATUSES:\n            raise InteractionsProtocolError(\n                error_type=\"unknown_interaction_status\",\n                error_message=f\"unknown Interactions API status: {status or '<empty>'}\",\n            )\n        environment_id = payload.get(\"environment_id\")\n        if environment_id is not None:\n            if not isinstance(environment_id, str):\n                raise InteractionsProtocolError(\n                    error_type=\"invalid_environment_id\",\n                    error_message=\"environment_id must be a string\",\n                )\n            self._resource_id(environment_id, \"environment_id\")\n        raw_steps = payload.get(\"steps\") or []\n        if not isinstance(raw_steps, list) or not all(\n            isinstance(step, dict) for step in raw_steps\n        ):\n            raise InteractionsProtocolError(\n                error_type=\"invalid_steps\",\n                error_message=\"steps must be an array of objects\",\n            )\n        usage_raw = payload.get(\"usage\") or {}\n        if not isinstance(usage_raw, dict):\n            usage_raw = {}\n        input_tokens = self._nonnegative_int(usage_raw.get(\"total_input_tokens\"))\n        output_tokens = self._nonnegative_int(usage_raw.get(\"total_output_tokens\"))\n        total_tokens = self._nonnegative_int(usage_raw.get(\"total_tokens\"))\n        if total_tokens == 0 and (input_tokens or output_tokens):\n            total_tokens = input_tokens + output_tokens\n        return ProviderInteraction(\n            id=interaction_id,\n            provider_status=status,\n            environment_id=environment_id,\n            steps=tuple(dict(step) for step in raw_steps),\n            usage=UsageInfo(\n                input_tokens=input_tokens,\n                output_tokens=output_tokens,\n                total_tokens=total_tokens,\n            ),\n            raw=dict(payload),\n            lease=lease,\n        )\n\n    @staticmethod\n    def _nonnegative_int(value: Any) -> int:\n        try:\n            return max(0, int(value or 0))\n        except (TypeError, ValueError):\n            return 0\n\n    @staticmethod\n    def _resource_id(value: str, label: str) -> str:\n        if not _RESOURCE_ID_RE.fullmatch(value):\n            raise ValueError(f\"invalid {label}\")\n        return value\n\n    @staticmethod\n    def _interaction_error_message(payload: Mapping[str, Any]) -> Optional[str]:\n        error = payload.get(\"error\")\n        if isinstance(error, dict):\n            message = error.get(\"message\")\n            if isinstance(message, str):\n                return message[:1000]\n        return None\n\n    @staticmethod\n    def _as_provider_error(exc: Exception) -> ProviderError:\n        if isinstance(exc, ProviderError):\n            return exc\n        return ProviderError(\n            error_type=exc.__class__.__name__,\n            error_message=str(exc)[:500],\n            retryable=isinstance(exc, (TimeoutError, ConnectionError)),\n        )\n\n    @staticmethod\n    def _raise_for_status(\n        response: HTTPResponse,\n        *,\n        secret: Optional[str] = None,\n    ) -> None:\n        if 200 <= response.status < 300:\n            return\n        message = \"\"\n        try:\n            payload = json.loads(response.body.decode(\"utf-8\"))\n            error = payload.get(\"error\") if isinstance(payload, dict) else None\n            if isinstance(error, dict):\n                message = str(error.get(\"message\") or error.get(\"status\") or \"\")\n            elif error:\n                message = str(error)\n        except Exception:\n            message = response.body.decode(\"utf-8\", errors=\"replace\")\n        if secret and secret in message:\n            message = message.replace(secret, \"[REDACTED]\")\n        raise ProviderError(\n            error_type=\"http_error\",\n            error_code=str(response.status),\n            error_message=(message or \"Google Interactions API request failed\")[:500],\n            retryable=response.status in {408, 429, 500, 502, 503, 504},\n            status_code=response.status,\n        )\n\n    @staticmethod\n    def _atomic_write(path: Path, content: bytes) -> None:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        temp = path.with_name(f\".{path.name}.{uuid.uuid4().hex}.tmp\")\n        try:\n            with temp.open(\"xb\") as handle:\n                handle.write(content)\n                handle.flush()\n                os.fsync(handle.fileno())\n            os.replace(temp, path)\n        finally:\n            try:\n                temp.unlink(missing_ok=True)\n            except OSError:\n                pass\n\n    def _safe_extract_tar(self, content: bytes, destination: Path) -> None:\n        destination.mkdir(parents=True, exist_ok=True)\n        destination_root = destination.resolve()\n        with tarfile.open(fileobj=io.BytesIO(content), mode=\"r:*\") as archive:\n            members = archive.getmembers()\n            if len(members) > self.max_snapshot_members:\n                raise ValueError(\"environment snapshot has too many members\")\n            total_size = 0\n            for member in members:\n                name = member.name\n                posix = PurePosixPath(name)\n                if (\n                    not name\n                    or \"\\\\\" in name\n                    or posix.is_absolute()\n                    or \"..\" in posix.parts\n                    or member.issym()\n                    or member.islnk()\n                    or not (member.isdir() or member.isreg())\n                ):\n                    raise ValueError(f\"unsafe environment snapshot member: {name!r}\")\n                target = (destination_root / Path(*posix.parts)).resolve()\n                if target != destination_root and destination_root not in target.parents:\n                    raise ValueError(f\"snapshot member escapes destination: {name!r}\")\n                total_size += max(0, int(member.size))\n                if total_size > self.max_snapshot_unpacked_bytes:\n                    raise ValueError(\"environment snapshot expands beyond the configured limit\")\n            archive.extractall(destination_root, members=members, filter=\"data\")\n\n\n__all__ = [\n    \"ACTIVE_INTERACTION_STATUSES\",\n    \"ANTIGRAVITY_AGENT\",\n    \"AntigravityInteractionsClient\",\n    \"AsyncHTTPTransport\",\n    \"HTTPResponse\",\n    \"INTERACTIONS_API_REVISION\",\n    \"INTERACTION_STATUSES\",\n    \"InteractionDeadlineExceeded\",\n    \"InteractionsProtocolError\",\n    \"ProviderInteraction\",\n    \"TERMINAL_INTERACTION_STATUSES\",\n    \"UrllibAsyncHTTPTransport\",\n]\n", "limiter_supabase.py": "\"\"\"Construction helpers for the dedicated Google AI quota ledger.\n\nThe limiter has its own Supabase project so that quota accounting is not tied\nto the bot's general-purpose Supabase project.  Callers may supply an explicit\nlegacy factory for rollout compatibility, but dedicated configuration always\nwins and partial dedicated configuration is an error.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom collections.abc import Callable, Mapping\nfrom typing import Any\nfrom urllib.parse import urlsplit\n\n\nGOOGLE_AI_LIMITER_SUPABASE_URL_ENV = \"GOOGLE_AI_LIMITER_SUPABASE_URL\"\nGOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV = (\n    \"GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY\"\n)\nGOOGLE_AI_LIMITER_ALLOW_LEGACY_FALLBACK_ENV = (\n    \"GOOGLE_AI_LIMITER_ALLOW_LEGACY_FALLBACK\"\n)\n\n\nclass GoogleAILimiterSupabaseConfigurationError(RuntimeError):\n    \"\"\"Raised before provider use when the limiter backend is misconfigured.\"\"\"\n\n\ndef _validated_url(raw_url: str) -> str:\n    url = raw_url.strip().rstrip(\"/\")\n    parsed = urlsplit(url)\n    if (\n        parsed.scheme not in {\"http\", \"https\"}\n        or not parsed.hostname\n        or parsed.username is not None\n        or parsed.password is not None\n        or parsed.path not in {\"\", \"/\"}\n        or parsed.query\n        or parsed.fragment\n    ):\n        raise GoogleAILimiterSupabaseConfigurationError(\n            f\"{GOOGLE_AI_LIMITER_SUPABASE_URL_ENV} must be an http(s) origin\"\n        )\n    return url\n\n\ndef build_google_ai_limiter_supabase_client(\n    *,\n    fallback_factory: Callable[[], Any | None] | None = None,\n    require_configured: bool = False,\n    environ: Mapping[str, str] | None = None,\n    client_factory: Callable[[str, str], Any] | None = None,\n) -> Any | None:\n    \"\"\"Build the Supabase client used only for Google AI quota accounting.\n\n    ``GOOGLE_AI_LIMITER_SUPABASE_URL`` and\n    ``GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY`` are an atomic pair.  A legacy\n    factory is accepted only when the caller explicitly opts into the\n    local-development compatibility flag.  Production and remote runtimes must\n    never silently move quota accounting to another Supabase project.\n    \"\"\"\n\n    source = os.environ if environ is None else environ\n    raw_url = str(source.get(GOOGLE_AI_LIMITER_SUPABASE_URL_ENV, \"\") or \"\").strip()\n    service_key = str(\n        source.get(GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV, \"\") or \"\"\n    ).strip()\n\n    if bool(raw_url) != bool(service_key):\n        missing = (\n            GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV\n            if raw_url\n            else GOOGLE_AI_LIMITER_SUPABASE_URL_ENV\n        )\n        raise GoogleAILimiterSupabaseConfigurationError(\n            f\"dedicated Google AI limiter configuration is incomplete: missing {missing}\"\n        )\n\n    if not raw_url:\n        allow_legacy_fallback = str(\n            source.get(GOOGLE_AI_LIMITER_ALLOW_LEGACY_FALLBACK_ENV, \"\") or \"\"\n        ).strip().lower() in {\"1\", \"true\", \"yes\", \"on\"}\n        if fallback_factory is not None and allow_legacy_fallback and not require_configured:\n            client = fallback_factory()\n            if client is not None:\n                return client\n        if require_configured:\n            raise GoogleAILimiterSupabaseConfigurationError(\n                \"Google AI limiter Supabase is not configured\"\n            )\n        return None\n\n    url = _validated_url(raw_url)\n    factory = client_factory\n    if factory is None:\n        try:\n            from supabase import create_client\n        except Exception as exc:  # pragma: no cover - deployment dependency guard\n            raise GoogleAILimiterSupabaseConfigurationError(\n                \"supabase package is unavailable for the dedicated Google AI limiter\"\n            ) from exc\n        factory = create_client\n\n    try:\n        client = factory(url, service_key)\n    except Exception as exc:\n        raise GoogleAILimiterSupabaseConfigurationError(\n            \"failed to construct the dedicated Google AI limiter Supabase client\"\n        ) from exc\n    if client is None:\n        raise GoogleAILimiterSupabaseConfigurationError(\n            \"dedicated Google AI limiter Supabase client factory returned no client\"\n        )\n    return client\n\n\n__all__ = [\n    \"GOOGLE_AI_LIMITER_SUPABASE_SERVICE_KEY_ENV\",\n    \"GOOGLE_AI_LIMITER_SUPABASE_URL_ENV\",\n    \"GOOGLE_AI_LIMITER_ALLOW_LEGACY_FALLBACK_ENV\",\n    \"GoogleAILimiterSupabaseConfigurationError\",\n    \"build_google_ai_limiter_supabase_client\",\n]\n", "secrets.py": "\"\"\"Secrets provider with fallback chain.\n\nFallback order:\n1. Environment variables (os.getenv)\n2. Kaggle Secrets (kaggle_secrets.UserSecretsClient)\n3. Encrypted datasets (Fernet + 2 private Kaggle datasets)\n\nBased on existing implementation in kaggle/UniversalFestivalParser/src/secrets.py\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom google_ai.exceptions import SecretsError\n\nlogger = logging.getLogger(__name__)\n\n\nclass SecretsProvider:\n    \"\"\"Unified secrets provider with fallback chain.\n    \n    Supports:\n    - Single secrets: get_secret(\"GOOGLE_API_KEY\")\n    - Secret pools: get_secret_pool(\"GOOGLE_API_KEY\") -> [\"key1\", \"key2\", ...]\n    \n    For pools, looks for GOOGLE_API_KEY, GOOGLE_API_KEY_2, GOOGLE_API_KEY_3, etc.\n    \"\"\"\n    \n    def __init__(\n        self,\n        cipher_dataset_path: str = \"/kaggle/input/eve-secrets-cipher\",\n        key_dataset_path: str = \"/kaggle/input/eve-secrets-key\",\n        # Legacy paths for backward compatibility\n        legacy_cipher_path: str = \"/kaggle/input/gemma-cipher\",\n        legacy_key_path: str = \"/kaggle/input/gemma-key\",\n    ):\n        self.cipher_dataset_path = cipher_dataset_path\n        self.key_dataset_path = key_dataset_path\n        self.legacy_cipher_path = legacy_cipher_path\n        self.legacy_key_path = legacy_key_path\n        self._cache: dict[str, str] = {}\n        self._bundle_loaded = False\n    \n    def get_secret(self, name: str) -> Optional[str]:\n        \"\"\"Get a secret by name.\n        \n        Fallback order:\n        1. Environment variable\n        2. Kaggle Secrets\n        3. Encrypted bundle from datasets\n        \n        Returns None if not found.\n        \"\"\"\n        # Check cache first\n        if name in self._cache:\n            return self._cache[name]\n        \n        # 1. Try environment variable\n        value = os.getenv(name)\n        if value:\n            logger.debug(\"Secret %s found in environment\", name)\n            self._cache[name] = value\n            return value\n        \n        # 2. Try Kaggle Secrets\n        value = self._get_from_kaggle_secrets(name)\n        if value:\n            logger.debug(\"Secret %s found in Kaggle Secrets\", name)\n            self._cache[name] = value\n            return value\n        \n        # 3. Try encrypted datasets\n        value = self._get_from_encrypted_bundle(name)\n        if value:\n            logger.debug(\"Secret %s found in encrypted bundle\", name)\n            self._cache[name] = value\n            return value\n        \n        logger.warning(\"Secret %s not found in any source\", name)\n        return None\n    \n    def get_secret_pool(self, prefix: str) -> list[str]:\n        \"\"\"Get all secrets matching a prefix pattern.\n        \n        For prefix \"GOOGLE_API_KEY\", returns values of:\n        - GOOGLE_API_KEY\n        - GOOGLE_API_KEY_2\n        - GOOGLE_API_KEY_3\n        - etc.\n        \n        Returns list of values (not keys).\n        \"\"\"\n        values = []\n        \n        # First key (no suffix)\n        value = self.get_secret(prefix)\n        if value:\n            values.append(value)\n        \n        # Numbered keys\n        for i in range(2, 100):  # Support up to 99 keys\n            value = self.get_secret(f\"{prefix}_{i}\")\n            if value:\n                values.append(value)\n            else:\n                break  # Stop on first missing\n        \n        return values\n    \n    def _get_from_kaggle_secrets(self, name: str) -> Optional[str]:\n        \"\"\"Try to get secret from Kaggle Secrets API.\"\"\"\n        try:\n            from kaggle_secrets import UserSecretsClient\n            secrets = UserSecretsClient()\n            value = secrets.get_secret(name)\n            if value:\n                return value\n        except ImportError:\n            logger.debug(\"kaggle_secrets not available\")\n        except Exception as e:\n            logger.debug(\"Kaggle Secrets error for %s: %s\", name, e)\n        return None\n    \n    def _get_from_encrypted_bundle(self, name: str) -> Optional[str]:\n        \"\"\"Try to get secret from encrypted dataset bundle.\"\"\"\n        if not self._bundle_loaded:\n            self._load_bundle()\n        return self._cache.get(name)\n    \n    def _load_bundle(self) -> None:\n        \"\"\"Load and decrypt secrets bundle from datasets.\"\"\"\n        self._bundle_loaded = True\n        \n        # Try new bundle format first\n        bundle = self._try_load_bundle(\n            self.cipher_dataset_path,\n            self.key_dataset_path,\n            bundle_file=\"secrets.enc\",\n            key_file=\"fernet.keys\",\n        )\n        \n        if bundle:\n            self._cache.update(bundle)\n            return\n        \n        # Try legacy format (single key file)\n        legacy_key = self._try_load_legacy_key()\n        if legacy_key:\n            self._cache[\"GOOGLE_API_KEY\"] = legacy_key\n    \n    def _try_load_bundle(\n        self,\n        cipher_path: str,\n        key_path: str,\n        bundle_file: str,\n        key_file: str,\n    ) -> Optional[dict[str, str]]:\n        \"\"\"Try to load and decrypt a secrets bundle.\"\"\"\n        cipher_file = Path(cipher_path) / bundle_file\n        keys_file = Path(key_path) / key_file\n        \n        if not cipher_file.exists() or not keys_file.exists():\n            return None\n        \n        try:\n            from cryptography.fernet import Fernet, MultiFernet\n            \n            # Load key ring (multiple Fernet keys for rotation)\n            key_lines = keys_file.read_text().strip().split(\"\\n\")\n            fernets = [Fernet(k.strip().encode()) for k in key_lines if k.strip()]\n            \n            if not fernets:\n                logger.warning(\"No valid Fernet keys in %s\", keys_file)\n                return None\n            \n            multi_fernet = MultiFernet(fernets)\n            \n            # Decrypt bundle\n            encrypted = cipher_file.read_bytes()\n            decrypted = multi_fernet.decrypt(encrypted)\n            bundle = json.loads(decrypted.decode(\"utf-8\"))\n            \n            logger.info(\"Loaded %d secrets from encrypted bundle\", len(bundle))\n            return bundle\n            \n        except ImportError:\n            logger.warning(\"cryptography package not installed\")\n        except Exception as e:\n            logger.error(\"Failed to decrypt bundle: %s\", e)\n        \n        return None\n    \n    def _try_load_legacy_key(self) -> Optional[str]:\n        \"\"\"Try to load single key from legacy dataset format.\"\"\"\n        cipher_file = Path(self.legacy_cipher_path) / \"google_api_key.enc\"\n        key_file = Path(self.legacy_key_path) / \"fernet.key\"\n        \n        if not cipher_file.exists() or not key_file.exists():\n            return None\n        \n        try:\n            from cryptography.fernet import Fernet\n            \n            fernet_key = key_file.read_bytes().strip()\n            fernet = Fernet(fernet_key)\n            \n            encrypted = cipher_file.read_bytes()\n            decrypted = fernet.decrypt(encrypted).decode(\"utf-8\").strip()\n            \n            # Validate Google API key format\n            if not decrypted.startswith(\"AIza\"):\n                logger.warning(\"Decrypted key has unexpected format\")\n            \n            logger.info(\"Loaded API key from legacy encrypted datasets\")\n            return decrypted\n            \n        except ImportError:\n            logger.warning(\"cryptography package not installed\")\n        except Exception as e:\n            logger.error(\"Failed to decrypt legacy key: %s\", e)\n        \n        return None\n\n\n# Module-level singleton\n_provider: Optional[SecretsProvider] = None\n\n\ndef get_provider() -> SecretsProvider:\n    \"\"\"Get or create the global secrets provider.\"\"\"\n    global _provider\n    if _provider is None:\n        _provider = SecretsProvider()\n    return _provider\n\n\ndef get_secret(name: str) -> Optional[str]:\n    \"\"\"Get a secret by name (module-level convenience function).\"\"\"\n    return get_provider().get_secret(name)\n\n\ndef get_secret_pool(prefix: str) -> list[str]:\n    \"\"\"Get all secrets matching a prefix (module-level convenience function).\"\"\"\n    return get_provider().get_secret_pool(prefix)\n\n\ndef create_encrypted_bundle(\n    secrets: dict[str, str],\n    output_dir: str | Path,\n    cipher_filename: str = \"secrets.enc\",\n    key_filename: str = \"fernet.keys\",\n) -> tuple[Path, Path]:\n    \"\"\"Create encrypted bundle files for Kaggle datasets.\n    \n    This is a helper function for setting up the datasets.\n    Run this locally to generate the files to upload.\n    \n    Args:\n        secrets: Dict of secret_name -> secret_value\n        output_dir: Directory to save files\n        cipher_filename: Name of cipher file\n        key_filename: Name of key file\n        \n    Returns:\n        Tuple of (cipher_file_path, key_file_path)\n    \"\"\"\n    from cryptography.fernet import Fernet\n    \n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    \n    # Generate new Fernet key\n    fernet_key = Fernet.generate_key()\n    fernet = Fernet(fernet_key)\n    \n    # Encrypt bundle as JSON\n    bundle_json = json.dumps(secrets, ensure_ascii=False)\n    encrypted = fernet.encrypt(bundle_json.encode(\"utf-8\"))\n    \n    # Save files\n    cipher_path = output_dir / cipher_filename\n    key_path = output_dir / key_filename\n    \n    cipher_path.write_bytes(encrypted)\n    key_path.write_text(fernet_key.decode(\"utf-8\"))\n    \n    print(f\"Created {cipher_path} (upload to eve-secrets-cipher dataset)\")\n    print(f\"Created {key_path} (upload to eve-secrets-key dataset)\")\n    print(\"IMPORTANT: Keep these datasets PRIVATE!\")\n    \n    return cipher_path, key_path\n"}
_GUIDE_EMBEDDED_ROOT = (_GuideNotebookPath.cwd() / 'embedded_repo_bundle').resolve()
_GUIDE_EMBEDDED_PACKAGE = _GUIDE_EMBEDDED_ROOT / 'google_ai'
_GUIDE_EMBEDDED_PACKAGE.mkdir(parents=True, exist_ok=True)
for _guide_name, _guide_body in _GUIDE_EMBEDDED_GOOGLE_AI.items():
    _guide_target = _GUIDE_EMBEDDED_PACKAGE / _guide_name
    _guide_target.parent.mkdir(parents=True, exist_ok=True)
    _guide_target.write_text(_guide_body, encoding='utf-8')
if str(_GUIDE_EMBEDDED_ROOT) not in _GuideNotebookSys.path:
    _GuideNotebookSys.path.insert(0, str(_GUIDE_EMBEDDED_ROOT))
__file__ = str((_GuideNotebookPath.cwd() / 'guide_excursions_monitor.py').resolve())

import asyncio
import base64
import hashlib
import html
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import urllib.request
import zipfile
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_REPO = WORK_DIR / "repo_bundle"
RESULT_PATH = WORK_DIR / "guide_excursions_results.json"
MEDIA_OUTPUT_DIR = WORK_DIR / "guide_media"
REPO_BOOTSTRAP_WAIT_SECONDS = max(0, int(os.getenv("GUIDE_MONITORING_REPO_BOOTSTRAP_WAIT_SECONDS", "15") or 15))
GUIDE_MEDIA_OUTPUT_LIMIT_PER_POST = max(
    1,
    min(int(os.getenv("GUIDE_MEDIA_OUTPUT_LIMIT_PER_POST", "6") or 6), 10),
)
GUIDE_MEDIA_OUTPUT_MAX_MB = max(
    1,
    min(int(os.getenv("GUIDE_MEDIA_OUTPUT_MAX_MB", "20") or 20), 50),
)

SCRIPT_DIR = Path(__file__).resolve().parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

def _load_status_loader():
    try:
        from kaggle_status_client import load_status_client as loader
        return loader
    except Exception as exc:
        print(f"[kaggle_status] import failed: {exc}", flush=True)
    for root in [SCRIPT_DIR, Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        candidates = [root / "kaggle_status_client.py"]
        try:
            candidates.extend(sorted(root.rglob("kaggle_status_client.py")))
        except Exception:
            pass
        for candidate in candidates:
            if not candidate.exists():
                continue
            try:
                spec = importlib.util.spec_from_file_location("events_bot_kaggle_status_client", candidate)
                if spec and spec.loader:
                    module = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(module)
                    print(f"[kaggle_status] loaded helper from {candidate}", flush=True)
                    return module.load_status_client
            except Exception as exc:
                print(f"[kaggle_status] helper load failed from {candidate}: {exc}", flush=True)
    return None


load_status_client = _load_status_loader()

STATUS_PROGRESS: dict[str, object] = {"phase": "bootstrap"}
STATUS_CLIENT = load_status_client(log=lambda message: print(message, flush=True)) if load_status_client else None


def _status_event(event: str, *, phase: str | None = None, status: str | None = None, progress: dict | None = None, message: str | None = None) -> None:
    if STATUS_CLIENT is None:
        return
    try:
        STATUS_CLIENT.event(
            event,
            phase=phase,
            status=status,
            progress=progress,
            message=message,
        )
    except Exception:
        print(f"[kaggle_status] failed to emit {event}", flush=True)


def _status_progress() -> dict[str, object]:
    return dict(STATUS_PROGRESS)


def ensure_libs() -> None:
    modules = [
        ("telethon", "telethon"),
        ("google.generativeai", "google-generativeai"),
        ("cryptography", "cryptography"),
        ("supabase", "supabase"),
        ("nest_asyncio", "nest_asyncio"),
    ]
    missing: list[str] = []
    for module_name, package_name in modules:
        try:
            __import__(module_name)
        except Exception:
            missing.append(package_name)
    if missing:
        print(f"Installing Python packages: {', '.join(missing)}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Python packages already available: telethon, google-generativeai, cryptography, supabase", flush=True)


ensure_libs()


def _bootstrap_repo_bundle() -> None:
    bundled_package = SCRIPT_DIR / "google_ai" / "__init__.py"
    if bundled_package.exists():
        print(f"Using bundled google_ai from {bundled_package.parent}", flush=True)
        return
    try:
        if importlib.util.find_spec("google_ai") is not None:
            print("Using google_ai already available on kernel path", flush=True)
            return
    except Exception:
        pass
    deadline = time.monotonic() + REPO_BOOTSTRAP_WAIT_SECONDS
    last_snapshot: list[str] = []
    while True:
        repo_zip_path: Path | None = None
        repo_tree_root: Path | None = None
        flat_repo_root: Path | None = None
        snapshot: list[str] = []
        for path in INPUT_ROOT.rglob("*"):
            if path.is_file():
                snapshot.append(str(path.relative_to(INPUT_ROOT)))
                if len(snapshot) >= 40:
                    break
        last_snapshot = snapshot
        for path in INPUT_ROOT.rglob("repo_bundle.zip"):
            if path.is_file():
                repo_zip_path = path
                break
        if repo_zip_path is None:
            for init_path in INPUT_ROOT.rglob("__init__.py"):
                if init_path.parent.name == "google_ai":
                    repo_tree_root = init_path.parent.parent
                    break
                parent = init_path.parent
                if all((parent / name).is_file() for name in ("__init__.py", "client.py", "exceptions.py", "secrets.py")):
                    flat_repo_root = parent
                    break
        if repo_zip_path is not None:
            if WORK_REPO.exists():
                shutil.rmtree(WORK_REPO)
            WORK_REPO.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(repo_zip_path) as zip_file:
                zip_file.extractall(WORK_REPO)
            sys.path.insert(0, str(WORK_REPO))
            print(f"Bootstrapped repo bundle from {repo_zip_path}", flush=True)
            return
        if repo_tree_root is not None:
            sys.path.insert(0, str(repo_tree_root))
            print(f"Bootstrapped repo bundle from {repo_tree_root}", flush=True)
            return
        if flat_repo_root is not None:
            if WORK_REPO.exists():
                shutil.rmtree(WORK_REPO)
            package_root = WORK_REPO / "google_ai"
            package_root.mkdir(parents=True, exist_ok=True)
            for source_path in sorted(flat_repo_root.glob("*.py")):
                if source_path.is_file():
                    shutil.copy2(source_path, package_root / source_path.name)
            sys.path.insert(0, str(WORK_REPO))
            print(f"Bootstrapped flat repo bundle from {flat_repo_root}", flush=True)
            return
        if time.monotonic() >= deadline:
            print(
                (
                    "Repo bundle not found under /kaggle/input; relying on kernel sources only "
                    f"visible={last_snapshot} wait_s={REPO_BOOTSTRAP_WAIT_SECONDS}"
                ),
                flush=True,
            )
            return
        time.sleep(5)
from cryptography.fernet import Fernet  # noqa: E402
from telethon import TelegramClient, functions  # noqa: E402
from telethon.sessions import StringSession  # noqa: E402

DEFAULT_GUIDE_MONITORING_MODEL = "models/gemma-4-31b-it"
DEFAULT_GUIDE_MONITORING_SCREEN_MODEL = "models/gemma-4-31b-it"
# Tier-1 announce/block extraction is routed to gemini-3.1-flash-lite by
# default, mirroring the Smart Update facts_extract migration: extraction
# completeness is a known quality bottleneck (GE-EVAL-02 reportage-tail miss)
# and Lite's 500 RPD/key cap is comfortable at current guide-track load
# (~20-30 extract calls/day). Screen, dedup, route_weaver enrich and other
# stages intentionally stay on Gemma 4 so the Lite RPD budget is not burned
# on classifier-style work.
DEFAULT_GUIDE_MONITORING_EXTRACT_MODEL = "models/gemini-3.1-flash-lite"
MODEL = DEFAULT_GUIDE_MONITORING_MODEL
SCREEN_MODEL = DEFAULT_GUIDE_MONITORING_SCREEN_MODEL
EXTRACT_MODEL = DEFAULT_GUIDE_MONITORING_EXTRACT_MODEL
GOOGLE_KEY_ENV = "GOOGLE_API_KEY2"
GOOGLE_FALLBACK_KEY_ENV = "GOOGLE_API_KEY"
GOOGLE_ACCOUNT_ENV = "GOOGLE_API_LOCALNAME2"
GOOGLE_ACCOUNT_FALLBACK_ENV = "GOOGLE_API_LOCALNAME"
LLM_TIMEOUT_SECONDS = 120
LLM_TIMEOUT_RETRY_ATTEMPTS = 1
LLM_PROVIDER_5XX_RETRY_ATTEMPTS = 1
ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS = 45
GUIDE_OCR_ENABLED = True
GUIDE_OCR_IMAGE_LIMIT_PER_POST = 2
GUIDE_OCR_MAX_IMAGE_BYTES = 6 * 1024 * 1024
GUIDE_OCR_TEXT_LIMIT = 1200
VK_API_VERSION = "5.199"
VK_TIMEOUT_SECONDS = 20
_GEMMA_CLIENTS: dict[str, Any] = {}
_SUPABASE_CLIENT: Any | None = None
_LLM_GATEWAY_LOGGED = False

URL_RE = re.compile(r"https?://[^\s<>()]+", re.I)
USERNAME_RE = re.compile(r"(?<!\w)@([A-Za-z0-9_]{4,64})")
PHONE_RE = re.compile(r"(?:(?:\+7|8)[\s(.-]*)?(?:\d[\s().-]*){10,11}")
DATE_RE = re.compile(
    r"\b(?:\d{1,2}[./]\d{1,2}(?:[./]\d{2,4})?|\d{1,2}\s+(?:январ|феврал|март|апрел|мая|май|июн|июл|август|сентябр|октябр|ноябр|декабр)[а-я]*)\b",
    re.I,
)
TIME_RE = re.compile(r"\b([01]?\d|2[0-3]):([0-5]\d)\b")
KEYCAP_DIGIT_RE = re.compile(r"([0-9])\ufe0f?\u20e3")


def _normalize_model_name(model: str) -> str:
    raw = (model or "").strip()
    if raw.startswith("models/"):
        return raw
    return f"models/{raw}"


def _normalize_keycap_digit_dates(text: str | None) -> str:
    """Turn emoji keycap digits into normal digits for schedule anchoring only."""
    return KEYCAP_DIGIT_RE.sub(r"\1", str(text or ""))


def _retry_after_seconds(message: str) -> float | None:
    match = re.search(r"retry after\s+(\d+)\s*ms", message or "", flags=re.I)
    if not match:
        return None
    try:
        delay_ms = int(match.group(1))
    except Exception:
        return None
    if delay_ms <= 0:
        return None
    return max(0.5, min(delay_ms / 1000.0, 65.0))


def _provider_5xx_status(exc: Exception) -> int | None:
    status_code = int(getattr(exc, "status_code", 0) or 0)
    if status_code in {500, 502, 503, 504}:
        return status_code
    message = str(exc or "")
    if "Unknown field for Schema" in message or "anyOf" in message:
        return None
    for code in (500, 502, 503, 504):
        if str(code) in message:
            return code
    lowered = message.lower()
    if "internalservererror" in lowered or "internal error" in lowered or "unavailable" in lowered:
        return 500
    return None


def refresh_runtime_settings() -> None:
    global MODEL, SCREEN_MODEL, EXTRACT_MODEL, GOOGLE_KEY_ENV, GOOGLE_ACCOUNT_ENV, LLM_TIMEOUT_SECONDS
    global LLM_TIMEOUT_RETRY_ATTEMPTS, LLM_PROVIDER_5XX_RETRY_ATTEMPTS
    global ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS
    global GUIDE_OCR_ENABLED, GUIDE_OCR_IMAGE_LIMIT_PER_POST, GUIDE_OCR_MAX_IMAGE_BYTES, GUIDE_OCR_TEXT_LIMIT
    global VK_API_VERSION, VK_TIMEOUT_SECONDS
    global _GEMMA_CLIENTS, _SUPABASE_CLIENT, _LLM_GATEWAY_LOGGED
    MODEL = (os.getenv("GUIDE_MONITORING_MODEL") or DEFAULT_GUIDE_MONITORING_MODEL).strip()
    SCREEN_MODEL = (os.getenv("GUIDE_MONITORING_SCREEN_MODEL") or DEFAULT_GUIDE_MONITORING_SCREEN_MODEL).strip()
    EXTRACT_MODEL = (
        os.getenv("GUIDE_MONITORING_EXTRACT_MODEL") or DEFAULT_GUIDE_MONITORING_EXTRACT_MODEL
    ).strip()
    GOOGLE_KEY_ENV = (os.getenv("GUIDE_MONITORING_GOOGLE_KEY_ENV") or "GOOGLE_API_KEY2").strip() or "GOOGLE_API_KEY2"
    GOOGLE_ACCOUNT_ENV = (os.getenv("GUIDE_MONITORING_GOOGLE_ACCOUNT_ENV") or "GOOGLE_API_LOCALNAME2").strip() or "GOOGLE_API_LOCALNAME2"
    try:
        LLM_TIMEOUT_SECONDS = max(30, int(float((os.getenv("GUIDE_MONITORING_LLM_TIMEOUT_SEC") or "120").strip() or 120)))
    except Exception:
        LLM_TIMEOUT_SECONDS = 120
    try:
        LLM_TIMEOUT_RETRY_ATTEMPTS = max(
            0,
            min(int(float((os.getenv("GUIDE_MONITORING_LLM_TIMEOUT_RETRIES") or "1").strip() or 1)), 3),
        )
    except Exception:
        LLM_TIMEOUT_RETRY_ATTEMPTS = 1
    try:
        ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS = max(
            20,
            min(int(float((os.getenv("GUIDE_MONITORING_ANNOUNCE_MULTI_FULL_TIMEOUT_SEC") or "45").strip() or 45)), LLM_TIMEOUT_SECONDS),
        )
    except Exception:
        ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS = min(45, LLM_TIMEOUT_SECONDS)
    try:
        LLM_PROVIDER_5XX_RETRY_ATTEMPTS = max(
            0,
            min(int(float((os.getenv("GUIDE_MONITORING_LLM_PROVIDER_5XX_RETRIES") or "1").strip() or 1)), 3),
        )
    except Exception:
        LLM_PROVIDER_5XX_RETRY_ATTEMPTS = 1
    GUIDE_OCR_ENABLED = (os.getenv("GUIDE_MONITORING_OCR_ENABLED") or "1").strip().lower() in {
        "1",
        "true",
        "yes",
        "on",
    }
    try:
        GUIDE_OCR_IMAGE_LIMIT_PER_POST = max(
            1,
            min(int(float((os.getenv("GUIDE_MONITORING_OCR_IMAGE_LIMIT") or "2").strip() or 2)), 4),
        )
    except Exception:
        GUIDE_OCR_IMAGE_LIMIT_PER_POST = 2
    try:
        GUIDE_OCR_MAX_IMAGE_BYTES = max(
            512 * 1024,
            min(int(float((os.getenv("GUIDE_MONITORING_OCR_MAX_IMAGE_BYTES") or str(6 * 1024 * 1024)).strip() or (6 * 1024 * 1024))), 12 * 1024 * 1024),
        )
    except Exception:
        GUIDE_OCR_MAX_IMAGE_BYTES = 6 * 1024 * 1024
    try:
        GUIDE_OCR_TEXT_LIMIT = max(
            300,
            min(int(float((os.getenv("GUIDE_MONITORING_OCR_TEXT_LIMIT") or "1200").strip() or 1200)), 3000),
        )
    except Exception:
        GUIDE_OCR_TEXT_LIMIT = 1200
    VK_API_VERSION = (os.getenv("GUIDE_MONITORING_VK_API_VERSION") or "5.199").strip() or "5.199"
    try:
        VK_TIMEOUT_SECONDS = max(
            5,
            min(int(float((os.getenv("GUIDE_MONITORING_VK_TIMEOUT_SEC") or "20").strip() or 20)), 60),
        )
    except Exception:
        VK_TIMEOUT_SECONDS = 20
    _GEMMA_CLIENTS = {}
    _SUPABASE_CLIENT = None
    _LLM_GATEWAY_LOGGED = False


class _GuideSecretsProviderAdapter:
    def __init__(self, base: Any):
        self.base = base

    def get_secret(self, name: str) -> str | None:
        if name == "GOOGLE_API_KEY":
            return self.base.get_secret(GOOGLE_KEY_ENV) or self.base.get_secret(GOOGLE_FALLBACK_KEY_ENV)
        return self.base.get_secret(name)

    def get_secret_pool(self, prefix: str) -> list[str]:
        if prefix == "GOOGLE_API_KEY":
            keys: list[str] = []
            primary = self.get_secret("GOOGLE_API_KEY")
            if primary:
                keys.append(primary)
            return keys
        getter = getattr(self.base, "get_secret_pool", None)
        if callable(getter):
            return list(getter(prefix) or [])
        return []


def _build_supabase_client() -> Any | None:
    _bootstrap_repo_bundle()
    from google_ai.limiter_supabase import build_google_ai_limiter_supabase_client
    return build_google_ai_limiter_supabase_client(require_configured=True)


def _get_supabase_client() -> Any | None:
    global _SUPABASE_CLIENT
    if _SUPABASE_CLIENT is None:
        _SUPABASE_CLIENT = _build_supabase_client()
    return _SUPABASE_CLIENT


def _guide_account_name() -> str | None:
    return (os.getenv(GOOGLE_ACCOUNT_ENV) or os.getenv(GOOGLE_ACCOUNT_FALLBACK_ENV) or "").strip() or None


def _log_llm_gateway_once() -> None:
    global _LLM_GATEWAY_LOGGED
    if _LLM_GATEWAY_LOGGED:
        return
    print(
        (
            "Guide monitor llm_gateway="
            f"google_ai key_env={GOOGLE_KEY_ENV} "
            f"account_env={GOOGLE_ACCOUNT_ENV} "
            f"account_name={_guide_account_name() or '-'} "
            f"supabase={'yes' if _get_supabase_client() is not None else 'no'} "
            f"timeout={LLM_TIMEOUT_SECONDS}s "
            f"timeout_retries={LLM_TIMEOUT_RETRY_ATTEMPTS} "
            f"announce_multi_full_timeout={ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS}s "
            f"provider_5xx_retries={LLM_PROVIDER_5XX_RETRY_ATTEMPTS} "
            f"reserve_fallback={os.getenv('GOOGLE_AI_ALLOW_RESERVE_FALLBACK', '0')} "
            f"local_fallback={os.getenv('GOOGLE_AI_LOCAL_LIMITER_FALLBACK', '0')}"
        ),
        flush=True,
    )
    _LLM_GATEWAY_LOGGED = True


def _get_gemma_client(consumer: str) -> Any:
    client = _GEMMA_CLIENTS.get(consumer)
    if client is None:
        _bootstrap_repo_bundle()
        from google_ai import GoogleAIClient, SecretsProvider

        _log_llm_gateway_once()
        client = GoogleAIClient(
            supabase_client=_get_supabase_client(),
            secrets_provider=_GuideSecretsProviderAdapter(SecretsProvider()),
            consumer=consumer,
            account_name=_guide_account_name(),
        )
        _GEMMA_CLIENTS[consumer] = client
        print(
            (
                f"[gemma:client] consumer={consumer} "
                f"model_family=gemma-only account={_guide_account_name() or '-'} "
                f"key_env={GOOGLE_KEY_ENV}"
            ),
            flush=True,
        )
    return client


def _find_file(name: str) -> Path:
    for path in INPUT_ROOT.rglob(name):
        if path.is_file():
            return path
    raise RuntimeError(f"{name} not found under {INPUT_ROOT}")


def load_runtime_config() -> dict[str, Any]:
    config_path = _find_file("config.json")
    secrets_path = _find_file("secrets.enc")
    key_path = _find_file("fernet.key")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    decrypted = Fernet(key_path.read_bytes()).decrypt(secrets_path.read_bytes())
    secrets = json.loads(decrypted.decode("utf-8"))
    if not isinstance(config, dict) or not isinstance(secrets, dict):
        raise RuntimeError("Invalid config/secrets payload")
    for key, value in secrets.items():
        if value is None:
            continue
        os.environ.setdefault(str(key), str(value))
    return config


def _resolve_auth_bundle() -> tuple[str | None, dict[str, Any] | None]:
    allow_non_s22 = (
        (os.getenv("GUIDE_MONITORING_ALLOW_NON_S22_AUTH") or "0").strip().lower()
        in {"1", "true", "yes", "on"}
    )
    bundle_env = (os.getenv("GUIDE_MONITORING_AUTH_BUNDLE_ENV") or "").strip()
    if bundle_env:
        if bundle_env != "TELEGRAM_AUTH_BUNDLE_S22" and not allow_non_s22:
            raise RuntimeError(
                "GUIDE_MONITORING_AUTH_BUNDLE_ENV may use only TELEGRAM_AUTH_BUNDLE_S22 for Kaggle "
                "unless GUIDE_MONITORING_ALLOW_NON_S22_AUTH=1 is explicitly set"
            )
        candidates = [bundle_env]
    else:
        candidates = ["TELEGRAM_AUTH_BUNDLE_S22"]
        if (os.getenv("TELEGRAM_AUTH_BUNDLE_S22") or "").strip() == "" and (
            os.getenv("TELEGRAM_AUTH_BUNDLE_E2E") or ""
        ).strip():
            if not allow_non_s22:
                raise RuntimeError(
                    "Guide Kaggle monitoring requires TELEGRAM_AUTH_BUNDLE_S22; refusing to fall back to "
                    "TELEGRAM_AUTH_BUNDLE_E2E without GUIDE_MONITORING_ALLOW_NON_S22_AUTH=1"
                )
            candidates.append("TELEGRAM_AUTH_BUNDLE_E2E")
    for key in candidates:
        if not key:
            continue
        raw = (os.getenv(key) or "").strip()
        if not raw:
            continue
        decoded = base64.urlsafe_b64decode(raw.encode("ascii")).decode("utf-8")
        payload = json.loads(decoded)
        if isinstance(payload, dict) and str(payload.get("session") or "").strip():
            return key, payload
    return None, None


async def create_client() -> TelegramClient:
    api_id = int((os.getenv("TG_API_ID") or os.getenv("TELEGRAM_API_ID") or "0").strip() or 0)
    api_hash = (os.getenv("TG_API_HASH") or os.getenv("TELEGRAM_API_HASH") or "").strip()
    if not api_id or not api_hash:
        raise RuntimeError("Missing TG_API_ID/TG_API_HASH")
    _bundle_env, bundle = _resolve_auth_bundle()
    session = str((bundle or {}).get("session") or os.getenv("TG_SESSION") or os.getenv("TELEGRAM_SESSION") or "").strip()
    if not session:
        raise RuntimeError("Missing TELEGRAM auth bundle or session")
    client = TelegramClient(
        StringSession(session),
        api_id,
        api_hash,
        device_model=str((bundle or {}).get("device_model") or "Kaggle Guide Monitor"),
        system_version=str((bundle or {}).get("system_version") or "Linux"),
        app_version=str((bundle or {}).get("app_version") or "1.0"),
        lang_code=str((bundle or {}).get("lang_code") or "ru"),
        system_lang_code=str((bundle or {}).get("system_lang_code") or "ru"),
    )
    await client.connect()
    if not await client.is_user_authorized():
        raise RuntimeError("Telethon client is not authorized")
    return client


async def ensure_client_connected(client: TelegramClient, *, force_reconnect: bool = False) -> None:
    connected = False
    try:
        connected = bool(client.is_connected())
    except Exception:
        connected = False
    if force_reconnect and connected:
        await client.disconnect()
        connected = False
    if not connected:
        await client.connect()
    if not await client.is_user_authorized():
        raise RuntimeError("Telethon client is not authorized")


def _message_text(message: Any) -> str:
    return str(getattr(message, "message", None) or getattr(message, "text", None) or "").strip()


def _message_media_kind(message: Any) -> str | None:
    if getattr(message, "photo", None):
        return "photo"
    if getattr(message, "video", None):
        return "video"
    doc = getattr(message, "document", None)
    if doc is None:
        return None
    mime = str(getattr(doc, "mime_type", None) or "").lower()
    if mime.startswith("video/"):
        return "video"
    if mime.startswith("image/"):
        return "photo"
    return None


def _reactions_payload(message: Any) -> tuple[int | None, dict[str, int] | None]:
    reactions = getattr(message, "reactions", None)
    if not reactions or not getattr(reactions, "results", None):
        return None, None
    out: dict[str, int] = {}
    total = 0
    for item in getattr(reactions, "results", []) or []:
        count = int(getattr(item, "count", 0) or 0)
        reaction = getattr(item, "reaction", None)
        emoji = str(getattr(reaction, "emoticon", None) or getattr(reaction, "document_id", None) or reaction or "")
        if not emoji:
            emoji = "reaction"
        out[emoji] = count
        total += count
    return total or None, out or None


@dataclass(slots=True)
class ScannedPost:
    message_id: int
    grouped_id: int | None
    post_date: datetime
    source_url: str
    text: str
    views: int | None
    forwards: int | None
    reactions_total: int | None
    reactions_json: dict[str, int] | None
    media_refs: list[dict[str, Any]]
    media_assets: list[dict[str, Any]]


def collapse_ws(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_title_key(value: str | None) -> str:
    text = collapse_ws(value).lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-zа-яё0-9]+", " ", text, flags=re.I)
    text = re.sub(r"\b(?:экскурсия|экскурсии|прогулка|прогулки|тур|маршрут|авторская|пешеходная|поездка|путешествие)\b", " ", text, flags=re.I)
    return collapse_ws(text)


def build_source_fingerprint(*, title_normalized: str, date_iso: str | None, time_text: str | None) -> str:
    import hashlib

    payload = "|".join([str(title_normalized or ""), str(date_iso or ""), str(time_text or "")])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


async def materialize_post_media_assets(
    client: TelegramClient,
    *,
    username: str,
    post: ScannedPost,
) -> list[dict[str, Any]]:
    if not post.media_refs:
        return []
    entity = await client.get_entity(username)
    MEDIA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out: list[dict[str, Any]] = []
    for idx, media_ref in enumerate(post.media_refs[:GUIDE_MEDIA_OUTPUT_LIMIT_PER_POST]):
        message_id = int(media_ref.get("message_id") or 0)
        if message_id <= 0:
            continue
        try:
            message = await client.get_messages(entity, ids=message_id)
        except Exception:
            continue
        if not message:
            continue
        kind = _message_media_kind(message) or str(media_ref.get("kind") or "").strip().lower()
        if kind not in {"photo", "video"}:
            continue
        try:
            downloaded = await client.download_media(message, file=bytes)
        except Exception:
            continue
        if not downloaded:
            continue
        payload = bytes(downloaded)
        if not payload:
            continue
        if len(payload) > GUIDE_MEDIA_OUTPUT_MAX_MB * 1024 * 1024:
            continue
        ext = ".mp4" if kind == "video" else ".jpg"
        rel_path = f"guide_media/{username}_{post.message_id}_{message_id}_{idx}{ext}"
        asset_path = WORK_DIR / rel_path
        asset_path.parent.mkdir(parents=True, exist_ok=True)
        asset_path.write_bytes(payload)
        out.append(
            {
                "message_id": message_id,
                "kind": kind,
                "grouped_id": int(media_ref.get("grouped_id") or 0) or None,
                "relative_path": rel_path,
                "size_bytes": len(payload),
            }
        )
    return out


def _collapse_group(messages: list[Any], *, username: str) -> ScannedPost | None:
    ordered = sorted(messages, key=lambda item: int(getattr(item, "id", 0) or 0))
    anchor = next((msg for msg in ordered if _message_text(msg)), None) or ordered[-1]
    anchor_id = int(getattr(anchor, "id", 0) or 0)
    post_date = getattr(anchor, "date", None) or getattr(ordered[-1], "date", None) or datetime.now(timezone.utc)
    if post_date.tzinfo is None:
        post_date = post_date.replace(tzinfo=timezone.utc)
    text = "\n".join(_message_text(msg) for msg in ordered if _message_text(msg)).strip()
    views = max((getattr(msg, "views", None) or 0) for msg in ordered) or None
    forwards = max((getattr(msg, "forwards", None) or 0) for msg in ordered) or None
    reactions_total = None
    reactions_json = None
    for msg in ordered:
        total, payload = _reactions_payload(msg)
        if total is not None and ((reactions_total or -1) < total):
            reactions_total = total
            reactions_json = payload
    media_refs: list[dict[str, Any]] = []
    for msg in ordered:
        kind = _message_media_kind(msg)
        if not kind:
            continue
        media_refs.append(
            {
                "message_id": int(getattr(msg, "id", 0) or 0),
                "kind": kind,
                "grouped_id": int(getattr(msg, "grouped_id", 0) or 0) or None,
            }
        )
    if not text and not media_refs:
        return None
    return ScannedPost(
        message_id=anchor_id,
        grouped_id=int(getattr(anchor, "grouped_id", 0) or 0) or None,
        post_date=post_date.astimezone(timezone.utc),
        source_url=f"https://t.me/{username}/{anchor_id}",
        text=text,
        views=views,
        forwards=forwards,
        reactions_total=reactions_total,
        reactions_json=reactions_json,
        media_refs=media_refs,
        media_assets=[],
    )


async def scan_source_posts(client: TelegramClient, *, username: str, limit: int, days_back: int) -> tuple[dict[str, Any], list[ScannedPost]]:
    entity = await client.get_entity(username)
    source_title = str(getattr(entity, "title", None) or getattr(entity, "first_name", None) or "").strip() or None
    about_text = ""
    about_links: list[str] = []
    try:
        full = await client(functions.channels.GetFullChannelRequest(channel=entity))
        about_text = str(getattr(full.full_chat, "about", None) or "").strip()
    except Exception:
        try:
            full = await client(functions.users.GetFullUserRequest(id=entity))
            about_text = str(getattr(full.full_user, "about", None) or "").strip()
        except Exception:
            about_text = ""
    if about_text:
        seen: set[str] = set()
        for token in about_text.replace("\n", " ").split():
            raw = str(token or "").strip("()[]{}<>.,!?:;\"'")
            if raw.startswith("http://") or raw.startswith("https://"):
                if raw not in seen:
                    seen.add(raw)
                    about_links.append(raw)
    messages = await client.get_messages(entity, limit=max(1, int(limit)))
    cutoff = datetime.now(timezone.utc) - timedelta(days=max(1, int(days_back)))
    singles: list[Any] = []
    grouped: dict[int, list[Any]] = {}
    for msg in messages:
        msg_date = getattr(msg, "date", None)
        if msg_date is None:
            continue
        if msg_date.tzinfo is None:
            msg_date = msg_date.replace(tzinfo=timezone.utc)
        if msg_date.astimezone(timezone.utc) < cutoff:
            continue
        if getattr(msg, "action", None):
            continue
        grouped_id = int(getattr(msg, "grouped_id", 0) or 0)
        if grouped_id:
            grouped.setdefault(grouped_id, []).append(msg)
        else:
            singles.append(msg)
    posts: list[ScannedPost] = []
    for msg in singles:
        collapsed = _collapse_group([msg], username=username)
        if collapsed:
            posts.append(collapsed)
    for group in grouped.values():
        collapsed = _collapse_group(group, username=username)
        if collapsed:
            posts.append(collapsed)
    posts.sort(key=lambda item: (item.post_date, item.message_id), reverse=True)
    return {"source_title": source_title, "about_text": about_text or None, "about_links": about_links}, posts


def _vk_token() -> str:
    token = (os.getenv("GUIDE_MONITORING_VK_TOKEN") or "").strip()
    if token:
        return token
    token_env = (os.getenv("GUIDE_MONITORING_VK_TOKEN_ENV") or "VK_ACCESS_TOKEN5").strip() or "VK_ACCESS_TOKEN5"
    return (os.getenv(token_env) or "").strip()


def _vk_api_call(method: str, params: dict[str, Any]) -> Any:
    token = _vk_token()
    if not token:
        raise RuntimeError("GUIDE_MONITORING_VK_TOKEN is missing in Kaggle runtime")
    payload = {key: value for key, value in params.items() if value is not None}
    payload["access_token"] = token
    payload["v"] = VK_API_VERSION
    url = f"https://api.vk.com/method/{method}?{urllib.parse.urlencode(payload)}"
    request = urllib.request.Request(url, headers={"User-Agent": "events-bot-guide-monitor/1.0"})
    with urllib.request.urlopen(request, timeout=VK_TIMEOUT_SECONDS) as response:
        raw = response.read().decode("utf-8")
    data = json.loads(raw)
    if isinstance(data, dict) and data.get("error"):
        error = data["error"]
        if isinstance(error, dict):
            raise RuntimeError(f"VK API {method} error {error.get('error_code')}: {error.get('error_msg')}")
        raise RuntimeError(f"VK API {method} error: {error}")
    return data.get("response") if isinstance(data, dict) else data


def _vk_group_from_response(response: Any) -> dict[str, Any]:
    if isinstance(response, dict):
        groups = response.get("groups")
        if isinstance(groups, list) and groups:
            return groups[0] if isinstance(groups[0], dict) else {}
    if isinstance(response, list) and response:
        return response[0] if isinstance(response[0], dict) else {}
    return {}


def _vk_wall_items(response: Any) -> list[dict[str, Any]]:
    if isinstance(response, dict):
        items = response.get("items")
        return [item for item in (items or []) if isinstance(item, dict)]
    if isinstance(response, list):
        return [item for item in response[1:] if isinstance(item, dict)]
    return []


def _vk_extract_links(text: str) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for match in URL_RE.findall(text or ""):
        link = match.strip(".,);:!?\"'")
        if link and link not in seen:
            seen.add(link)
            out.append(link)
    return out


def _vk_post_text(post: dict[str, Any]) -> str:
    text = str(post.get("text") or "")
    text = text.replace("<br>", "\n").replace("<br/>", "\n").replace("<br />", "\n")
    return html.unescape(text).strip()


def _safe_media_fragment(value: Any) -> str:
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value or "").strip())
    return text.strip("._") or "media"


def _vk_best_photo_url(photo: dict[str, Any]) -> str | None:
    sizes = photo.get("sizes") if isinstance(photo.get("sizes"), list) else []
    best_url: str | None = None
    best_score = -1
    for size in sizes:
        if not isinstance(size, dict):
            continue
        url = str(size.get("url") or "").strip()
        if not url:
            continue
        try:
            score = int(size.get("width") or 0) * int(size.get("height") or 0)
        except Exception:
            score = 0
        if score > best_score:
            best_score = score
            best_url = url
    return best_url


def _vk_photo_media_refs(post: dict[str, Any]) -> list[dict[str, Any]]:
    refs: list[dict[str, Any]] = []
    attachments = post.get("attachments") if isinstance(post.get("attachments"), list) else []
    message_id = int(post.get("id") or 0)
    for idx, att in enumerate(attachments[:GUIDE_MEDIA_OUTPUT_LIMIT_PER_POST]):
        if not isinstance(att, dict) or att.get("type") != "photo":
            continue
        photo = att.get("photo") if isinstance(att.get("photo"), dict) else {}
        url = _vk_best_photo_url(photo)
        if not url:
            continue
        refs.append(
            {
                "message_id": message_id,
                "kind": "photo",
                "attachment_index": idx,
                "owner_id": int(photo.get("owner_id") or 0) or None,
                "id": int(photo.get("id") or 0) or None,
                "url": url,
            }
        )
    return refs


def materialize_vk_post_media_assets(*, username: str, post: ScannedPost) -> list[dict[str, Any]]:
    if not post.media_refs:
        return []
    MEDIA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out: list[dict[str, Any]] = []
    max_bytes = GUIDE_MEDIA_OUTPUT_MAX_MB * 1024 * 1024
    for idx, media_ref in enumerate(post.media_refs[:GUIDE_MEDIA_OUTPUT_LIMIT_PER_POST]):
        if str(media_ref.get("kind") or "").strip().lower() != "photo":
            continue
        url = str(media_ref.get("url") or "").strip()
        if not url:
            continue
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "events-bot-guide-monitor/1.0"})
            with urllib.request.urlopen(request, timeout=VK_TIMEOUT_SECONDS) as response:
                payload = response.read(max_bytes + 1)
        except Exception as exc:
            print(
                f"[guide:vk-media:error] source=vk:{username} message_id={post.message_id} error={type(exc).__name__}: {exc}",
                flush=True,
            )
            continue
        if not payload or len(payload) > max_bytes:
            continue
        rel_path = (
            f"guide_media/{_safe_media_fragment(username)}_"
            f"{int(post.message_id)}_{int(media_ref.get('attachment_index') or idx)}_{idx}.jpg"
        )
        asset_path = WORK_DIR / rel_path
        asset_path.parent.mkdir(parents=True, exist_ok=True)
        asset_path.write_bytes(payload)
        out.append(
            {
                "message_id": int(media_ref.get("message_id") or post.message_id),
                "kind": "photo",
                "attachment_index": int(media_ref.get("attachment_index") or idx),
                "owner_id": media_ref.get("owner_id"),
                "id": media_ref.get("id"),
                "relative_path": rel_path,
                "size_bytes": len(payload),
            }
        )
    return out


def _vk_post_to_scanned_post(
    post: dict[str, Any],
    *,
    group_id: int,
    username: str,
    owner_type: str = "group",
) -> ScannedPost | None:
    message_id = int(post.get("id") or 0)
    timestamp = int(post.get("date") or 0)
    if message_id <= 0 or timestamp <= 0:
        return None
    likes = int(((post.get("likes") or {}) if isinstance(post.get("likes"), dict) else {}).get("count") or 0)
    comments = int(((post.get("comments") or {}) if isinstance(post.get("comments"), dict) else {}).get("count") or 0)
    reposts = int(((post.get("reposts") or {}) if isinstance(post.get("reposts"), dict) else {}).get("count") or 0)
    reactions_json = {
        key: value
        for key, value in {"likes": likes, "comments": comments, "reposts": reposts}.items()
        if value > 0
    }
    views_raw = post.get("views")
    views = int(views_raw.get("count") or 0) if isinstance(views_raw, dict) and views_raw.get("count") is not None else None
    text = _vk_post_text(post)
    attachments = post.get("attachments") if isinstance(post.get("attachments"), list) else []
    if not text and not attachments:
        return None
    if owner_type == "user":
        source_url = f"https://vk.com/wall{group_id}_{message_id}"
    else:
        source_url = f"https://vk.com/wall-{group_id}_{message_id}"
    return ScannedPost(
        message_id=message_id,
        grouped_id=None,
        post_date=datetime.fromtimestamp(timestamp, tz=timezone.utc),
        source_url=source_url,
        text=text,
        views=views,
        forwards=reposts or None,
        reactions_total=sum(reactions_json.values()) or None,
        reactions_json=reactions_json or None,
        media_refs=_vk_photo_media_refs(post),
        media_assets=[],
    )


def _vk_resolve_source(username: str) -> tuple[int, str, str, str]:
    """Return ``(numeric_id, screen_name, display_name, owner_type)``.

    First tries ``utils.resolveScreenName`` to learn whether the URL points
    at a community (``group``/``page``/``event``) or a personal page
    (``user``). Falls back to ``groups.getById`` when resolution returned
    nothing, so plain community shortnames keep working.
    """
    resolved_type: str | None = None
    resolved_id: int | None = None
    try:
        rs = _vk_api_call("utils.resolveScreenName", {"screen_name": username})
    except Exception:
        rs = None
    if isinstance(rs, dict):
        rtype = str(rs.get("type") or "").strip().lower()
        try:
            obj_id = int(rs.get("object_id") or 0)
        except (TypeError, ValueError):
            obj_id = 0
        if obj_id > 0 and rtype in {"group", "page", "public", "event", "user"}:
            resolved_id = obj_id
            resolved_type = "user" if rtype == "user" else "group"
    if resolved_type == "user":
        user_response = _vk_api_call(
            "users.get",
            {"user_ids": resolved_id, "fields": "screen_name,about,status,site"},
        )
        users = user_response if isinstance(user_response, list) else []
        user = users[0] if users and isinstance(users[0], dict) else {}
        user_id = int(user.get("id") or resolved_id or 0)
        if user_id <= 0:
            raise RuntimeError(f"VK user not found: {username}")
        domain = collapse_ws(user.get("screen_name")) or username
        first = collapse_ws(user.get("first_name") or "")
        last = collapse_ws(user.get("last_name") or "")
        display = (f"{first} {last}".strip()) or domain
        return user_id, domain, display, "user"
    group_arg = resolved_id if resolved_id is not None else username
    group_response = _vk_api_call(
        "groups.getById",
        {"group_ids": group_arg, "fields": "description,site,screen_name,name"},
    )
    group = _vk_group_from_response(group_response)
    group_id = int(group.get("id") or 0)
    if group_id <= 0:
        raise RuntimeError(f"VK group not found: {username}")
    domain = collapse_ws(group.get("screen_name")) or username
    return group_id, domain, collapse_ws(group.get("name")) or domain, "group"


async def scan_vk_source_posts(
    source_payload: dict[str, Any],
    *,
    limit: int,
    days_back: int,
) -> tuple[dict[str, Any], list[ScannedPost]]:
    username = collapse_ws(source_payload.get("username")).lstrip("@")
    if not username:
        raise RuntimeError("VK source username is empty")
    group_id, domain, display_name, owner_type = await asyncio.to_thread(
        _vk_resolve_source, username
    )
    about_text = ""
    description = ""
    site = ""
    if owner_type == "group":
        group_response = await asyncio.to_thread(
            _vk_api_call,
            "groups.getById",
            {"group_ids": group_id, "fields": "description,site,screen_name,name"},
        )
        group = _vk_group_from_response(group_response)
        description = collapse_ws(group.get("description"))
        site = collapse_ws(group.get("site"))
        about_text = "\n".join(part for part in (description, site) if part).strip()
        domain = collapse_ws(group.get("screen_name")) or domain
    else:
        user_response = await asyncio.to_thread(
            _vk_api_call,
            "users.get",
            {"user_ids": group_id, "fields": "screen_name,about,status,site"},
        )
        users = user_response if isinstance(user_response, list) else []
        user = users[0] if users and isinstance(users[0], dict) else {}
        about = collapse_ws(user.get("about"))
        status = collapse_ws(user.get("status"))
        site = collapse_ws(user.get("site"))
        about_text = "\n".join(part for part in (about, status, site) if part).strip()
        domain = collapse_ws(user.get("screen_name")) or domain
    owner_id_signed = group_id if owner_type == "user" else -group_id
    wall_response = await asyncio.to_thread(
        _vk_api_call,
        "wall.get",
        {"owner_id": str(owner_id_signed), "count": max(1, int(limit)), "filter": "owner"},
    )
    cutoff = datetime.now(timezone.utc) - timedelta(days=max(1, int(days_back)))
    posts: list[ScannedPost] = []
    for item in _vk_wall_items(wall_response):
        scanned = _vk_post_to_scanned_post(
            item, group_id=group_id, username=domain, owner_type=owner_type
        )
        if scanned and scanned.post_date >= cutoff:
            posts.append(scanned)
    posts.sort(key=lambda item: (item.post_date, item.message_id), reverse=True)
    links = _vk_extract_links(about_text)
    source_url = collapse_ws(source_payload.get("source_url")) or f"https://vk.com/{domain}"
    if source_url and source_url not in links:
        links.insert(0, source_url)
    return {
        "source_title": display_name or source_payload.get("title"),
        "about_text": about_text or None,
        "about_links": links,
        "source_url": source_url,
    }, posts


def _detect_image_mime(data: bytes) -> str:
    if data.startswith(b"\xff\xd8\xff"):
        return "image/jpeg"
    if data.startswith(b"\x89PNG\r\n\x1a\n"):
        return "image/png"
    if data.startswith(b"GIF87a") or data.startswith(b"GIF89a"):
        return "image/gif"
    if data.startswith(b"RIFF") and data[8:12] == b"WEBP":
        return "image/webp"
    return "image/jpeg"


def _ocr_chunk_text(chunk: Mapping[str, Any]) -> str:
    parts = [collapse_ws(chunk.get("title")), collapse_ws(chunk.get("text"))]
    return "\n".join(part for part in parts if part).strip()


def _post_has_photo_media(post: ScannedPost) -> bool:
    for item in post.media_refs or []:
        kind = collapse_ws(item.get("kind")).lower()
        if kind == "photo":
            return True
    return False


def _should_run_post_ocr(post: ScannedPost, source_kind: str, flags: dict[str, Any], *, base_pass: bool) -> bool:
    if not GUIDE_OCR_ENABLED or not _post_has_photo_media(post):
        return False
    if base_pass:
        return True
    text_len = len(collapse_ws(post.text))
    if text_len <= 220:
        return True
    if bool(flags.get("grouped_album_present")):
        return True
    if any(bool(flags.get(key)) for key in ("has_date_signal", "has_booking_signal", "has_status_signal")):
        return True
    return source_kind in {"guide_personal", "guide_project", "organization_with_tours", "excursion_operator"}


async def _collect_post_ocr_inputs(
    client: TelegramClient,
    *,
    username: str,
    post: ScannedPost,
) -> list[dict[str, Any]]:
    if not _post_has_photo_media(post):
        return []
    entity = await client.get_entity(username)
    out: list[dict[str, Any]] = []
    for media_ref in post.media_refs:
        if len(out) >= GUIDE_OCR_IMAGE_LIMIT_PER_POST:
            break
        if collapse_ws(media_ref.get("kind")).lower() != "photo":
            continue
        message_id = int(media_ref.get("message_id") or 0)
        if message_id <= 0:
            continue
        try:
            message = await client.get_messages(entity, ids=message_id)
        except Exception:
            continue
        if not message:
            continue
        try:
            downloaded = await client.download_media(message, file=bytes)
        except Exception:
            continue
        if not downloaded:
            continue
        payload = bytes(downloaded)
        if not payload or len(payload) > GUIDE_OCR_MAX_IMAGE_BYTES:
            continue
        out.append(
            {
                "message_id": message_id,
                "mime_type": _detect_image_mime(payload),
                "data": payload,
                "sha256": hashlib.sha256(payload).hexdigest(),
            }
        )
    return out


async def _ocr_post_image(
    image_payload: Mapping[str, Any],
    *,
    consumer: str,
    model: str,
    source_username: str,
    post: ScannedPost,
    image_index: int,
) -> dict[str, Any] | None:
    mime_type = collapse_ws(image_payload.get("mime_type")) or "image/jpeg"
    image_bytes = image_payload.get("data")
    if not isinstance(image_bytes, (bytes, bytearray)) or not image_bytes:
        return None
    schema = {
        "type": "object",
        "properties": {
            "ocr_text": {"type": "string"},
            "ocr_title": {"type": "string"},
            "excursion_signal": {"type": "boolean"},
            "schedule_signal": {"type": "boolean"},
            "booking_signal": {"type": "boolean"},
        },
        "required": [
            "ocr_text",
            "ocr_title",
            "excursion_signal",
            "schedule_signal",
            "booking_signal",
        ],
    }
    prompt = [
        {
            "text": (
                "You read one Telegram image related to guide excursions. Return only JSON.\n"
                "Fields:\n"
                "- ocr_text: all readable Russian/English text from the image, preserving dates, times, prices, handles, phones and links.\n"
                "- ocr_title: dominant route/excursion title from the image, or empty string if there is no reliable title.\n"
                "- excursion_signal: true only if the image itself shows a concrete excursion/walk/tour/route signal.\n"
                "- schedule_signal: true only if the image contains an explicit date/time or same-day relative schedule marker.\n"
                "- booking_signal: true only if the image contains explicit booking/contact/link/phone/DM signal.\n"
                "Do not invent unreadable text. Ignore decorative slogans unless they carry excursion facts.\n"
                f"Post context: url={post.source_url} post_date_utc={post.post_date.isoformat()}."
            )
        },
        {
            "inline_data": {
                "mime_type": mime_type,
                "data": bytes(image_bytes),
            }
        },
    ]
    data = await ask_gemma(
        model,
        prompt,
        consumer=consumer,
        max_output_tokens=420,
        response_schema=schema,
        log_context=(
            f"source=@{source_username} message_id={post.message_id} "
            f"media_message_id={int(image_payload.get('message_id') or 0)} "
            f"image_index={image_index} sha={collapse_ws(image_payload.get('sha256'))[:12] or '-'}"
        ),
    )
    if not isinstance(data, dict):
        return None
    text = collapse_ws(data.get("ocr_text"))
    title = collapse_ws(data.get("ocr_title"))
    if not text and not title:
        return None
    return {
        "title": title or None,
        "text": text[:GUIDE_OCR_TEXT_LIMIT] if text else "",
        "excursion_signal": bool(data.get("excursion_signal")),
        "schedule_signal": bool(data.get("schedule_signal")),
        "booking_signal": bool(data.get("booking_signal")),
    }


async def collect_post_ocr_chunks(
    client: TelegramClient,
    *,
    username: str,
    post: ScannedPost,
    model: str,
) -> list[dict[str, Any]]:
    image_inputs = await _collect_post_ocr_inputs(client, username=username, post=post)
    if not image_inputs:
        return []
    chunks: list[dict[str, Any]] = []
    for idx, image_payload in enumerate(image_inputs, start=1):
        try:
            chunk = await _ocr_post_image(
                image_payload,
                consumer="guide_scout_ocr",
                model=model,
                source_username=username,
                post=post,
                image_index=idx,
            )
        except Exception as exc:
            print(
                (
                    "[guide:ocr:error] "
                    f"source=@{username} message_id={post.message_id} "
                    f"media_message_id={int(image_payload.get('message_id') or 0)} "
                    f"image_index={idx} sha={collapse_ws(image_payload.get('sha256'))[:12] or '-'} "
                    f"error={type(exc).__name__}: {exc}"
                ),
                flush=True,
            )
            continue
        if not chunk:
            print(
                (
                    "[guide:ocr:empty] "
                    f"source=@{username} message_id={post.message_id} "
                    f"media_message_id={int(image_payload.get('message_id') or 0)} "
                    f"image_index={idx} sha={collapse_ws(image_payload.get('sha256'))[:12] or '-'}"
                ),
                flush=True,
            )
            continue
        print(
            (
                "[guide:ocr:ok] "
                f"source=@{username} message_id={post.message_id} "
                f"media_message_id={int(image_payload.get('message_id') or 0)} "
                f"image_index={idx} sha={collapse_ws(image_payload.get('sha256'))[:12] or '-'} "
                f"text_chars={len(collapse_ws(chunk.get('text')))} "
                f"title={'yes' if collapse_ws(chunk.get('title')) else 'no'} "
                f"signals=excursion:{int(bool(chunk.get('excursion_signal')))},"
                f"schedule:{int(bool(chunk.get('schedule_signal')))},"
                f"booking:{int(bool(chunk.get('booking_signal')))}"
            ),
            flush=True,
        )
        chunks.append(
            {
                "id": f"O{idx}",
                "message_id": int(image_payload.get("message_id") or 0) or None,
                "sha256": collapse_ws(image_payload.get("sha256")) or None,
                **chunk,
            }
        )
    return chunks


def prefilter_flags(post: ScannedPost, *, ocr_chunks: list[dict[str, Any]] | None = None) -> dict[str, Any]:
    ocr_chunks = [item for item in (ocr_chunks or []) if isinstance(item, dict)]
    ocr_text = "\n".join(_ocr_chunk_text(item) for item in ocr_chunks if _ocr_chunk_text(item))
    text = collapse_ws("\n".join(part for part in (post.text, ocr_text) if collapse_ws(part))).lower()
    schedule_text = _normalize_keycap_digit_dates(text)
    has_ocr_excursion_signal = any(bool(item.get("excursion_signal")) for item in ocr_chunks)
    has_ocr_schedule_signal = any(bool(item.get("schedule_signal")) for item in ocr_chunks)
    has_ocr_booking_signal = any(bool(item.get("booking_signal")) for item in ocr_chunks)
    return {
        "has_date_signal": bool(DATE_RE.search(schedule_text) or "завтра" in text or "сегодня" in text or has_ocr_schedule_signal),
        "has_time_signal": bool(TIME_RE.search(text)),
        "has_price_signal": any(token in text for token in ("стоимость", "цена", "руб", "₽")),
        "has_booking_signal": bool(URL_RE.search(text) or USERNAME_RE.search(text) or PHONE_RE.search(text) or "запись" in text or "бронир" in text or has_ocr_booking_signal),
        "has_status_signal": any(token in text for token in ("мест нет", "лист ожидания", "sold out", "перенос", "отмена", "осталось", "последние места")),
        "has_group_signal": any(token in text for token in ("по запросу", "организованные группы", "для групп", "школьн", "семь")),
        "has_excursion_keywords": any(token in text for token in ("экскурс", "прогул", "маршрут", "путешеств", "тур ")) or has_ocr_excursion_signal,
        "has_ocr_excursion_signal": has_ocr_excursion_signal,
        "grouped_album_present": bool(post.grouped_id),
        "message_url": post.source_url,
    }


def prefilter_pass(post: ScannedPost, source_kind: str, flags: dict[str, Any]) -> bool:
    if not (flags["has_excursion_keywords"] or flags.get("has_ocr_excursion_signal")):
        return False
    if source_kind == "aggregator" and not (
        flags.get("has_ocr_excursion_signal")
        or any(token in collapse_ws(post.text).lower() for token in ("авторская", "приглашаем", "пешеходная"))
    ):
        return False
    return any(flags[key] for key in ("has_date_signal", "has_booking_signal", "has_status_signal", "grouped_album_present"))


def _line_cleanup(line: str) -> str:
    line = str(line or "").replace("\xa0", " ").strip()
    line = re.sub(r"^[•\-\u2022▪▫◾◽]+\s*", "", line)
    return line.strip()


def _looks_decorative_line(line: str) -> bool:
    scan_line = _normalize_keycap_digit_dates(line)
    return bool(scan_line) and not bool(re.search(r"[0-9A-Za-zА-Яа-яЁё]", scan_line))


def _looks_generic_preamble(line: str) -> bool:
    low = collapse_ws(line).lower()
    return any(
        token in low
        for token in (
            "экскурсии и путешествия на",
            "в марте у меня для вас насыщенная программа",
            "весна, идём гулять",
            "весна, идем гулять",
        )
    )


def _has_schedule_anchor(line: str) -> bool:
    scan_line = _normalize_keycap_digit_dates(line)
    low = collapse_ws(scan_line).lower()
    if any(token in low for token in ("мест нет", "лист ожидания", "уже набрана")):
        return False
    return bool(DATE_RE.search(scan_line) or "завтра" in low or "сегодня" in low)


def _is_section_break(line: str) -> bool:
    cleaned = collapse_ws(line)
    if not cleaned:
        return False
    if TIME_RE.search(cleaned):
        return False
    low = cleaned.lower()
    if low.startswith(("обзорные экскурсии", "апрельская премьера", "аудиоквест", "бесплатные лекции")):
        return True
    return cleaned.endswith(":") and len(cleaned.split()) <= 6 and not _has_schedule_anchor(cleaned)


def split_occurrence_blocks(text: str) -> list[str]:
    lines = [_line_cleanup(line) for line in str(text or "").splitlines()]
    lines = [line for line in lines if line and not _looks_decorative_line(line)]
    if not lines:
        return []

    blocks: list[list[str]] = []
    current: list[str] = []
    preamble: list[str] = []
    anchors = 0
    section_break_seen = False
    for line in lines:
        if anchors > 0 and current and _is_section_break(line):
            blocks.append(current)
            current = []
            preamble = [line]
            section_break_seen = True
            continue
        if _has_schedule_anchor(line) and current:
            blocks.append(current)
            current = [line]
            anchors += 1
            continue
        if _has_schedule_anchor(line) and not current:
            anchors += 1
            carry_preamble = [item for item in preamble if not _looks_generic_preamble(item)]
            current = [*carry_preamble, line] if carry_preamble else [line]
            preamble = []
            continue
        if anchors == 0:
            preamble.append(line)
            continue
        current.append(line)
    if current:
        blocks.append(current)

    if anchors <= 1 and not section_break_seen:
        payload = collapse_ws(text)
        return [payload] if payload else []
    return [collapse_ws("\n".join(block)) for block in blocks if collapse_ws("\n".join(block))]


def build_occurrence_blocks(text: str, *, limit: int = 8) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for idx, block in enumerate(split_occurrence_blocks(text), start=1):
        cleaned = collapse_ws(block)
        if not cleaned:
            continue
        schedule_anchor_text = collapse_ws(_normalize_keycap_digit_dates(cleaned))
        low = cleaned.lower()
        block_payload = {
            "id": f"B{idx}",
            "text": cleaned[:1200],
            "has_schedule_anchor": _has_schedule_anchor(cleaned),
            "has_time_signal": bool(TIME_RE.search(cleaned)),
            "looks_detail_pending": any(
                token in low
                for token in (
                    "подробности позже",
                    "подробности будут позже",
                    "детали позже",
                    "детали будут позже",
                )
            ),
        }
        if schedule_anchor_text and schedule_anchor_text != cleaned:
            block_payload["schedule_anchor_text"] = schedule_anchor_text[:1200]
        blocks.append(block_payload)
        if len(blocks) >= limit:
            break
    return blocks


def split_text_chunks(text: str, *, limit: int = 8) -> list[dict[str, str]]:
    chunks: list[dict[str, str]] = []
    for idx, block in enumerate(re.split(r"\n{2,}", str(text or "").strip()), start=1):
        cleaned = collapse_ws(block)
        if not cleaned:
            continue
        chunks.append({"id": f"T{idx}", "text": cleaned[:700]})
        if len(chunks) >= limit:
            break
    if not chunks and collapse_ws(text):
        chunks.append({"id": "T1", "text": collapse_ws(text)[:700]})
    return chunks


def _compact_source_payload(source_payload: dict[str, Any]) -> dict[str, Any]:
    return {
        "username": collapse_ws(source_payload.get("username")),
        "source_kind": collapse_ws(source_payload.get("source_kind")),
        "title": collapse_ws(source_payload.get("title")),
        "display_name": collapse_ws(source_payload.get("display_name")),
        "marketing_name": collapse_ws(source_payload.get("marketing_name")),
        "base_region": collapse_ws(source_payload.get("base_region")),
    }


def _compact_screen_payload(screen: dict[str, Any]) -> dict[str, Any]:
    return {
        "decision": collapse_ws(screen.get("decision")),
        "post_kind": collapse_ws(screen.get("post_kind")),
        "extract_mode": collapse_ws(screen.get("extract_mode")),
        "digest_eligible_default": collapse_ws(screen.get("digest_eligible_default")),
        "base_region_fit": collapse_ws(screen.get("base_region_fit")),
        "contains_future_public_signal": bool(screen.get("contains_future_public_signal")),
        "contains_past_report_signal": bool(screen.get("contains_past_report_signal")),
    }


def _compact_post_payload(
    post: ScannedPost,
    *,
    flags: dict[str, Any],
    for_extract: bool = False,
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    excerpt_limit = 1500 if for_extract else 700
    chunk_limit = 3 if for_extract else 2
    chunk_text_limit = 420 if for_extract else 220
    chunks = []
    for item in split_text_chunks(post.text, limit=chunk_limit):
        chunks.append(
            {
                "id": item.get("id"),
                "text": collapse_ws(item.get("text"))[:chunk_text_limit],
            }
        )
    compact_flags = {
        key: bool(flags.get(key))
        for key in (
            "has_date_signal",
            "has_time_signal",
            "has_price_signal",
            "has_booking_signal",
            "has_status_signal",
            "has_group_signal",
            "has_excursion_keywords",
            "has_ocr_excursion_signal",
            "grouped_album_present",
        )
    }
    compact_ocr_chunks: list[dict[str, Any]] = []
    for item in ocr_chunks or []:
        text = collapse_ws(item.get("text"))[:chunk_text_limit]
        title = collapse_ws(item.get("title"))[:120]
        if not text and not title:
            continue
        compact_ocr_chunks.append(
            {
                "id": item.get("id"),
                "title": title or None,
                "text": text or "",
                "excursion_signal": bool(item.get("excursion_signal")),
                "schedule_signal": bool(item.get("schedule_signal")),
                "booking_signal": bool(item.get("booking_signal")),
            }
        )
        if len(compact_ocr_chunks) >= 3:
            break
    payload = {
        "message_id": post.message_id,
        "post_date_utc": post.post_date.isoformat(),
        "message_url": post.source_url,
        "text_excerpt": collapse_ws(post.text)[:excerpt_limit],
        "text_chunks": chunks,
        "ocr_chunks": compact_ocr_chunks,
        "media_hints": {
            "photo_count": sum(1 for item in post.media_refs if collapse_ws(item.get("kind")).lower() == "photo"),
            "video_count": sum(1 for item in post.media_refs if collapse_ws(item.get("kind")).lower() == "video"),
        },
        "prefilter_flags": compact_flags,
    }
    if for_extract:
        payload["schedule_blocks"] = _compact_occurrence_blocks(post.text, limit=8)
    return payload


def _compact_occurrence_blocks(text: str, *, limit: int = 5) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for block in build_occurrence_blocks(text, limit=limit):
        payload = {
            "id": block.get("id"),
            "text": collapse_ws(block.get("text"))[:420],
            "has_schedule_anchor": bool(block.get("has_schedule_anchor")),
            "has_time_signal": bool(block.get("has_time_signal")),
            "looks_detail_pending": bool(block.get("looks_detail_pending")),
        }
        if collapse_ws(block.get("schedule_anchor_text")):
            payload["schedule_anchor_text"] = collapse_ws(block.get("schedule_anchor_text"))[:420]
        blocks.append(payload)
    return blocks


def _string_list_value(value: Any, *, limit: int = 8) -> list[str]:
    out: list[str] = []
    if isinstance(value, str):
        raw = collapse_ws(value)
        items = re.split(r"\s*[;,]\s*", raw) if raw else []
    elif isinstance(value, (list, tuple, set)):
        items = list(value)
    else:
        items = []
    for item in items:
        text = collapse_ws(item)
        if not text or text in out:
            continue
        out.append(text)
        if len(out) >= limit:
            break
    return out


def _extract_json(raw: str) -> Any | None:
    cleaned = str(raw or "").strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", cleaned)
        cleaned = cleaned.replace("```", "")
    try:
        data = json.loads(cleaned)
        return data if isinstance(data, (dict, list)) else None
    except Exception:
        pass
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end > start:
        try:
            data = json.loads(cleaned[start : end + 1])
            return data if isinstance(data, (dict, list)) else None
        except Exception:
            return None
    start = cleaned.find("[")
    end = cleaned.rfind("]")
    if start >= 0 and end > start:
        try:
            data = json.loads(cleaned[start : end + 1])
            return data if isinstance(data, list) else None
        except Exception:
            return None
    return None


_OCCURRENCE_FIELD_TYPES: dict[str, dict[str, Any]] = {
    "source_block_id": {"type": "string"},
    "canonical_title": {"type": "string"},
    "title_normalized": {"type": "string"},
    "date": {"type": "string"},
    "time": {"type": "string"},
    "duration_text": {"type": "string"},
    "city": {"type": "string"},
    "meeting_point": {"type": "string"},
    "route_summary": {"type": "string"},
    "audience_fit": {"type": "array", "items": {"type": "string"}},
    "group_format": {"type": "string"},
    "price_text": {"type": "string"},
    "booking_text": {"type": "string"},
    "booking_url": {"type": "string"},
    "channel_url": {"type": "string"},
    "status": {"type": "string"},
    "seats_text": {"type": "string"},
    "summary_one_liner": {"type": "string"},
    "digest_blurb": {"type": "string"},
    "digest_eligible": {"type": "boolean"},
    "digest_eligibility_reason": {"type": "string"},
    "is_last_call": {"type": "boolean"},
    "post_kind": {"type": "string"},
    "availability_mode": {"type": "string"},
    "guide_names": {"type": "array", "items": {"type": "string"}},
    "organizer_names": {"type": "array", "items": {"type": "string"}},
    "base_region_fit": {"type": "string"},
    "fact_pack": {"type": "object"},
    "fact_claims": {"type": "array", "items": {"type": "object"}},
    "template_hint": {"type": "object"},
    "profile_hint": {"type": "object"},
}


def _occurrence_schema(*keys: str) -> dict[str, Any]:
    properties = {
        key: dict(_OCCURRENCE_FIELD_TYPES[key])
        for key in keys
        if key in _OCCURRENCE_FIELD_TYPES
    }
    return {
        "type": "object",
        "properties": properties,
    }


def _occurrences_wrapper_schema(*keys: str) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "occurrences": {
                "type": "array",
                "items": _occurrence_schema(*keys),
            }
        },
        "required": ["occurrences"],
    }


def _single_occurrence_wrapper_schema(*keys: str) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "occurrence": _occurrence_schema(*keys)
        },
        "required": ["occurrence"],
    }


async def ask_gemma(
    model: str,
    prompt: Any,
    *,
    consumer: str,
    max_output_tokens: int = 2200,
    response_schema: dict[str, Any] | None = None,
    timeout_seconds: int | None = None,
    timeout_retries: int | None = None,
    provider_5xx_retries: int | None = None,
    log_context: str | None = None,
) -> Any | None:
    if not (os.getenv(GOOGLE_KEY_ENV) or os.getenv(GOOGLE_FALLBACK_KEY_ENV) or "").strip():
        raise RuntimeError(f"{GOOGLE_KEY_ENV} is missing in Kaggle runtime")
    client = _get_gemma_client(consumer)
    call_timeout = max(1, int(timeout_seconds or LLM_TIMEOUT_SECONDS))
    call_timeout_retries = LLM_TIMEOUT_RETRY_ATTEMPTS if timeout_retries is None else max(0, int(timeout_retries))
    call_provider_5xx_retries = (
        LLM_PROVIDER_5XX_RETRY_ATTEMPTS if provider_5xx_retries is None else max(0, int(provider_5xx_retries))
    )
    timeout_retries_used = 0
    provider_5xx_retries_used = 0
    context_suffix = f" {collapse_ws(log_context)}" if collapse_ws(log_context) else ""
    for attempt in range(4):
        try:
            raw, _usage = await asyncio.wait_for(
                client.generate_content_async(
                    model=model,
                    prompt=prompt,
                    generation_config={
                        "temperature": 0,
                        **(
                            {
                                "response_mime_type": "application/json",
                                "response_schema": response_schema,
                            }
                            if response_schema is not None
                            else {}
                        ),
                    },
                    max_output_tokens=max_output_tokens,
                ),
                timeout=call_timeout,
            )
            return _extract_json(raw or "")
        except Exception as exc:
            retry_after = _retry_after_seconds(str(exc))
            if retry_after is not None and attempt < 3:
                await asyncio.sleep(min(90.0, retry_after + 1.0))
                continue
            if isinstance(exc, asyncio.TimeoutError) and attempt < 3 and timeout_retries_used < call_timeout_retries:
                timeout_retries_used += 1
                delay = min(12.0, 2.0 * timeout_retries_used)
                print(
                    (
                        "[gemma:retry] "
                        f"consumer={consumer} model={model} reason=timeout "
                        f"attempt={attempt + 1} retry={timeout_retries_used}/{call_timeout_retries} "
                        f"delay={delay:.1f}s"
                        f"{context_suffix}"
                    ),
                    flush=True,
                )
                await asyncio.sleep(delay)
                continue
            status_5xx = _provider_5xx_status(exc)
            if status_5xx is not None and attempt < 3 and provider_5xx_retries_used < call_provider_5xx_retries:
                provider_5xx_retries_used += 1
                delay = min(15.0, 3.0 * provider_5xx_retries_used)
                print(
                    (
                        "[gemma:retry] "
                        f"consumer={consumer} model={model} reason=provider_{status_5xx} "
                        f"attempt={attempt + 1} retry={provider_5xx_retries_used}/{call_provider_5xx_retries} "
                        f"delay={delay:.1f}s"
                        f"{context_suffix}"
                    ),
                    flush=True,
                )
                await asyncio.sleep(delay)
                continue
            raise
    return None


def _boolish(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    raw = collapse_ws(value)
    if not raw:
        return False
    return raw.lower() in {"1", "true", "yes", "y", "on", "да"}


def _normalize_choice(value: Any, *, allowed: set[str], default: str, aliases: dict[str, str] | None = None) -> str:
    raw = collapse_ws(value).lower()
    if aliases:
        raw = aliases.get(raw, raw)
    return raw if raw in allowed else default


def _normalize_reasons(value: Any, *, limit: int = 3) -> list[str]:
    if not isinstance(value, list):
        return []
    out: list[str] = []
    for item in value:
        text = collapse_ws(item)
        if not text or text in out:
            continue
        out.append(text[:220])
        if len(out) >= limit:
            break
    return out


def _normalize_digest_eligibility(
    *,
    date_iso: str | None,
    availability_mode: str | None,
    status: str | None,
    digest_eligible: bool,
    digest_reason: str | None,
) -> tuple[bool, str | None]:
    reason = collapse_ws(digest_reason)
    if reason in {"tentative_or_free_date", "sold_out", "cancelled", "missing_date", "not_scheduled_public", "non_target"}:
        return False, reason
    if not collapse_ws(date_iso):
        return False, reason or "missing_date"
    if collapse_ws(availability_mode) and collapse_ws(availability_mode) != "scheduled_public":
        return False, reason or "not_scheduled_public"
    if collapse_ws(status) == "cancelled":
        return False, reason or "cancelled"
    return bool(digest_eligible), reason or None


def _coerce_occurrence_items(data: Any) -> list[dict[str, Any]]:
    if isinstance(data, list):
        return [item for item in data if isinstance(item, dict)]
    if not isinstance(data, dict):
        return []
    occurrences = data.get("occurrences")
    if isinstance(occurrences, list):
        return [item for item in occurrences if isinstance(item, dict)]
    occurrence = data.get("occurrence")
    if isinstance(occurrence, dict):
        return [occurrence]
    return []


def _has_material_value(value: Any) -> bool:
    if isinstance(value, str):
        return bool(collapse_ws(value))
    if isinstance(value, (list, tuple, set)):
        return any(_has_material_value(item) for item in value)
    if isinstance(value, dict):
        return any(_has_material_value(item) for item in value.values())
    return value is not None


def _seed_fact_pack_from_occurrence(item: dict[str, Any]) -> dict[str, Any]:
    pack = dict(item.get("fact_pack") or {}) if isinstance(item.get("fact_pack"), dict) else {}
    for key in (
        "canonical_title",
        "title_normalized",
        "date",
        "time",
        "duration_text",
        "city",
        "meeting_point",
        "route_summary",
        "audience_fit",
        "group_format",
        "price_text",
        "booking_text",
        "booking_url",
        "status",
        "seats_text",
        "summary_one_liner",
        "digest_blurb",
        "digest_eligible",
        "digest_eligibility_reason",
        "is_last_call",
        "post_kind",
        "availability_mode",
        "guide_names",
        "organizer_names",
        "base_region_fit",
    ):
        value = item.get(key)
        if _has_material_value(value) and key not in pack:
            pack[key] = value
    return pack


def _merge_string_lists(left: Any, right: Any, *, limit: int = 8) -> list[str]:
    return _string_list_value([*(_string_list_value(left, limit=limit)), *(_string_list_value(right, limit=limit))], limit=limit)


def _merge_occurrence_layers(base: dict[str, Any], patch: dict[str, Any]) -> dict[str, Any]:
    merged = dict(base)
    protected_keys = {
        "canonical_title",
        "title_normalized",
        "date",
        "time",
        "source_block_id",
        "source_fingerprint",
        "channel_url",
    }
    list_keys = {"audience_fit", "guide_names", "organizer_names"}
    for key, value in patch.items():
        if key == "fact_pack":
            continue
        if key == "fact_claims":
            existing = merged.get("fact_claims") if isinstance(merged.get("fact_claims"), list) else []
            incoming = value if isinstance(value, list) else []
            merged["fact_claims"] = [*existing, *incoming]
            continue
        if key in list_keys:
            merged[key] = _merge_string_lists(merged.get(key), value, limit=8 if key == "audience_fit" else 4)
            continue
        if key in {"template_hint", "profile_hint"}:
            current = merged.get(key) if isinstance(merged.get(key), dict) else {}
            incoming = value if isinstance(value, dict) else {}
            merged[key] = {**current, **incoming}
            continue
        if key in protected_keys and _has_material_value(merged.get(key)):
            continue
        if _has_material_value(value):
            merged[key] = value
    merged["fact_pack"] = {
        **_seed_fact_pack_from_occurrence(base),
        **(patch.get("fact_pack") if isinstance(patch.get("fact_pack"), dict) else {}),
        **_seed_fact_pack_from_occurrence(merged),
    }
    return merged


def _semantic_focus_excerpt(post: ScannedPost, *, source_block_id: str | None = None) -> str:
    if source_block_id:
        for block in build_occurrence_blocks(post.text, limit=8):
            if collapse_ws(block.get("id")) == collapse_ws(source_block_id):
                return collapse_ws(block.get("text"))[:900]
    return collapse_ws(post.text)[:900]


async def screen_post(
    source_payload: dict[str, Any],
    post: ScannedPost,
    flags: dict[str, Any],
    *,
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    compact_source = _compact_source_payload(source_payload)
    compact_post = _compact_post_payload(post, flags=flags, for_extract=False, ocr_chunks=ocr_chunks)
    schema = {
        "type": "object",
        "properties": {
            "decision": {"type": "string", "enum": ["ignore", "announce", "status_update", "template_only"]},
            "post_kind": {
                "type": "string",
                "enum": [
                    "announce_single",
                    "announce_multi",
                    "status_update",
                    "reportage",
                    "template_signal",
                    "on_demand_offer",
                    "mixed_or_non_target",
                ],
            },
            "extract_mode": {"type": "string", "enum": ["none", "announce", "status", "template"]},
            "digest_eligible_default": {"type": "string", "enum": ["yes", "no", "mixed"]},
            "contains_future_public_signal": {"type": "boolean"},
            "contains_past_report_signal": {"type": "boolean"},
            "base_region_fit": {"type": "string", "enum": ["inside", "outside", "ambiguous", "unknown"]},
            "reasons": {"type": "array", "items": {"type": "string"}},
            "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        },
        "required": [
            "decision",
            "post_kind",
            "extract_mode",
            "digest_eligible_default",
            "contains_future_public_signal",
            "contains_past_report_signal",
            "base_region_fit",
            "reasons",
            "confidence",
        ],
    }
    prompt = (
        "Classify one Telegram post from a guide/excursions source. Return only JSON.\n"
        "Allowed values:\n"
        "- decision: ignore | announce | status_update | template_only\n"
        "- post_kind: announce_single | announce_multi | status_update | reportage | template_signal | on_demand_offer | mixed_or_non_target\n"
        "- extract_mode: none | announce | status | template\n"
        "- digest_eligible_default: yes | no | mixed\n"
        "- base_region_fit: inside | outside | ambiguous | unknown\n"
        "- confidence: low | medium | high\n"
        "Rules:\n"
        "- announce/status_update only if the post text or OCR contains a real guided public excursion/walk/tour/route signal grounded in the input\n"
        "- do not treat a dated event as an excursion just because it is posted by a guide source; the public product must be a guided walk/excursion/tour/route/storytelling visit\n"
        "- volunteer cleanups, subbotniks, restoration work days, community service, lectures without a guided route, and generic meetups are mixed_or_non_target/ignore unless the post explicitly announces a guided excursion or walk as the primary public offer\n"
        "- if there is a concrete future walk/excursion with date/time/meeting point, prefer announce or status_update over reportage\n"
        "- if the body is mostly historical/reportage but ends with or inserts a concrete future excursion CTA — including relative date markers (this Sunday, tomorrow, next weekend) or a named guide — treat it as announce or status_update, not reportage; absence of exact time or meeting point is fine\n"
        "- on-demand offers or posts saying only that dates remain without naming the dates may be announce/template_only, but digest_eligible_default must be no or mixed until a concrete future date is grounded\n"
        "- generic calendars, inspiration, bloom/lifestyle, or travel-wishlist posts without a concrete excursion -> ignore\n"
        "- if one post clearly contains several different excursions led by the source's own guide, use announce_multi\n"
        "- a post that enumerates multiple festivals/events across different cities or regions as a round-up/travel calendar is template_only or ignore, even when one entry falls inside source.base_region; individual enumerated entries are not per-guide excursions and must not be materialized as announce\n"
        "- base_region_fit is your own semantic judgement about whether the concrete excursion(s) in the post take place inside source.base_region; it must not be derived from keyword matching alone\n"
        "- if the post is about places outside source.base_region (other regions/countries/cities) or is a multi-region travel calendar, set base_region_fit=outside or ambiguous and prefer decision=ignore or template_only\n"
        "- if source.base_region is empty or the post has no concrete place, use base_region_fit=unknown\n"
        "- do not invent facts\n"
        "- reasons must be 1-3 short grounded strings\n"
        "- no reasoning, analysis, or hidden thinking traces\n"
        f"Input:\n{json.dumps({'source': compact_source, 'post': compact_post}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        SCREEN_MODEL,
        prompt,
        consumer="guide_scout_screen",
        max_output_tokens=160,
        response_schema=schema,
    )
    if not isinstance(data, dict):
        return {
            "decision": "ignore",
            "post_kind": "mixed_or_non_target",
            "extract_mode": "none",
            "digest_eligible_default": "mixed",
            "contains_future_public_signal": False,
            "contains_past_report_signal": False,
            "base_region_fit": "unknown",
            "reasons": ["llm_parse_failed"],
            "confidence": "low",
        }
    decision = _normalize_choice(
        data.get("decision"),
        allowed={"ignore", "announce", "status_update", "template_only"},
        default="ignore",
    )
    post_kind = _normalize_choice(
        data.get("post_kind"),
        allowed={
            "announce_single",
            "announce_multi",
            "status_update",
            "reportage",
            "template_signal",
            "on_demand_offer",
            "mixed_or_non_target",
        },
        default="mixed_or_non_target",
    )
    extract_mode = _normalize_choice(
        data.get("extract_mode"),
        allowed={"none", "announce", "status", "template"},
        default="none",
    )
    digest_eligible_default = _normalize_choice(
        data.get("digest_eligible_default"),
        allowed={"yes", "no", "mixed"},
        aliases={"true": "yes", "false": "no"},
        default="mixed",
    )
    base_region_fit = _normalize_choice(
        data.get("base_region_fit"),
        allowed={"inside", "outside", "ambiguous", "unknown"},
        default="unknown",
    )
    confidence = _normalize_choice(
        data.get("confidence"),
        allowed={"low", "medium", "high"},
        default="low",
    )
    return {
        "decision": decision,
        "post_kind": post_kind,
        "extract_mode": extract_mode,
        "digest_eligible_default": digest_eligible_default,
        "contains_future_public_signal": _boolish(data.get("contains_future_public_signal")),
        "contains_past_report_signal": _boolish(data.get("contains_past_report_signal")),
        "base_region_fit": base_region_fit,
        "reasons": _normalize_reasons(data.get("reasons")),
        "confidence": confidence,
    }


async def _extract_announce_post_tier1(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
    timeout_seconds: int | None = None,
    timeout_retries: int | None = None,
    provider_5xx_retries: int | None = None,
) -> list[dict[str, Any]]:
    compact_source = _compact_source_payload(source_payload)
    compact_screen = _compact_screen_payload(screen)
    compact_post = _compact_post_payload(post, flags=flags, for_extract=True, ocr_chunks=ocr_chunks)
    schema = _occurrences_wrapper_schema(
        "source_block_id",
        "canonical_title",
        "title_normalized",
        "date",
        "time",
        "duration_text",
        "city",
        "meeting_point",
        "route_summary",
        "audience_fit",
        "group_format",
        "price_text",
        "booking_text",
        "booking_url",
        "channel_url",
        "status",
        "seats_text",
        "digest_eligible",
        "digest_eligibility_reason",
        "is_last_call",
        "post_kind",
        "availability_mode",
        "guide_names",
        "organizer_names",
        "base_region_fit",
        "fact_pack",
    )
    prompt = (
        "You are trail_scout.announce_extract_tier1.v1.\n"
        "Extract Tier-1 public occurrence facts from one Telegram post about guide excursions.\n"
        "Return only JSON with key occurrences.\n"
        "Rules:\n"
        "- extract only real excursion occurrences or direct public updates about a specific occurrence\n"
        "- ignore past occurrences for MVP\n"
        "- if the post contains several dated schedule lines, return one occurrence per dated line even when they share one booking/contact block\n"
        "- use post.schedule_blocks as the complete schedule index when text_excerpt is shortened; keep source_block_id equal to the block id when a block id is available\n"
        "- numeric emoji keycaps in dates are normal digits: 3️⃣ мая means 3 мая; 1️⃣3️⃣ мая means 13 мая\n"
        "- if a dated line has no explicit closed/sold-out/cancelled marker and the post has shared booking/contact/meeting facts, set status=available, availability_mode=scheduled_public, digest_eligible=true\n"
        "- booking_url must be a current booking/registration/details link for this occurrence; in multi-date schedules do not use an inline route-title link to an older wall post unless the surrounding text explicitly says it is the current booking/details page for the same dated line\n"
        "- if a dated line is explicitly tentative/preliminary/only hoped-for or says it is just a free date to move another walk into, do not mark it digest-ready; use digest_eligible=false with digest_eligibility_reason=tentative_or_free_date\n"
        "- when a dated line says places are gone/sold out/full/cancelled, set the matching unavailable status and digest_eligible=false\n"
        "- do not output an extra template/no-date occurrence for a route already covered by concrete dated occurrences in this same post\n"
        "- title_normalized must be a short stable route identity core; do not include guide names, organizer/source labels, parentheses, dates, times, marketing suffixes, or availability words there\n"
        "- volunteer cleanups, subbotniks, restoration work days, community service, lectures without a guided route, and generic meetups are not excursion occurrences unless the block explicitly makes a guided excursion/walk/tour the primary public offer\n"
        "- use OCR facts when the poster carries operational details missing from the text, but only if they are explicit on the poster\n"
        "- set base_region_fit per occurrence by your own judgement versus source.base_region (inside|outside|ambiguous|unknown); do not rely on keyword matching\n"
        "- if an occurrence clearly takes place outside source.base_region, set base_region_fit=outside; do not silently drop it, let the server filter\n"
        "- do not invent details; if a field is unclear, leave it empty\n"
        "- keep only Tier-1 public facts; no extra commentary or hidden thinking traces\n\n"
        f"Input:\n{json.dumps({'source': compact_source, 'screen': compact_screen, 'post': compact_post}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="guide_scout_announce_tier1_extract",
        max_output_tokens=520,
        response_schema=schema,
        timeout_seconds=timeout_seconds,
        timeout_retries=timeout_retries,
        provider_5xx_retries=provider_5xx_retries,
    )
    return _coerce_occurrence_items(data)


async def _extract_announce_post_tier1_failopen_for_block_rescue(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    try:
        return await _extract_announce_post_tier1(
            source_payload,
            post=post,
            flags=flags,
            screen=screen,
            ocr_chunks=ocr_chunks,
            timeout_seconds=ANNOUNCE_MULTI_FULL_TIMEOUT_SECONDS,
            timeout_retries=0,
        )
    except Exception as exc:
        print(
            (
                "[guide:announce_extract:warning] "
                f"message_id={post.message_id} mode=block_rescue "
                f"error={type(exc).__name__}: {exc}"
            ),
            flush=True,
        )
        return []


async def _extract_status_post(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    compact_source = _compact_source_payload(source_payload)
    compact_screen = _compact_screen_payload(screen)
    compact_post = _compact_post_payload(post, flags=flags, for_extract=True, ocr_chunks=ocr_chunks)
    schema = _occurrences_wrapper_schema(
        "source_block_id",
        "canonical_title",
        "title_normalized",
        "date",
        "time",
        "city",
        "meeting_point",
        "route_summary",
        "price_text",
        "booking_text",
        "booking_url",
        "status",
        "seats_text",
        "digest_eligible",
        "digest_eligibility_reason",
        "is_last_call",
        "post_kind",
        "availability_mode",
        "guide_names",
        "organizer_names",
        "base_region_fit",
        "fact_claims",
        "fact_pack",
    )
    prompt = (
        "You are trail_scout.status_claim_extract.v1.\n"
        "Extract one occurrence or direct occurrence update from a Telegram post.\n"
        "Return only JSON with key occurrences.\n"
        "Rules:\n"
        "- focus on status deltas such as last_call, seats left, moved time, changed meeting point, cancellation, or clarified booking\n"
        "- use OCR only for explicit grounded deltas visible on the poster/image\n"
        "- do not invent missing fields\n"
        "- fact_claims should use only claim_role anchor or status_delta\n"
        "- no extra commentary or hidden thinking traces\n\n"
        f"Input:\n{json.dumps({'source': compact_source, 'screen': compact_screen, 'post': compact_post}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="guide_scout_status_claim_extract",
        max_output_tokens=380,
        response_schema=schema,
    )
    return _coerce_occurrence_items(data)


async def _extract_template_post(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    compact_source = _compact_source_payload(source_payload)
    compact_screen = _compact_screen_payload(screen)
    compact_post = _compact_post_payload(post, flags=flags, for_extract=True, ocr_chunks=ocr_chunks)
    schema = _occurrences_wrapper_schema(
        "source_block_id",
        "canonical_title",
        "title_normalized",
        "city",
        "route_summary",
        "audience_fit",
        "group_format",
        "booking_text",
        "booking_url",
        "post_kind",
        "availability_mode",
        "guide_names",
        "organizer_names",
        "base_region_fit",
        "digest_eligible",
        "digest_eligibility_reason",
        "template_hint",
        "profile_hint",
        "fact_claims",
        "fact_pack",
    )
    prompt = (
        "You are trail_scout.template_extract.v1.\n"
        "Extract template-level excursion information from a Telegram post that has no concrete future occurrence yet.\n"
        "Return only JSON with key occurrences.\n"
        "Rules:\n"
        "- no future date means digest_eligible must stay false\n"
        "- if the same post also contains concrete dated occurrence blocks for this route, do not create a duplicate template occurrence\n"
        "- volunteer cleanups, subbotniks, restoration work days, community service, lectures without a guided route, and generic meetups are not template excursions unless a guided route/walk/tour is the primary reusable offer\n"
        "- use template_hint for reusable route/topic information\n"
        "- OCR may contribute reusable route/topic facts only when they are explicit on the poster/image\n"
        "- do not invent schedule facts\n"
        "- no extra commentary or hidden thinking traces\n\n"
        f"Input:\n{json.dumps({'source': compact_source, 'screen': compact_screen, 'post': compact_post}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="guide_scout_template_extract",
        max_output_tokens=320,
        response_schema=schema,
    )
    return _coerce_occurrence_items(data)


def _clean_occurrence_payload(
    item: dict[str, Any],
    *,
    post: ScannedPost,
    source_payload: dict[str, Any],
    screen: dict[str, Any] | None = None,
) -> dict[str, Any] | None:
    title = collapse_ws(item.get("canonical_title"))
    if not title:
        return None
    title_normalized = collapse_ws(item.get("title_normalized")) or normalize_title_key(title)
    if not title_normalized:
        return None
    date_iso = collapse_ws(item.get("date") or item.get("date_iso")) or None
    time_text = collapse_ws(item.get("time") or item.get("time_text")) or None
    route_summary = collapse_ws(item.get("route_summary")) or None
    city = collapse_ws(item.get("city")) or None
    meeting_point = collapse_ws(item.get("meeting_point")) or None
    summary_one_liner = collapse_ws(item.get("summary_one_liner")) or None
    base_region_fit = (
        collapse_ws(item.get("base_region_fit"))
        or collapse_ws((screen or {}).get("base_region_fit"))
        or "unknown"
    )
    if base_region_fit == "outside":
        return None
    availability_mode = collapse_ws(item.get("availability_mode")) or "scheduled_public"
    status = collapse_ws(item.get("status")) or "scheduled"
    digest_eligible, digest_reason = _normalize_digest_eligibility(
        date_iso=date_iso,
        availability_mode=availability_mode,
        status=status,
        digest_eligible=_boolish(item.get("digest_eligible")),
        digest_reason=collapse_ws(item.get("digest_eligibility_reason")) or None,
    )
    out = {
        "canonical_title": title,
        "title_normalized": title_normalized,
        "date": date_iso,
        "time": time_text,
        "duration_text": collapse_ws(item.get("duration_text")) or None,
        "city": city,
        "meeting_point": meeting_point,
        "route_summary": route_summary,
        "audience_fit": _string_list_value(item.get("audience_fit"), limit=8),
        "group_format": collapse_ws(item.get("group_format")) or None,
        "price_text": collapse_ws(item.get("price_text")) or None,
        "booking_text": collapse_ws(item.get("booking_text")) or None,
        "booking_url": collapse_ws(item.get("booking_url")) or None,
        "channel_url": collapse_ws(item.get("channel_url")) or post.source_url,
        "status": status,
        "seats_text": collapse_ws(item.get("seats_text")) or None,
        "summary_one_liner": summary_one_liner,
        "digest_blurb": collapse_ws(item.get("digest_blurb")) or None,
        "digest_eligible": digest_eligible,
        "digest_eligibility_reason": digest_reason,
        "is_last_call": _boolish(item.get("is_last_call")),
        "post_kind": collapse_ws(item.get("post_kind")) or "announce_single",
        "availability_mode": availability_mode,
        "guide_names": _string_list_value(item.get("guide_names"), limit=4),
        "organizer_names": _string_list_value(item.get("organizer_names"), limit=4),
        "source_block_id": collapse_ws(item.get("source_block_id")) or None,
        "base_region_fit": base_region_fit,
        "source_fingerprint": collapse_ws(item.get("source_fingerprint")) or build_source_fingerprint(title_normalized=title_normalized, date_iso=date_iso, time_text=time_text),
        "fact_pack": item.get("fact_pack") if isinstance(item.get("fact_pack"), dict) else {},
        "fact_claims": item.get("fact_claims") if isinstance(item.get("fact_claims"), list) else [],
        "template_hint": item.get("template_hint") if isinstance(item.get("template_hint"), dict) else {},
        "profile_hint": item.get("profile_hint") if isinstance(item.get("profile_hint"), dict) else {},
    }
    if not out["fact_pack"]:
        out["fact_pack"] = {}
    out["fact_pack"]["base_region_fit"] = base_region_fit
    if route_summary and "route_summary" not in out["fact_pack"]:
        out["fact_pack"]["route_summary"] = route_summary
    if out["duration_text"] and "duration_text" not in out["fact_pack"]:
        out["fact_pack"]["duration_text"] = out["duration_text"]
    if out["group_format"] and "group_format" not in out["fact_pack"]:
        out["fact_pack"]["group_format"] = out["group_format"]
    return out


async def _extract_occurrence_block(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    block: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any] | None:
    compact_source = _compact_source_payload(source_payload)
    compact_screen = _compact_screen_payload(screen)
    compact_post = _compact_post_payload(post, flags=flags, for_extract=True, ocr_chunks=ocr_chunks)
    compact_post_for_block = {
        "message_id": compact_post.get("message_id"),
        "post_date_utc": compact_post.get("post_date_utc"),
        "message_url": compact_post.get("message_url"),
        "post_context_excerpt": collapse_ws(compact_post.get("text_excerpt"))[:1400],
        "ocr_chunks": compact_post.get("ocr_chunks") or [],
        "media_hints": compact_post.get("media_hints") or {},
        "prefilter_flags": compact_post.get("prefilter_flags") or {},
    }
    schema = _single_occurrence_wrapper_schema(
        "source_block_id",
        "canonical_title",
        "title_normalized",
        "date",
        "time",
        "duration_text",
        "city",
        "meeting_point",
        "route_summary",
        "audience_fit",
        "group_format",
        "price_text",
        "booking_text",
        "booking_url",
        "channel_url",
        "status",
        "seats_text",
        "digest_eligible",
        "digest_eligibility_reason",
        "is_last_call",
        "post_kind",
        "availability_mode",
        "guide_names",
        "organizer_names",
        "base_region_fit",
        "fact_pack",
    )
    prompt = (
        "You are trail_scout.announce_extract_tier1.v1.\n"
        "Extract Tier 1 guide-excursion facts for exactly one candidate occurrence block from a multi-announcement Telegram post.\n"
        "Return only JSON with key occurrence.\n"
        "If the block is not a real public or template-like excursion signal inside the base region, return {\"occurrence\": {}}.\n"
        "Rules:\n"
        "- treat the block as one primary excursion candidate\n"
        "- materialize only if the block is primarily a guided public excursion/walk/tour/route or direct update about one\n"
        "- do not materialize volunteer cleanups, subbotniks, restoration work days, community service, lectures without a guided route, or generic meetups unless a guided excursion/walk/tour is the primary public offer\n"
        "- if title/date/route signal is present, materialize the occurrence even when some details are still pending\n"
        "- numeric emoji keycaps in dates are normal digits: 3️⃣ мая means 3 мая; 1️⃣3️⃣ мая means 13 мая; use occurrence_block.schedule_anchor_text only as a normalized reading aid\n"
        "- use post.post_context_excerpt only for shared facts that clearly apply to all schedule blocks, such as common booking/contact, organizer, price policy, or meeting context\n"
        "- do not borrow title/date/time/route facts from a different dated block in post.post_context_excerpt\n"
        "- if the block has a future date/time and no closed/sold-out/cancelled marker, set status=available, availability_mode=scheduled_public, digest_eligible=true\n"
        "- if the block is explicitly tentative/preliminary/only hoped-for or says it is just a free date to move another walk into, do not mark it digest-ready; use digest_eligible=false with digest_eligibility_reason=tentative_or_free_date\n"
        "- if the block says places are gone/sold out/full/cancelled, set the matching unavailable status and digest_eligible=false\n"
        "- title_normalized must be a short stable route identity core; do not include guide names, organizer/source labels, parentheses, dates, times, marketing suffixes, or availability words there\n"
        "- OCR may rescue missing title/date/time facts only when they are explicit on the poster/image\n"
        "- ignore unrelated side notes inside the same block\n"
        "- set base_region_fit per occurrence by your own judgement versus source.base_region (inside|outside|ambiguous|unknown); do not rely on keyword matching\n"
        "- do not invent details; keep source_block_id equal to the input block id\n"
        "- no extra commentary or hidden thinking traces\n\n"
        f"Input:\n{json.dumps({'source': compact_source, 'screen': compact_screen, 'post': compact_post_for_block, 'occurrence_block': {'id': block.get('id'), 'text': collapse_ws(block.get('text'))[:900], 'schedule_anchor_text': collapse_ws(block.get('schedule_anchor_text'))[:900], 'has_schedule_anchor': bool(block.get('has_schedule_anchor')), 'has_time_signal': bool(block.get('has_time_signal')), 'looks_detail_pending': bool(block.get('looks_detail_pending'))}}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="guide_scout_tier1_extract_block",
        max_output_tokens=340,
        response_schema=schema,
    )
    items = _coerce_occurrence_items(data)
    item = items[0] if items else None
    if not isinstance(item, dict):
        return None
    if not collapse_ws(item.get("source_block_id")):
        item["source_block_id"] = block.get("id")
    semantic_patch = await _extract_occurrence_semantics_failopen(
        source_payload,
        post=post,
        flags=flags,
        screen=screen,
        occurrence_seed=item,
        focus_excerpt=_semantic_focus_excerpt(post, source_block_id=block.get("id")),
        ocr_chunks=ocr_chunks,
    )
    return _clean_occurrence_payload(
        _merge_occurrence_layers(item, semantic_patch),
        post=post,
        source_payload=source_payload,
        screen=screen,
    )


async def _extract_occurrence_block_failopen(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    block: dict[str, Any],
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any] | None:
    try:
        return await _extract_occurrence_block(
            source_payload,
            post=post,
            flags=flags,
            screen=screen,
            block=block,
            ocr_chunks=ocr_chunks,
        )
    except Exception as exc:
        print(
            (
                "[guide:block_extract:warning] "
                f"message_id={post.message_id} block_id={collapse_ws(block.get('id')) or '-'} "
                f"error={type(exc).__name__}: {exc}"
            ),
            flush=True,
        )
        return None


async def _extract_occurrence_semantics(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    occurrence_seed: dict[str, Any],
    focus_excerpt: str,
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    compact_source = _compact_source_payload(source_payload)
    compact_screen = _compact_screen_payload(screen)
    compact_post = {
        "message_id": post.message_id,
        "message_url": post.source_url,
        "post_date_utc": post.post_date.isoformat(),
        "focus_excerpt": collapse_ws(focus_excerpt)[:900],
        "ocr_chunks": [
            {
                "id": item.get("id"),
                "title": collapse_ws(item.get("title"))[:120] or None,
                "text": collapse_ws(item.get("text"))[:280],
                "excursion_signal": bool(item.get("excursion_signal")),
                "schedule_signal": bool(item.get("schedule_signal")),
                "booking_signal": bool(item.get("booking_signal")),
            }
            for item in (ocr_chunks or [])
            if collapse_ws(item.get("text")) or collapse_ws(item.get("title"))
        ][:3],
    }
    seed = {
        "source_block_id": collapse_ws(occurrence_seed.get("source_block_id")),
        "canonical_title": collapse_ws(occurrence_seed.get("canonical_title")),
        "date": collapse_ws(occurrence_seed.get("date")),
        "time": collapse_ws(occurrence_seed.get("time")),
        "city": collapse_ws(occurrence_seed.get("city")),
        "meeting_point": collapse_ws(occurrence_seed.get("meeting_point")),
        "route_summary": collapse_ws(occurrence_seed.get("route_summary")),
        "price_text": collapse_ws(occurrence_seed.get("price_text")),
        "booking_text": collapse_ws(occurrence_seed.get("booking_text")),
        "guide_names": _string_list_value(occurrence_seed.get("guide_names"), limit=4),
        "organizer_names": _string_list_value(occurrence_seed.get("organizer_names"), limit=4),
        "base_region_fit": collapse_ws(occurrence_seed.get("base_region_fit")),
        "post_kind": collapse_ws(occurrence_seed.get("post_kind")),
        "availability_mode": collapse_ws(occurrence_seed.get("availability_mode")),
    }
    schema = _single_occurrence_wrapper_schema(
        "audience_fit",
        "group_format",
        "duration_text",
        "route_summary",
        "city",
        "meeting_point",
        "price_text",
        "booking_text",
        "booking_url",
        "status",
        "seats_text",
        "summary_one_liner",
        "digest_blurb",
        "digest_eligible",
        "digest_eligibility_reason",
        "is_last_call",
        "guide_names",
        "organizer_names",
        "base_region_fit",
        "fact_claims",
        "template_hint",
        "profile_hint",
        "fact_pack",
    )
    prompt = (
        "You are route_weaver.enrich.v1 for guide excursions.\n"
        "Enrich one already extracted occurrence with missing semantic facts from the same Telegram post.\n"
        "Return only JSON with key occurrence.\n"
        "Rules:\n"
        "- do not rename or replace canonical_title/date/time/source_block_id from the seed\n"
        "- fill only facts supported by the focus excerpt or OCR chunks from the same post\n"
        "- preserve the dominant term family from the source: прогулка stays прогулка, экскурсия stays экскурсия\n"
        "- do not downgrade a seed with concrete future date/time/booking/meeting facts to status unknown or digest_eligible=false unless the focus excerpt explicitly says sold out/full/cancelled/past/private\n"
        "- for a concrete future public schedule with no disqualifying status, keep or set status=available, availability_mode=scheduled_public, digest_eligible=true\n"
        "- volunteer cleanups, subbotniks, restoration work days, community service, lectures without a guided route, and generic meetups stay digest_eligible=false unless the guided excursion/walk/tour is the primary public offer\n"
        "- base_region_fit is your own semantic judgement versus source.base_region; do not weaken seed.base_region_fit unless the focus excerpt explicitly contradicts it\n"
        "- summary_one_liner and digest_blurb are optional; leave them empty if evidence is weak\n"
        "- fact_claims may use only claim_role: anchor, support, status_delta, template_hint, guide_profile_hint\n"
        "- no extra commentary or hidden thinking traces\n\n"
        f"Input:\n{json.dumps({'source': compact_source, 'screen': compact_screen, 'post': compact_post, 'occurrence_seed': seed}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="route_weaver_enrich",
        max_output_tokens=380,
        response_schema=schema,
    )
    items = _coerce_occurrence_items(data)
    item = items[0] if items else None
    return item if isinstance(item, dict) else {}


async def _extract_occurrence_semantics_failopen(
    source_payload: dict[str, Any],
    *,
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    occurrence_seed: dict[str, Any],
    focus_excerpt: str,
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    try:
        return await _extract_occurrence_semantics(
            source_payload,
            post=post,
            flags=flags,
            screen=screen,
            occurrence_seed=occurrence_seed,
            focus_excerpt=focus_excerpt,
            ocr_chunks=ocr_chunks,
        )
    except Exception as exc:
        print(
            (
                "[guide:enrich:warning] "
                f"message_id={post.message_id} block_id={collapse_ws(occurrence_seed.get('source_block_id')) or '-'} "
                f"error={type(exc).__name__}: {exc}"
            ),
            flush=True,
        )
        return {}


async def extract_post(
    source_payload: dict[str, Any],
    post: ScannedPost,
    flags: dict[str, Any],
    screen: dict[str, Any],
    *,
    ocr_chunks: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    occurrence_blocks = build_occurrence_blocks(post.text, limit=6)
    is_multi = str(screen.get("post_kind") or "") == "announce_multi"
    extract_mode = str(screen.get("extract_mode") or "none")
    if extract_mode == "status":
        items = await _extract_status_post(source_payload, post=post, flags=flags, screen=screen, ocr_chunks=ocr_chunks)
    elif extract_mode == "template":
        items = await _extract_template_post(source_payload, post=post, flags=flags, screen=screen, ocr_chunks=ocr_chunks)
    elif is_multi and occurrence_blocks:
        items = await _extract_announce_post_tier1_failopen_for_block_rescue(
            source_payload,
            post=post,
            flags=flags,
            screen=screen,
            ocr_chunks=ocr_chunks,
        )
    else:
        items = await _extract_announce_post_tier1(source_payload, post=post, flags=flags, screen=screen, ocr_chunks=ocr_chunks)
    cleaned: list[dict[str, Any]] = []
    seen_fingerprints: set[str] = set()
    if isinstance(items, list):
        for item in items:
            if not isinstance(item, dict):
                continue
            merged_item = item
            if extract_mode == "announce":
                semantic_patch = await _extract_occurrence_semantics_failopen(
                    source_payload,
                    post=post,
                    flags=flags,
                    screen=screen,
                    occurrence_seed=item,
                    focus_excerpt=_semantic_focus_excerpt(post, source_block_id=item.get("source_block_id")),
                    ocr_chunks=ocr_chunks,
                )
                merged_item = _merge_occurrence_layers(item, semantic_patch)
            occurrence = _clean_occurrence_payload(merged_item, post=post, source_payload=source_payload, screen=screen)
            if occurrence:
                fingerprint = str(occurrence.get("source_fingerprint") or "")
                if fingerprint and fingerprint in seen_fingerprints:
                    continue
                if fingerprint:
                    seen_fingerprints.add(fingerprint)
                cleaned.append(occurrence)
    if is_multi and occurrence_blocks:
        covered_block_ids = {
            collapse_ws(item.get("source_block_id"))
            for item in cleaned
            if collapse_ws(item.get("source_block_id"))
        }
        for block in occurrence_blocks:
            block_id = collapse_ws(block.get("id"))
            if not block_id or block_id in covered_block_ids or not bool(block.get("has_schedule_anchor")):
                continue
            rescued = await _extract_occurrence_block_failopen(
                source_payload,
                post=post,
                flags=flags,
                screen=screen,
                block=block,
                ocr_chunks=ocr_chunks,
            )
            if not rescued:
                continue
            fingerprint = str(rescued.get("source_fingerprint") or "")
            if fingerprint and fingerprint in seen_fingerprints:
                continue
            if fingerprint:
                seen_fingerprints.add(fingerprint)
            cleaned.append(rescued)
            covered_block_ids.add(block_id)
    return cleaned


def _llm_status_from_exception(exc: Exception) -> str:
    rate_limit_error = None
    provider_error = None
    try:
        from google_ai.exceptions import ProviderError, RateLimitError

        provider_error = ProviderError
        rate_limit_error = RateLimitError
    except Exception:
        provider_error = None
        rate_limit_error = None
    if rate_limit_error is not None and isinstance(exc, rate_limit_error):
        blocked = (getattr(exc, "blocked_reason", None) or "unknown").strip() or "unknown"
        return f"llm_deferred_rate_limit:{blocked}"
    if provider_error is not None and isinstance(exc, provider_error) and int(getattr(exc, "status_code", 0) or 0) == 429:
        return "llm_deferred_provider_429"
    if isinstance(exc, asyncio.TimeoutError):
        return "llm_deferred_timeout"
    return f"llm_error:{type(exc).__name__}"


def _summarize_source_stats(posts: list[dict[str, Any]]) -> dict[str, int]:
    llm_ok = 0
    llm_deferred = 0
    llm_error = 0
    for post in posts:
        status = str(post.get("llm_status") or "").strip()
        if status == "ok":
            llm_ok += 1
        elif status.startswith("llm_deferred"):
            llm_deferred += 1
        elif status not in {"", "skipped_prefilter"}:
            llm_error += 1
    return {
        "posts_total": len(posts),
        "prefilter_true": sum(1 for post in posts if bool(post.get("prefilter_passed"))),
        "llm_ok": llm_ok,
        "llm_deferred": llm_deferred,
        "llm_error": llm_error,
        "occurrences_total": sum(len(post.get("occurrences") or []) for post in posts),
    }


def _summarize_run_stats(sources_output: list[dict[str, Any]]) -> dict[str, int]:
    totals = {
        "sources_total": len(sources_output),
        "posts_total": 0,
        "prefilter_true": 0,
        "llm_ok": 0,
        "llm_deferred": 0,
        "llm_error": 0,
        "occurrences_total": 0,
    }
    for source_payload in sources_output:
        stats = source_payload.get("stats") if isinstance(source_payload.get("stats"), dict) else {}
        for key in ("posts_total", "prefilter_true", "llm_ok", "llm_deferred", "llm_error", "occurrences_total"):
            totals[key] += int(stats.get(key) or 0)
    return totals


async def process_source(client: TelegramClient | None, source_payload: dict[str, Any], *, limit: int, days_back: int) -> dict[str, Any]:
    platform = collapse_ws(source_payload.get("platform")).lower() or "telegram"
    username = str(source_payload.get("username") or "").strip()
    source_kind = str(source_payload.get("source_kind") or "").strip()
    source_label = f"{platform}:{username}" if platform != "telegram" else f"@{username}"
    print(f"[source:start] {source_label} kind={source_kind or 'unknown'} limit={limit} days_back={days_back}", flush=True)
    if platform == "vk":
        meta, posts = await scan_vk_source_posts(source_payload, limit=limit, days_back=days_back)
    else:
        if client is None:
            raise RuntimeError("Telegram client is unavailable")
        try:
            await ensure_client_connected(client)
            meta, posts = await scan_source_posts(client, username=username, limit=limit, days_back=days_back)
        except Exception as exc:
            if "disconnected" not in str(exc).lower():
                raise
            await ensure_client_connected(client, force_reconnect=True)
            meta, posts = await scan_source_posts(client, username=username, limit=limit, days_back=days_back)
    out_posts: list[dict[str, Any]] = []
    errors: list[str] = []
    for post in posts:
        ocr_chunks: list[dict[str, Any]] = []
        ocr_status = "skipped"
        base_flags = prefilter_flags(post)
        base_pass = prefilter_pass(post, source_kind, base_flags)
        if platform == "telegram" and client is not None and _should_run_post_ocr(post, source_kind, base_flags, base_pass=base_pass):
            try:
                ocr_chunks = await collect_post_ocr_chunks(
                    client,
                    username=username,
                    post=post,
                    model=EXTRACT_MODEL,
                )
                ocr_status = "ok" if ocr_chunks else "empty"
            except Exception as exc:
                ocr_status = f"error:{type(exc).__name__}"
                print(
                    f"[guide:ocr:error] source=@{username} message_id={post.message_id} error={type(exc).__name__}: {exc}",
                    flush=True,
                )
        flags = prefilter_flags(post, ocr_chunks=ocr_chunks)
        passes = prefilter_pass(post, source_kind, flags)
        if passes and platform == "telegram" and client is not None:
            media_assets = await materialize_post_media_assets(client, username=username, post=post)
        elif passes and platform == "vk":
            media_assets = materialize_vk_post_media_assets(username=username, post=post)
        else:
            media_assets = []
        payload: dict[str, Any] = {
            "message_id": post.message_id,
            "grouped_id": post.grouped_id,
            "post_date": post.post_date.isoformat(),
            "source_url": post.source_url,
            "text": post.text,
            "views": post.views,
            "forwards": post.forwards,
            "reactions_total": post.reactions_total,
            "reactions_json": post.reactions_json,
            "media_refs": post.media_refs,
            "media_assets": media_assets,
            "ocr_chunks": ocr_chunks,
            "ocr_status": ocr_status,
            "prefilter_passed": passes,
            "prefilter_flags": flags,
            "llm_status": "skipped_prefilter",
            "screen": {
                "decision": "ignore",
                "post_kind": "mixed_or_non_target",
                "extract_mode": "none",
                "digest_eligible_default": "mixed",
                "contains_future_public_signal": False,
                "contains_past_report_signal": False,
                "reasons": ["prefilter_false"],
                "confidence": "low",
            },
            "occurrences": [],
        }
        if passes:
            try:
                screen = await screen_post(
                    {
                        "platform": platform,
                        "username": username,
                        "source_kind": source_kind,
                        "title": meta.get("source_title"),
                        "display_name": source_payload.get("display_name"),
                        "marketing_name": source_payload.get("marketing_name"),
                        "base_region": source_payload.get("base_region"),
                        "trust_level": source_payload.get("trust_level"),
                        "flags": source_payload.get("flags"),
                    },
                    post,
                    flags,
                    ocr_chunks=ocr_chunks,
                )
                payload["screen"] = screen
                if screen.get("extract_mode") != "none" and screen.get("decision") != "ignore":
                    payload["occurrences"] = await extract_post(
                        {
                            "platform": platform,
                            "username": username,
                            "source_kind": source_kind,
                            "title": meta.get("source_title"),
                            "display_name": source_payload.get("display_name"),
                            "marketing_name": source_payload.get("marketing_name"),
                            "base_region": source_payload.get("base_region"),
                            "trust_level": source_payload.get("trust_level"),
                            "flags": source_payload.get("flags"),
                        },
                        post,
                        flags,
                        screen,
                        ocr_chunks=ocr_chunks,
                    )
                payload["llm_status"] = "ok"
            except Exception as exc:
                error_kind = _llm_status_from_exception(exc)
                payload["llm_status"] = error_kind
                payload["screen"] = {
                    "decision": "ignore",
                    "post_kind": "mixed_or_non_target",
                    "extract_mode": "none",
                    "digest_eligible_default": "mixed",
                    "contains_future_public_signal": False,
                    "contains_past_report_signal": False,
                    "reasons": [error_kind],
                    "confidence": "low",
                }
                errors.append(f"message_id={post.message_id}: {error_kind}: {exc}")
        out_posts.append(payload)
    source_status = "partial" if errors else "ok"
    print(
        (
            f"[source:done] {source_label} status={source_status} "
            f"posts={len(posts)} extracted={sum(len((item.get('occurrences') or [])) for item in out_posts)} "
            f"errors={len(errors)}"
        ),
        flush=True,
    )
    stats = _summarize_source_stats(out_posts)
    return {
        "platform": platform,
        "username": username,
        "source_title": meta.get("source_title"),
        "source_url": meta.get("source_url") or source_payload.get("source_url"),
        "source_kind": source_kind,
        "about_text": meta.get("about_text"),
        "about_links": meta.get("about_links"),
        "source_status": source_status,
        "posts_scanned": len(posts),
        "stats": stats,
        "errors": errors,
        "posts": out_posts,
    }


async def main() -> None:
    config = load_runtime_config()
    _bootstrap_repo_bundle()
    refresh_runtime_settings()
    run_id = str(config.get("run_id") or f"guide_kaggle_{int(datetime.now(timezone.utc).timestamp())}")
    started_at = datetime.now(timezone.utc).isoformat()
    sources = [item for item in (config.get("sources") or []) if isinstance(item, dict)]
    acquired_resources: list[str] = []
    STATUS_PROGRESS.update(
        {
            "phase": "preflight",
            "run_id": run_id,
            "sources_total": len(sources),
            "sources_done": 0,
            "limit_per_source": int(config.get("limit_per_source") or 25),
            "days_back": int(config.get("days_back") or 7),
            "progress_percent": 5,
            "progress_label": f"источники 0/{len(sources)}",
        }
    )
    _status_event("kernel_started", phase="preflight", status="running", progress=dict(STATUS_PROGRESS))
    if STATUS_CLIENT is not None:
        STATUS_CLIENT.start_alive(interval_seconds=60, progress_provider=_status_progress)
        for resource_key in STATUS_CLIENT.config.get("resource_leases") or []:
            if not STATUS_CLIENT.acquire_resource(str(resource_key), ttl_seconds=3 * 60 * 60):
                raise RuntimeError(f"Required Kaggle resource is busy: {resource_key}")
            acquired_resources.append(str(resource_key))
    print(
        (
            f"Guide monitor run_id={run_id} mode={config.get('scan_mode') or 'full'} "
            f"sources={len(sources)} limit={int(config.get('limit_per_source') or 25)} "
            f"days_back={int(config.get('days_back') or 7)} "
            f"screen_model={SCREEN_MODEL} extract_model={EXTRACT_MODEL}"
        ),
        flush=True,
    )
    has_telegram_sources = any(
        (collapse_ws(source.get("platform")).lower() or "telegram") == "telegram"
        for source in sources
    )
    client = await create_client() if has_telegram_sources else None
    partial = False
    sources_output: list[dict[str, Any]] = []
    try:
        limit = int(config.get("limit_per_source") or 25)
        days_back = int(config.get("days_back") or 7)
        for idx, source in enumerate(sources, start=1):
            if not isinstance(source, dict):
                continue
            STATUS_PROGRESS.update(
                {
                    "phase": "scan",
                    "source_index": idx,
                    "sources_total": len(sources),
                    "source": str(source.get("username") or source.get("source_url") or ""),
                    "sources_done": len(sources_output),
                    "progress_label": (
                        f"источники {idx}/{len(sources)} · "
                        f"{source.get('username') or source.get('source_url') or ''}"
                    ),
                }
            )
            _status_event("source_started", phase="scan", status="running", progress=dict(STATUS_PROGRESS))
            try:
                if client is not None and (collapse_ws(source.get("platform")).lower() or "telegram") == "telegram":
                    await ensure_client_connected(client)
                result = await process_source(client, source, limit=limit, days_back=days_back)
            except Exception as exc:
                partial = True
                result = {
                    "platform": collapse_ws(source.get("platform")).lower() or "telegram",
                    "username": str(source.get("username") or ""),
                    "source_title": str(source.get("title") or "") or None,
                    "source_url": str(source.get("source_url") or "") or None,
                    "source_kind": str(source.get("source_kind") or "") or None,
                    "source_status": "error",
                    "posts_scanned": 0,
                    "errors": [f"{type(exc).__name__}: {exc}"],
                    "posts": [],
                }
            if result.get("source_status") != "ok":
                partial = True
            sources_output.append(result)
            STATUS_PROGRESS.update(
                {
                    "sources_done": len(sources_output),
                    "posts_scanned": sum(int(item.get("posts_scanned") or 0) for item in sources_output),
                    "progress_label": (
                        f"источники {len(sources_output)}/{len(sources)} · "
                        f"посты {sum(int(item.get('posts_scanned') or 0) for item in sources_output)}"
                    ),
                }
            )
            _status_event("source_done", phase="scan", status="running", progress=dict(STATUS_PROGRESS))
    finally:
        if client is not None:
            await client.disconnect()
        for resource_key in acquired_resources:
            if STATUS_CLIENT is not None:
                STATUS_CLIENT.release_resource(resource_key)
    stats = _summarize_run_stats(sources_output)
    payload = {
        "schema_version": 1,
        "run_id": run_id,
        "scan_mode": str(config.get("scan_mode") or "full"),
        "started_at": started_at,
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "partial": partial,
        "stats": stats,
        "sources": sources_output,
    }
    RESULT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    STATUS_PROGRESS.update(
        {
            "phase": "report",
            "sources_done": len(sources_output),
            "posts_total": stats.get("posts_total", 0),
            "occurrences_total": stats.get("occurrences_total", 0),
            "partial": partial,
            "output": str(RESULT_PATH),
            "progress_percent": 100,
            "progress_label": (
                f"источники {len(sources_output)}/{len(sources)} · "
                f"посты {stats.get('posts_total', 0)} · экскурсии {stats.get('occurrences_total', 0)}"
            ),
        }
    )
    _status_event(
        "report_written",
        phase="report",
        status="done" if not partial else "partial",
        progress=dict(STATUS_PROGRESS),
    )
    if STATUS_CLIENT is not None:
        STATUS_CLIENT.stop_alive()
    print(
        (
            f"Guide monitor completed partial={partial} "
            f"sources={len(sources_output)} "
            f"posts={sum(int(item.get('posts_scanned') or 0) for item in sources_output)}"
        ),
        flush=True,
    )
    print(
        (
            "Guide monitor stats "
            f"posts_total={stats['posts_total']} "
            f"prefilter_true={stats['prefilter_true']} "
            f"llm_ok={stats['llm_ok']} "
            f"llm_deferred={stats['llm_deferred']} "
            f"llm_error={stats['llm_error']} "
            f"occurrences_total={stats['occurrences_total']}"
        ),
        flush=True,
    )
    print(f"Wrote {RESULT_PATH}")




In [ ]:
print('Guide notebook bootstrap complete', flush=True)
print(f'Guide notebook runner file={__file__}', flush=True)
print(f'Guide notebook embedded google_ai root={_GUIDE_EMBEDDED_ROOT}', flush=True)


In [ ]:
import asyncio
import nest_asyncio

def _guide_run_main_sync() -> None:
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    if loop.is_closed():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    if loop.is_running():
        nest_asyncio.apply(loop)
    loop.run_until_complete(main())

_guide_run_main_sync()
